# BTCUSD v15 fusion diagnostics (no training)

CPU-only. This notebook answers whether Label V2 direction heads
(10m / 15m / 30m / 1h) fail because of labels, leakage, or a market state
that does not determine the frozen-theta class. Last-bar direction
predictability (confusion, AUC, log-loss vs prior, knockout) runs on CPU
here. Checkpoint ``best.pt`` confusion is CLI-only. **Do not** train
`MtfFusionTransformer` here. Live remains v11. Do not copy anything into
`agent/model_storage/`. 2h stays an encoder input, not a research label.

Uses cached Delta India OHLCV when present (`/content/cache/btcusd_5m_raw.parquet`).

Edit repo `.py` files and regenerate:

    python scripts/colab/build_fusion_diagnostics_notebook.py

**Colab setup:** Runtime → CPU is enough.


## 01 Env

In [ ]:
# Diagnostics need sklearn/pandas; no GPU training extras.
!pip install -q --upgrade-strategy only-if-needed pyarrow requests scikit-learn scipy

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("numpy", np.__version__)
print("CPU diagnostics; Transformer training is out of scope.")


## Feature contract

In [ ]:
"""Constants shared between per-TF Colab training and agent inference."""

from __future__ import annotations

from typing import Any, Dict, Mapping, Optional, Tuple

# Bump when FEATURE_COLS, label heads, or ONNX outputs change (requires retrain).
FEATURE_CONTRACT_VERSION_V6 = "transformer_btcusd_per_tf_features_v6"
FEATURE_CONTRACT_VERSION_V7 = "transformer_btcusd_per_tf_features_v7"
FEATURE_CONTRACT_VERSION_V8 = "transformer_btcusd_per_tf_features_v8"
FEATURE_CONTRACT_VERSION_V9 = "transformer_btcusd_per_tf_features_v9"
FEATURE_CONTRACT_VERSION_V10 = "transformer_btcusd_mtf_fusion_v10"
FEATURE_CONTRACT_VERSION_V11 = "transformer_btcusd_mtf_fusion_v11"
FEATURE_CONTRACT_VERSION_V12 = "transformer_btcusd_mtf_fusion_v12"
FEATURE_CONTRACT_VERSION_V13 = "transformer_btcusd_mtf_fusion_v13"
FEATURE_CONTRACT_VERSION_V14 = "transformer_btcusd_mtf_fusion_v14"
FEATURE_CONTRACT_VERSION_V15 = "transformer_btcusd_mtf_fusion_v15"
# Live per-TF bundles remain v9. The fused live path uses V11 (2-class, no h10m head).
# V12/V13/V14/V15 are Label V2 research training only and must not load in FusionModelNode.
FEATURE_CONTRACT_VERSION = FEATURE_CONTRACT_VERSION_V9

SUPPORTED_RESOLUTIONS: Tuple[str, ...] = ("5m", "15m", "30m", "1h", "2h")
FUSION_INPUT_RESOLUTIONS: Tuple[str, ...] = ("5m", "10m", "30m", "1h", "2h")
FUSION_BUNDLE_DIR_NAME = "JackSparrow_Transformer_BTCUSD_mtf_fusion"
FUSION_V12_BUNDLE_DIR_NAME = "JackSparrow_Transformer_BTCUSD_mtf_fusion_v12"
FUSION_V13_BUNDLE_DIR_NAME = "JackSparrow_Transformer_BTCUSD_mtf_fusion_v13"
FUSION_V14_BUNDLE_DIR_NAME = "JackSparrow_Transformer_BTCUSD_mtf_fusion_v14"
FUSION_V15_BUNDLE_DIR_NAME = "JackSparrow_Transformer_BTCUSD_mtf_fusion_v15"
FUSION_MODEL_FAMILY = "jacksparrow_transformer_btcusd_mtf_fusion"
FUSION_ONNX_FILENAME = "btcusd_mtf_fusion.onnx"
FUSION_TARGET_WINDOW_MINUTES: int = 10 * 60
FUSION_EMBARGO_BARS: int = 24

RESOLUTION_MINUTES: Dict[str, int] = {
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
}

TF_KEYS: Tuple[str, ...] = tuple(f"tf_{r}" for r in SUPPORTED_RESOLUTIONS)

HTF_SOURCE_TFS: Tuple[str, ...] = ("15m", "30m", "1h", "2h")
HTF_FEATURE_FIELDS: Tuple[str, ...] = (
    "structure_bias",
    "trend_efficiency",
    "ema21_slope_atr",
    "dist_support_atr",
    "dist_resistance_atr",
    "range_width_atr",
)

NATIVE_STRUCTURE_COLS: Tuple[str, ...] = (
    "hh_count",
    "hl_count",
    "lh_count",
    "ll_count",
    "structure_bias",
    "last_swing_dir",
    "bars_since_swing",
    "swing_amp_atr",
    "unconfirmed_ext_atr",
    "trend_efficiency",
    "displacement_atr",
    "pct_with_trend",
    "price_vs_ema9_atr",
    "price_vs_ema21_atr",
    "price_vs_ema50_atr",
    "price_vs_ema200_atr",
    "ema9_vs_21_atr",
    "ema21_vs_50_atr",
    "ema50_vs_200_atr",
    "ema21_slope_atr",
    "ema50_slope_atr",
    "dist_to_support_atr",
    "dist_to_resistance_atr",
    "support_touch_count",
    "resistance_touch_count",
    "range_width_atr",
    "range_width_pctile",
    "atr_contraction",
    "breakout_size_atr",
    "breakout_vol_ratio",
    "pre_breakout_comp",
    "bars_since_breakout",
    "retest_dist_atr",
    "failed_break",
    "peak_diff_atr",
    "trough_diff_atr",
    "peak_sep_bars",
    "trough_sep_bars",
    "dist_neck_atr",
    "high_slope_atr",
    "low_slope_atr",
    "convergence",
    "width_now_atr",
    "pole_disp_atr",
    "flag_width_atr",
    "flag_slope_atr",
)

HTF_STRUCTURE_COLS: Tuple[str, ...] = tuple(
    f"htf_{tf}_{field}" for tf in HTF_SOURCE_TFS for field in HTF_FEATURE_FIELDS
)

_V4_FEATURE_COLS: Tuple[str, ...] = (
    "ret_1",
    "rv_16",
    "rv_96",
    "ema50_dist_pct",
    "macd_hist",
    "rsi_14",
    "adx_14",
    "obv_z",
    "vol_z",
    "body_ratio",
    "upper_wick_ratio",
    "lower_wick_ratio",
    "close_loc",
    "range_atr",
    "body_atr",
    "gap_atr",
    "inside_bar",
    "outside_bar",
    "engulf_score",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "dist_to_resistance_pct",
    "dist_to_support_pct",
    "funding_rate",
    "oi_z",
    "funding_zscore",
    "funding_mom",
    "funding_rate_roc",
    "oi_change_2",
    "oi_delta_z",
    "oi_price_divergence",
    "oi_acceleration",
    "funding_x_oi",
)

# Native all-TF continuous columns (v4 plus causal structure/geometry).
FEATURE_COLS: Tuple[str, ...] = _V4_FEATURE_COLS + NATIVE_STRUCTURE_COLS

# v14 research continuous vector: drop duplicate scale encodings and pattern flags.
V14_DROPPED_COLS: Tuple[str, ...] = (
    "ema50_dist_pct",
    "dist_to_support_pct",
    "dist_to_resistance_pct",
    "funding_rate",
)

V14_FEATURE_GROUP_ORDER: Tuple[str, ...] = (
    "price",
    "volatility",
    "trend",
    "momentum",
    "structure",
    "candle",
    "chart",
    "flow",
    "context",
)

V14_FEATURE_GROUPS: Dict[str, Tuple[str, ...]] = {
    "price": ("ret_1",),
    "volatility": ("rv_16", "rv_96", "atr_contraction"),
    "trend": (
        "adx_14",
        "trend_efficiency",
        "displacement_atr",
        "pct_with_trend",
        "price_vs_ema9_atr",
        "price_vs_ema21_atr",
        "price_vs_ema50_atr",
        "price_vs_ema200_atr",
        "ema9_vs_21_atr",
        "ema21_vs_50_atr",
        "ema50_vs_200_atr",
        "ema21_slope_atr",
        "ema50_slope_atr",
    ),
    "momentum": ("macd_hist", "rsi_14"),
    "structure": (
        "hh_count",
        "hl_count",
        "lh_count",
        "ll_count",
        "structure_bias",
        "last_swing_dir",
        "bars_since_swing",
        "swing_amp_atr",
        "unconfirmed_ext_atr",
        "dist_to_support_atr",
        "dist_to_resistance_atr",
        "support_touch_count",
        "resistance_touch_count",
        "range_width_atr",
        "range_width_pctile",
    ),
    "candle": (
        "body_ratio",
        "upper_wick_ratio",
        "lower_wick_ratio",
        "close_loc",
        "range_atr",
        "body_atr",
        "gap_atr",
        "inside_bar",
        "outside_bar",
        "engulf_score",
    ),
    "chart": (
        "breakout_size_atr",
        "breakout_vol_ratio",
        "pre_breakout_comp",
        "bars_since_breakout",
        "retest_dist_atr",
        "failed_break",
        "peak_diff_atr",
        "trough_diff_atr",
        "peak_sep_bars",
        "trough_sep_bars",
        "dist_neck_atr",
        "high_slope_atr",
        "low_slope_atr",
        "convergence",
        "width_now_atr",
        "pole_disp_atr",
        "flag_width_atr",
        "flag_slope_atr",
    ),
    "flow": (
        "obv_z",
        "vol_z",
        "oi_z",
        "funding_zscore",
        "funding_mom",
        "funding_rate_roc",
        "oi_change_2",
        "oi_delta_z",
        "oi_price_divergence",
        "oi_acceleration",
        "funding_x_oi",
    ),
    "context": ("hour_sin", "hour_cos", "dow_sin", "dow_cos"),
}

V14_COUNT_COLS: Tuple[str, ...] = (
    "hh_count",
    "hl_count",
    "lh_count",
    "ll_count",
    "bars_since_swing",
    "support_touch_count",
    "resistance_touch_count",
    "bars_since_breakout",
    "peak_sep_bars",
    "trough_sep_bars",
)

V14_WINDOW_ZSCORE_COLS: Tuple[str, ...] = ("ret_1",)
V14_STATE_GROUP_ORDER: Tuple[str, ...] = ("candle_state", "chart_state")
V14_STRUCTURE_LOOKBACK_BARS: int = 96

PATH_LABEL_COLS: Tuple[str, ...] = (
    "mfe",
    "mae",
    "future_volatility",
    "trend_strength",
    "drawdown_before_mfe",
    "future_oi_change_pct",
    "future_volume_change_pct",
    "candle_follow_through_atr",
    "structure_delta",
)

CONTINUOUS_LABEL_COLS: Tuple[str, ...] = PATH_LABEL_COLS

PATH_LABEL_HORIZON_BARS: int = 8

# Legacy 8h reference (btcusd_15m_transformer used 32 bars on 15m).
REFERENCE_LABEL_HORIZON_MINUTES: int = 480

DEFAULT_PATH_LABEL_HORIZON_MINUTES: Dict[str, int] = {
    "5m": 240,
    "15m": 240,
    "30m": 480,
    "1h": 480,
    "2h": 480,
}

# Volume change excluded from loss (dominates shared encoder).
DEFAULT_CONTINUOUS_LOSS_WEIGHTS: Dict[str, float] = {
    "mfe": 0.5,
    "mae": 0.5,
    "future_volatility": 1.0,
    "trend_strength": 0.5,
    "drawdown_before_mfe": 0.5,
    "future_oi_change_pct": 0.25,
    "future_volume_change_pct": 0.0,
    "candle_follow_through_atr": 0.5,
    "structure_delta": 0.5,
}

REGIME_NAMES: Dict[int, str] = {
    0: "LOW",
    1: "NORMAL",
    2: "HIGH",
    3: "EXTREME",
}

CANDLE_CLASS_COL = "candle_class_id"
CANDLE_CLASS_CARDINALITY = 13
CANDLE_EMBED_DIM = 8
CHART_EMBED_DIM = 8

CANDLE_CLASS_NAMES: Dict[int, str] = {
    0: "FLAT_ZERO_RANGE",
    1: "DOJI_DRAGONFLY",
    2: "DOJI_GRAVESTONE",
    3: "DOJI_STANDARD",
    4: "MARUBOZU_BULL",
    5: "MARUBOZU_BEAR",
    6: "HAMMER_SHAPE",
    7: "INV_HAMMER_SHAPE",
    8: "SPINNING_TOP",
    9: "BELT_HOLD_BULL",
    10: "BELT_HOLD_BEAR",
    11: "STANDARD_BULL",
    12: "STANDARD_BEAR",
}

# Discrete training/inference columns (not continuous ONNX heads).
STRUCTURE_OUTCOME_COL = "future_structure_outcome"
FUTURE_CANDLE_COL = "future_candle_class"
STRUCTURE_OUTCOME_CARDINALITY = 6
N_AUX_CLASS_HEADS = 3  # vol regime + structure outcome + next-bar candle

# v7 next-candle structure (kept for legacy 5m ONNX decode).
NEXT_DIRECTION_COL = "next_direction"
NEXT_BODY_COL = "next_body"
NEXT_WICK_COL = "next_wick"
NEXT_RANGE_COL = "next_range"
CHART_PATTERN_COL = "chart_pattern_id"
VOLUME_STATE_COL = "volume_state"
VOLUME_CONFIRMS_COL = "volume_confirms"
PATTERN_ACTIVE_COL = "pattern_active"
SAMPLE_WEIGHT_COL = "sample_weight"

# v8 wall-clock behavior packets on the 5m grid (bars).
HORIZON_SPECS: Tuple[Tuple[str, int], ...] = (
    ("h5m", 1),
    ("h10m", 2),
    ("h15m", 3),
    ("h30m", 6),
    ("h1h", 12),
    ("h2h", 24),
)
HORIZON_KEYS: Tuple[str, ...] = tuple(key for key, _ in HORIZON_SPECS)
HORIZON_BARS_5M: Tuple[int, ...] = tuple(bars for _, bars in HORIZON_SPECS)
N_HORIZONS: int = len(HORIZON_SPECS)
MAX_V8_HORIZON_BARS: int = max(HORIZON_BARS_5M)
HORIZON_CONTINUOUS_FIELDS: Tuple[str, ...] = ("mfe", "mae", "vol", "trend_strength")
HORIZON_DIR_COLS: Tuple[str, ...] = tuple(f"{key}_dir" for key in HORIZON_KEYS)
HORIZON_STRUCTURE_COLS: Tuple[str, ...] = tuple(
    f"{key}_structure" for key in HORIZON_KEYS
)
V8_CONTINUOUS_LABEL_COLS: Tuple[str, ...] = tuple(
    f"{key}_{field}"
    for key in HORIZON_KEYS
    for field in HORIZON_CONTINUOUS_FIELDS
)
HORIZON_DIR_COLS_V7: Tuple[str, ...] = (
    "horizon_t1_dir",
    "horizon_t3_dir",
    "horizon_t6_dir",
    "horizon_t12_dir",
    "horizon_t24_dir",
)

NEXT_DIRECTION_CARDINALITY_V8 = 3
NEXT_DIRECTION_CARDINALITY = 5
NEXT_BODY_CARDINALITY = 3
NEXT_WICK_CARDINALITY = 4
NEXT_RANGE_CARDINALITY = 3
VOLUME_STATE_CARDINALITY = 3
CHART_PATTERN_CARDINALITY = 9

NEXT_DIRECTION_NAMES_V8: Dict[int, str] = {0: "BEARISH", 1: "NEUTRAL", 2: "BULLISH"}
NEXT_DIRECTION_NAMES: Dict[int, str] = {
    0: "STRONG_DOWN",
    1: "DOWN",
    2: "NEUTRAL",
    3: "UP",
    4: "STRONG_UP",
}
BULLISH_DIRECTION_NAMES = frozenset({"UP", "STRONG_UP", "BULLISH"})
BEARISH_DIRECTION_NAMES = frozenset({"DOWN", "STRONG_DOWN", "BEARISH"})
NEUTRAL_DIRECTION_NAMES = frozenset({"NEUTRAL"})
NEXT_BODY_NAMES: Dict[int, str] = {0: "SMALL", 1: "MEDIUM", 2: "LARGE"}
NEXT_WICK_NAMES: Dict[int, str] = {
    0: "BALANCED",
    1: "UPPER_REJECTION",
    2: "LOWER_REJECTION",
    3: "BOTH_REJECTION",
}
NEXT_RANGE_NAMES: Dict[int, str] = {0: "COMPRESSED", 1: "NORMAL", 2: "EXPANDED"}
VOLUME_STATE_NAMES: Dict[int, str] = {0: "DRY", 1: "NORMAL", 2: "EXPANSION"}
CHART_PATTERN_NAMES: Dict[int, str] = {
    0: "NONE",
    1: "FLAG_BULL",
    2: "FLAG_BEAR",
    3: "TRIANGLE",
    4: "DOUBLE_TOP",
    5: "DOUBLE_BOTTOM",
    6: "CHANNEL",
    7: "BREAKOUT",
    8: "FAILED_BREAK",
}

VOL_Z_EXPANSION = 0.5
VOL_Z_DRY = -0.5
BREAKOUT_VOL_CONFIRM = 1.2
HORIZON_DIR_ATR_WEAK = 0.5
HORIZON_DIR_ATR_STRONG = 2.0
HORIZON_DIR_ATR_DEADZONE = HORIZON_DIR_ATR_WEAK
# Fusion heads: h2h uses a wider ignore band so chop does not train the trunk.
FUSION_HORIZON_ATR_WEAK: Dict[str, float] = {
    "h30m": 0.5,
    "h1h": 0.5,
    "h2h": 0.75,
}

V7_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {
    "direction": 1.0,
    "body": 0.5,
    "wick": 0.5,
    "range": 0.5,
    "pattern": 0.5,
    "path": 1.0,
    "volume_state": 0.25,
    "pattern_validates": 0.25,
    "horizon": 0.35,
}

V8_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {
    "path": 1.0,
    "direction": 1.0,
    "structure": 0.5,
    "volume_state": 0.25,
}
V9_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {
    "path": 1.0,
    "direction": 1.0,
    "structure": 0.5,
    "volume_state": 0.25,
    "pattern": 0.25,
}

V7_CONTINUOUS_LOSS_WEIGHTS: Dict[str, float] = {
    "mfe": 0.5,
    "mae": 0.5,
    "future_volatility": 1.0,
    "trend_strength": 0.5,
    "drawdown_before_mfe": 0.5,
    "future_oi_change_pct": 0.25,
    "future_volume_change_pct": 0.25,
    "candle_follow_through_atr": 0.5,
    "structure_delta": 0.5,
}

STRUCTURE_OUTCOME_NAMES: Dict[int, str] = {
    0: "RANGE",
    1: "CONTINUATION_LONG",
    2: "CONTINUATION_SHORT",
    3: "BREAKOUT",
    4: "FAILED_BREAK",
    5: "REVERSAL",
}

ONNX_OUTPUT_NAMES_V6: Tuple[str, ...] = (
    "continuous_pred",
    "regime_logits",
    "structure_outcome_logits",
    "future_candle_logits",
)
# Live 15m–2h bundles still export v6 heads. 5m research is v8.
ONNX_OUTPUT_NAMES: Tuple[str, ...] = ONNX_OUTPUT_NAMES_V6

ONNX_OUTPUT_NAMES_V7_BASE: Tuple[str, ...] = (
    "next_direction_logits",
    "next_body_logits",
    "next_wick_logits",
    "next_range_logits",
    "next_pattern_logits",
    "continuous_pred",
    "volume_state_logits",
    "pattern_validates_logit",
)
ONNX_OUTPUT_NAMES_V7_HORIZONS: Tuple[str, ...] = (
    "horizon_t3_dir_logits",
    "horizon_t6_dir_logits",
    "horizon_t12_dir_logits",
    "horizon_t24_dir_logits",
)
ONNX_OUTPUT_NAMES_V7: Tuple[str, ...] = (
    ONNX_OUTPUT_NAMES_V7_BASE + ONNX_OUTPUT_NAMES_V7_HORIZONS
)
ONNX_OUTPUT_NAMES_V8_DIR: Tuple[str, ...] = tuple(
    f"{key}_dir_logits" for key in HORIZON_KEYS
)
ONNX_OUTPUT_NAMES_V8_STRUCTURE: Tuple[str, ...] = tuple(
    f"{key}_structure_logits" for key in HORIZON_KEYS
)
ONNX_OUTPUT_NAMES_V8: Tuple[str, ...] = (
    ONNX_OUTPUT_NAMES_V8_DIR
    + ONNX_OUTPUT_NAMES_V8_STRUCTURE
    + ("continuous_pred", "volume_state_logits")
)
ONNX_OUTPUT_NAMES_V9: Tuple[str, ...] = ONNX_OUTPUT_NAMES_V8 + ("chart_pattern_logits",)

# v11 fused multi-TF model: 2-class BEAR/BULL heads, no MFE/MAE.
# 5m/10m stay encoder inputs. h10m is not a trading head. HOLD is not a class.
FUSION_HORIZON_SPECS: Tuple[Tuple[str, int], ...] = (
    ("h30m", 6),
    ("h1h", 12),
    ("h2h", 24),
)
FUSION_HORIZON_KEYS: Tuple[str, ...] = tuple(key for key, _ in FUSION_HORIZON_SPECS)
FUSION_HORIZON_BARS_5M: Tuple[int, ...] = tuple(bars for _, bars in FUSION_HORIZON_SPECS)
N_FUSION_HORIZONS: int = len(FUSION_HORIZON_SPECS)
MAX_FUSION_HORIZON_BARS: int = max(FUSION_HORIZON_BARS_5M)
FUSION_DIR_COLS: Tuple[str, ...] = tuple(f"{key}_dir" for key in FUSION_HORIZON_KEYS)
FUSION_RETIRED_DIR_COLS: Tuple[str, ...] = ("h10m_dir",)
FUSION_IGNORE_INDEX = -1
FUSION_DIRECTION_CARDINALITY = 2
FUSION_DIRECTION_NAMES: Dict[int, str] = {0: "BEAR", 1: "BULL"}
FUSION_POSITION_LONG = "LONG"
FUSION_POSITION_SHORT = "SHORT"
FUSION_POSITION_HOLD = "HOLD"
FUSION_GRADE_HIGH = "HIGH"
FUSION_GRADE_MEDIUM = "MEDIUM"
FUSION_GRADE_LOW = "LOW"
# 2-class chance is 0.50; floors sit above noise (old 0.40/0.45 would auto-pass).
FUSION_HIGH_BALANCED_ACC = 0.58
FUSION_HIGH_MAX_ECE = 0.08
FUSION_MEDIUM_BALANCED_ACC = 0.55
FUSION_MIN_PROBABILITY = 0.60
# Frozen v10 4-head / 3-class names — live fusion must not load this layout.
ONNX_OUTPUT_NAMES_V10: Tuple[str, ...] = (
    "h10m_dir_logits",
    "h30m_dir_logits",
    "h1h_dir_logits",
    "h2h_dir_logits",
    "tf_fusion_logits",
)
ONNX_OUTPUT_NAMES_V11: Tuple[str, ...] = tuple(
    f"{key}_dir_logits" for key in FUSION_HORIZON_KEYS
) + ("tf_fusion_logits",)
FUSION_DURATION_ATR_MULT: Dict[str, Tuple[float, float]] = {
    "h30m": (1.0, 1.5),
    "h1h": (1.5, 2.25),
    "h2h": (2.0, 3.0),
}
FUSION_HORIZON_MINUTES: Dict[str, int] = {
    "h30m": 30,
    "h1h": 60,
    "h2h": 120,
}

# Label V2 research heads. Live fusion stays h30m/h1h/h2h. Encoder still sees 2h.
LABEL_V2_HORIZON_SPECS: Tuple[Tuple[str, int], ...] = (
    ("h10m", 2),
    ("h15m", 3),
    ("h30m", 6),
    ("h1h", 12),
)
LABEL_V2_HORIZON_KEYS: Tuple[str, ...] = tuple(key for key, _ in LABEL_V2_HORIZON_SPECS)
LABEL_V2_HORIZON_BARS_5M: Tuple[int, ...] = tuple(
    bars for _, bars in LABEL_V2_HORIZON_SPECS
)
N_LABEL_V2_HORIZONS: int = len(LABEL_V2_HORIZON_SPECS)
MAX_LABEL_V2_HORIZON_BARS: int = max(LABEL_V2_HORIZON_BARS_5M)
LABEL_V2_DURATION_ATR_MULT: Dict[str, Tuple[float, float]] = {
    "h10m": (0.5, 0.75),
    "h15m": (0.75, 1.25),
    "h30m": (1.0, 1.5),
    "h1h": (1.5, 2.25),
}
LABEL_V2_HORIZON_MINUTES: Dict[str, int] = {
    "h10m": 10,
    "h15m": 15,
    "h30m": 30,
    "h1h": 60,
}

# Label V2 research targets. Live v11 stays 2-class with ignore_index NEUTRAL.
LABEL_V2_DIRECTION_CARDINALITY = 3
LABEL_V2_DIRECTION_NAMES: Dict[int, str] = {0: "BEAR", 1: "NEUTRAL", 2: "BULL"}
LABEL_V2_THETA_GRID: Tuple[float, ...] = (0.25, 0.40, 0.50, 0.60, 0.75, 1.00)
LABEL_V2_TP_SL_GRID: Tuple[Tuple[float, float], ...] = (
    (0.5, 0.75),
    (0.75, 1.25),
    (1.0, 1.5),
    (1.5, 2.25),
    (2.0, 3.0),
)
LABEL_V2_TP_SL_TP_FIRST = 0
LABEL_V2_TP_SL_SL_FIRST = 1
LABEL_V2_TP_SL_NEITHER = 2
LABEL_V2_TP_SL_AMBIGUOUS = 3
LABEL_V2_TP_SL_NAMES: Dict[int, str] = {
    LABEL_V2_TP_SL_TP_FIRST: "TP_FIRST",
    LABEL_V2_TP_SL_SL_FIRST: "SL_FIRST",
    LABEL_V2_TP_SL_NEITHER: "NEITHER",
    LABEL_V2_TP_SL_AMBIGUOUS: "AMBIGUOUS",
}
LABEL_V2_PATH_FIELDS: Tuple[str, ...] = (
    "ret",
    "long_mfe",
    "long_mae",
    "short_mfe",
    "short_mae",
    "persist",
    "t_long_mfe",
    "t_short_mfe",
)
LABEL_V2_COLS: Tuple[str, ...] = tuple(
    f"{key}_{field}"
    for key in LABEL_V2_HORIZON_KEYS
    for field in LABEL_V2_PATH_FIELDS
)
LABEL_V2_MINORITY_RATE = 0.15
LABEL_V2_DISAGREE_MIN = 0.02
LABEL_V2_PERSIST_RANGE_MIN = 0.10
# Frozen from the four-horizon mix report (259,188 labeled 5m bars).
LABEL_V2_THETA_FROZEN: Dict[str, float] = {
    "h10m": 0.50,
    "h15m": 0.50,
    "h30m": 0.50,
    "h1h": 0.50,
}
LABEL_V2_REG_FIELDS: Tuple[str, ...] = (
    "ret",
    "long_mfe",
    "long_mae",
    "short_mfe",
    "short_mae",
)
LABEL_V2_PATH_LOSS_WEIGHTS: Dict[str, float] = {
    "dir": 1.0,
    "ret": 1.0,
    "long_mfe": 0.35,
    "long_mae": 0.35,
    "short_mfe": 0.35,
    "short_mae": 0.35,
}
# later_direction_only research pass: train 3-class CE, keep path heads untrained.
LABEL_V2_DIRECTION_ONLY_PATH_LOSS_WEIGHTS: Dict[str, float] = {
    "dir": 1.0,
    "ret": 0.0,
    "long_mfe": 0.0,
    "long_mae": 0.0,
    "short_mfe": 0.0,
    "short_mae": 0.0,
}
# Frozen v12 long-centric path outputs (ret/mfe/mae). Do not reuse for v13.
LABEL_V2_V12_REG_FIELDS: Tuple[str, ...] = ("ret", "mfe", "mae")
# 3-class chance is ~0.33; research grades only (not a live promote gate).
LABEL_V2_HIGH_BALANCED_ACC = 0.45
LABEL_V2_MEDIUM_BALANCED_ACC = 0.40
ONNX_OUTPUT_NAMES_V12: Tuple[str, ...] = (
    tuple(f"{key}_dir_logits" for key in FUSION_HORIZON_KEYS)
    + tuple(
        f"{key}_{field}"
        for key in FUSION_HORIZON_KEYS
        for field in LABEL_V2_V12_REG_FIELDS
    )
    + ("tf_fusion_logits",)
)
ONNX_OUTPUT_NAMES_V13: Tuple[str, ...] = (
    tuple(f"{key}_dir_logits" for key in FUSION_HORIZON_KEYS)
    + tuple(
        f"{key}_{field}"
        for key in FUSION_HORIZON_KEYS
        for field in LABEL_V2_REG_FIELDS
    )
    + ("tf_fusion_logits",)
)
ONNX_OUTPUT_NAMES_V15: Tuple[str, ...] = (
    tuple(f"{key}_dir_logits" for key in LABEL_V2_HORIZON_KEYS)
    + tuple(
        f"{key}_{field}"
        for key in LABEL_V2_HORIZON_KEYS
        for field in LABEL_V2_REG_FIELDS
    )
    + ("tf_fusion_logits",)
)

# Next-bar candle families for 5m timing (not model classes).
CANDLE_FAMILY_DOJI = frozenset({0, 1, 2, 3, 8})
CANDLE_FAMILY_BULL = frozenset({4, 6, 9, 11})
CANDLE_FAMILY_BEAR = frozenset({5, 7, 10, 12})

# Minimum test-set correlation for future_volatility before ONNX export.
MIN_EXPORT_VOL_CORR: Dict[str, float] = {
    "5m": 0.10,
    "15m": 0.10,
    "30m": 0.10,
    "1h": 0.08,
    "2h": 0.08,
}

PROMOTION_VOL_CORR: float = 0.15
PROMOTION_REGIME_ACCURACY: float = 0.35

EXPORT_QUALITY_DISCLAIMER = (
    "Sanity gates detect broken exports, not trading edge. "
    "Promotion tier targets are informational."
)

TRANSFORMER_METADATA_FILENAME = "metadata_transformer.json"
TRANSFORMER_FEATURE_CONFIG_FILENAME = "feature_config.json"

def candle_family_from_class(class_id: int) -> str:
    """Map a candle class id to bull / bear / doji for timing modifiers."""
    cid = int(class_id)
    if cid in CANDLE_FAMILY_BULL:
        return "bull"
    if cid in CANDLE_FAMILY_BEAR:
        return "bear"
    return "doji"

def compute_path_edge(mfe: float, mae: float) -> float:
    """Directional edge from predicted path asymmetry (MFE minus MAE).

    Alias of :func:`compute_long_edge` kept for train/serve telemetry compatibility.
    """
    return compute_long_edge(mfe, mae)

def compute_long_edge(mfe: float, mae: float) -> float:
    """Long-side path edge: upside (MFE) minus downside (MAE)."""
    return float(mfe) - float(mae)

def compute_short_edge(mfe: float, mae: float) -> float:
    """Short-side path edge: downside (MAE) minus upside (MFE)."""
    return float(mae) - float(mfe)

def path_favorable_adverse(
    mfe: float,
    mae: float,
    *,
    side: str,
) -> Tuple[float, float]:
    """Return (favorable_pct, adverse_pct) for bracket sizing.

    Labels are long-centric (MFE = upside, MAE = downside). For shorts, favorable
    excursion is downside (MAE) and adverse is upside (MFE).
    """
    s = str(side or "BUY").strip().upper()
    if s in ("SELL", "SHORT", "STRONG_SELL"):
        return float(mae), float(mfe)
    return float(mfe), float(mae)

def model_family_for_resolution(resolution: str) -> str:
    """Canonical model_family string for a TF bundle."""
    res = resolution.strip().lower()
    if res == "mtf_fusion":
        return FUSION_MODEL_FAMILY
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    return f"jacksparrow_transformer_btcusd_{res}"

def onnx_filename_for_resolution(resolution: str) -> str:
    res = resolution.strip().lower()
    if res == "mtf_fusion":
        return FUSION_ONNX_FILENAME
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    return f"btcusd_{res}_transformer.onnx"

def bundle_dir_name(resolution: str) -> str:
    res = resolution.strip().lower()
    if res == "mtf_fusion":
        return FUSION_BUNDLE_DIR_NAME
    return f"JackSparrow_Transformer_BTCUSD_{res}"

def horizon_bars_for_wall_minutes(wall_minutes: int, resolution_minutes: int) -> int:
    """Convert wall-clock minutes to native-TF bar count."""
    return max(1, int(round(int(wall_minutes) / resolution_minutes)))

def fusion_window_len(
    resolution: str,
    *,
    target_minutes: int = FUSION_TARGET_WINDOW_MINUTES,
) -> int:
    """Bar count so one TF window covers ``target_minutes`` of wall-clock time."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    return horizon_bars_for_wall_minutes(int(target_minutes), RESOLUTION_MINUTES[res])

def fusion_window_lens(
    resolutions: Tuple[str, ...] = FUSION_INPUT_RESOLUTIONS,
    *,
    target_minutes: int = FUSION_TARGET_WINDOW_MINUTES,
) -> Dict[str, int]:
    """Per-TF window lengths derived from the shared wall-clock target."""
    return {
        res: fusion_window_len(res, target_minutes=target_minutes) for res in resolutions
    }

def resolve_fusion_window_lens(
    window_lens: Optional[Mapping[str, int]] = None,
    window_len: Optional[int] = None,
    *,
    target_minutes: Optional[int] = None,
) -> Dict[str, int]:
    """Prefer an explicit per-TF map; else a uniform int; else the 10h target."""
    if window_lens:
        missing = [res for res in FUSION_INPUT_RESOLUTIONS if res not in window_lens]
        if missing:
            raise KeyError(f"window_lens missing resolutions: {missing}")
        return {res: int(window_lens[res]) for res in FUSION_INPUT_RESOLUTIONS}
    if window_len is not None:
        width = int(window_len)
        return {res: width for res in FUSION_INPUT_RESOLUTIONS}
    minutes = (
        int(target_minutes)
        if target_minutes is not None
        else FUSION_TARGET_WINDOW_MINUTES
    )
    return fusion_window_lens(target_minutes=minutes)

# Longest TF window (5m) for PE max_len and 5m decision-clock warm-up.
FUSION_WINDOW_LEN: int = fusion_window_len("5m")

def label_horizon_bars_for_resolution(resolution_minutes: int) -> int:
    """Legacy helper: 8h wall-clock in bars for a TF grid."""
    return horizon_bars_for_wall_minutes(REFERENCE_LABEL_HORIZON_MINUTES, resolution_minutes)

def path_label_horizon_bars_for_resolution(resolution: str) -> int:
    """Path-label forward window in bars for a TF."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    minutes = RESOLUTION_MINUTES[res]
    wall = DEFAULT_PATH_LABEL_HORIZON_MINUTES.get(res, REFERENCE_LABEL_HORIZON_MINUTES)
    return horizon_bars_for_wall_minutes(wall, minutes)

def continuous_loss_weights_for_resolution(resolution: str) -> Tuple[float, ...]:
    """Per-head loss weights aligned with CONTINUOUS_LABEL_COLS (0 = no gradient)."""
    weights = dict(DEFAULT_CONTINUOUS_LOSS_WEIGHTS)
    return tuple(float(weights.get(col, 1.0)) for col in CONTINUOUS_LABEL_COLS)

def default_training_config(resolution: str) -> Dict[str, Any]:
    """Default Colab training config for a single TF model."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    minutes = RESOLUTION_MINUTES[res]
    path_horizon = path_label_horizon_bars_for_resolution(res)
    return {
        "symbol": "BTCUSD",
        "resolution": res,
        "resolution_minutes": minutes,
        "history_days": 900,
        "base_url": "https://api.india.delta.exchange",
        "atr_period": 14,
        "path_label_horizon_bars": path_horizon,
        "label_horizon_minutes_path": DEFAULT_PATH_LABEL_HORIZON_MINUTES.get(
            res, REFERENCE_LABEL_HORIZON_MINUTES
        ),
        "continuous_loss_weights": list(continuous_loss_weights_for_resolution(res)),
        "mae_floor_atr_mult": 0.25,
        "vol_regime_quantiles": [0.25, 0.5, 0.75],
        "window_len": 128,
        "stride": 8,
        "train_frac": 0.65,
        "val_frac": 0.15,
        "embargo_bars": path_horizon,
        "batch_size": 128,
        "epochs": 120,
        "lr": 1e-4,
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "dropout": 0.25,
        "weight_decay": 1e-2,
        "early_stop_patience": 12,
        "early_stopping_enabled": True,
        "min_derivatives_coverage": 0.5,
        "derivatives_coverage_warn": 0.9,
        "min_export_vol_corr": MIN_EXPORT_VOL_CORR.get(res, 0.08),
        "default_threshold": 0.005,
        "seed": 42,
    }

def max_label_horizon_bars(
    path_label_horizon_bars: int = PATH_LABEL_HORIZON_BARS,
) -> int:
    """Maximum forward bars across all training labels."""
    return int(path_label_horizon_bars)

def scale_period(period: int, resolution_minutes: int, *, base_minutes: int = 5) -> int:
    """Scale indicator lookback to preserve wall-clock semantics across TFs."""
    return max(1, int(round(period * resolution_minutes / base_minutes)))

def feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """Continuous feature columns for a TF bundle (5m appends closed HTF context)."""
    res = resolution.strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution: {resolution!r}")
    if res == "5m":
        return FEATURE_COLS + HTF_STRUCTURE_COLS
    return FEATURE_COLS

def fusion_native_feature_cols() -> Tuple[str, ...]:
    """Native-TF columns for the fused model (no resampled HTF context)."""
    return FEATURE_COLS

def fusion_feature_cols_v14() -> Tuple[str, ...]:
    """Disjoint v14 continuous columns (no pattern flags, no duplicate scales)."""
    cols: Tuple[str, ...] = tuple(
        name
        for group in V14_FEATURE_GROUP_ORDER
        for name in V14_FEATURE_GROUPS[group]
    )
    return cols

def fusion_feature_groups_v14() -> Dict[str, Tuple[str, ...]]:
    """Copy of the disjoint v14 group map."""
    return {str(key): tuple(val) for key, val in V14_FEATURE_GROUPS.items()}

def v7_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """Input columns: native features plus causal chart_pattern_id."""
    return feature_cols_for_resolution(resolution) + (CHART_PATTERN_COL,)

def v8_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """v8 5m input columns (native features plus causal chart_pattern_id)."""
    return v7_feature_cols_for_resolution(resolution)

def v9_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:
    """v9 5m input columns: geometry only; chart_pattern_id is a target head."""
    return feature_cols_for_resolution(resolution)

def onnx_output_names_for_contract(
    contract_version: str,
    *,
    resolution: str = "5m",
) -> Tuple[str, ...]:
    """ONNX head names for a bundle contract."""
    ver = str(contract_version or "").strip()
    res = resolution.strip().lower()
    if ver == FEATURE_CONTRACT_VERSION_V15:
        return ONNX_OUTPUT_NAMES_V15
    if ver in (FEATURE_CONTRACT_VERSION_V13, FEATURE_CONTRACT_VERSION_V14):
        return ONNX_OUTPUT_NAMES_V13
    if ver == FEATURE_CONTRACT_VERSION_V12:
        return ONNX_OUTPUT_NAMES_V12
    if ver == FEATURE_CONTRACT_VERSION_V11 or res == "mtf_fusion":
        return ONNX_OUTPUT_NAMES_V11
    if ver == FEATURE_CONTRACT_VERSION_V10:
        return ONNX_OUTPUT_NAMES_V10
    if ver in (FEATURE_CONTRACT_VERSION, FEATURE_CONTRACT_VERSION_V9):
        if res == "5m":
            return ONNX_OUTPUT_NAMES_V9
        return ONNX_OUTPUT_NAMES_V6
    if ver == FEATURE_CONTRACT_VERSION_V8:
        if res == "5m":
            return ONNX_OUTPUT_NAMES_V8
        return ONNX_OUTPUT_NAMES_V6
    if ver == FEATURE_CONTRACT_VERSION_V7:
        if res == "5m":
            return ONNX_OUTPUT_NAMES_V7
        return ONNX_OUTPUT_NAMES_V7_BASE
    return ONNX_OUTPUT_NAMES_V6

def v8_future_leak_cols() -> frozenset:
    """Label/target columns that must never appear in 5m v8 model inputs."""
    leaked = set(V8_CONTINUOUS_LABEL_COLS)
    leaked.update(HORIZON_DIR_COLS)
    leaked.update(HORIZON_STRUCTURE_COLS)
    leaked.update(
        {
            VOLUME_STATE_COL,
            NEXT_DIRECTION_COL,
            NEXT_BODY_COL,
            NEXT_WICK_COL,
            NEXT_RANGE_COL,
            FUTURE_CANDLE_COL,
            "mfe",
            "mae",
            "future_volatility",
            "pattern_validates",
            *HORIZON_DIR_COLS_V7,
        }
    )
    return frozenset(leaked)

def expected_direction_from_chart_pattern(pattern_id: int) -> int:
    """Map chart pattern to expected next-candle direction (0/1/2). Neutral if none."""
    pid = int(pattern_id)
    if pid in (1, 5, 7):  # FLAG_BULL, DOUBLE_BOTTOM, BREAKOUT (unsigned handled elsewhere)
        return 2
    if pid in (2, 4, 8):  # FLAG_BEAR, DOUBLE_TOP, FAILED_BREAK
        return 0
    return 1

def default_research_config() -> Dict[str, Any]:
    """Colab research-pipeline defaults (5m multi-horizon path)."""
    return {
        "symbol": "BTCUSD",
        "resolution": "5m",
        "resolution_minutes": 5,
        "base_timeframe": "5m",
        "context_timeframes": ["15m", "30m", "1h", "2h"],
        "sequence_length": 64,
        "prediction_horizon": 1,
        "horizon_bars": list(HORIZON_BARS_5M),
        "train_ratio": 0.70,
        "validation_ratio": 0.15,
        "test_ratio": 0.15,
        "batch_size": 256,
        "epochs": 50,
        "learning_rate": 1e-4,
        "weight_decay": 1e-4,
        "dropout": 0.15,
        "early_stopping_patience": 8,
        "seed": 42,
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "stride": 4,
        "scaler_mode": "train_fit",
        "gate_chart_volume": False,
        "path_label_horizon_bars": MAX_V8_HORIZON_BARS,
        "embargo_bars": MAX_V8_HORIZON_BARS,
        "mae_floor_atr_mult": 0.25,
        "atr_period": 14,
        "history_days": 900,
        "base_url": "https://api.india.delta.exchange",
        "loss_weights": dict(V9_STRUCTURE_LOSS_WEIGHTS),
        "run_optuna": False,
        "optuna_trials": 0,
        "run_shap": False,
        "run_ablations": False,
        "run_walk_forward": False,
        "walk_forward_folds": 3,
    }

def default_fusion_training_config() -> Dict[str, Any]:
    """Defaults for the single multi-TF fusion trainer."""
    return {
        "symbol": "BTCUSD",
        "resolutions": list(FUSION_INPUT_RESOLUTIONS),
        "horizon_keys": list(LABEL_V2_HORIZON_KEYS),
        "horizon_bars": list(LABEL_V2_HORIZON_BARS_5M),
        "window_len": FUSION_WINDOW_LEN,
        "window_lens": fusion_window_lens(),
        "target_window_minutes": FUSION_TARGET_WINDOW_MINUTES,
        "stride": 4,
        "train_frac": 0.70,
        "val_frac": 0.15,
        "embargo_bars": FUSION_EMBARGO_BARS,
        "batch_size": 128,
        "epochs": 40,
        "lr": 1e-4,
        "weight_decay": 1e-3,
        "dropout": 0.30,
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "early_stop_patience": 8,
        "label_smoothing": 0.05,
        "horizon_loss_weights": [1.0, 1.0, 1.0, 1.0],
        "lr_schedule": "plateau",
        "seed": 42,
        "history_days": 900,
        "base_url": "https://api.india.delta.exchange",
        "atr_period": 14,
        "run_optuna": False,
        "optuna_trials": 3,
        "optuna_trial_epochs": 4,
        "optuna_refresh": False,
        "windows_in_ram": True,
        "pin_memory": True,
        "dataloader_workers": 2,
        "prefetch_factor": 4,
        "amp": True,
        "run_walk_forward": True,
        "walk_forward_folds": 3,
        "walk_forward_embargo": FUSION_EMBARGO_BARS,
        "min_probability": FUSION_MIN_PROBABILITY,
        "high_balanced_acc": LABEL_V2_HIGH_BALANCED_ACC,
        "high_max_ece": FUSION_HIGH_MAX_ECE,
        "medium_balanced_acc": LABEL_V2_MEDIUM_BALANCED_ACC,
        "n_classes": LABEL_V2_DIRECTION_CARDINALITY,
        "label_scheme": "label_v2",
        "experiment": "later_direction_only",
        "use_class_weights": False,
        "path_loss_weights": dict(LABEL_V2_DIRECTION_ONLY_PATH_LOSS_WEIGHTS),
        "label_v2_theta": dict(LABEL_V2_THETA_FROZEN),
        "run_shap": False,
        "shap_background": 32,
        "shap_explain_n": 64,
        "use_state_embeddings": True,
        "run_leave_one_group_out": False,
    }

def ablation_feature_groups(resolution: str = "5m") -> Dict[str, Tuple[str, ...]]:
    """Nested feature sets A-F for out-of-sample ablation."""
    all_cols = v9_feature_cols_for_resolution(resolution)
    ohlcv = ("ret_1", "rv_16", "rv_96", "hour_sin", "hour_cos", "dow_sin", "dow_cos")
    geometry = ohlcv + (
        "body_ratio",
        "upper_wick_ratio",
        "lower_wick_ratio",
        "close_loc",
        "range_atr",
        "body_atr",
        "gap_atr",
        "inside_bar",
        "outside_bar",
        "engulf_score",
    )
    trend = geometry + (
        "ema50_dist_pct",
        "macd_hist",
        "rsi_14",
        "adx_14",
        "price_vs_ema9_atr",
        "price_vs_ema21_atr",
        "price_vs_ema50_atr",
        "price_vs_ema200_atr",
        "ema9_vs_21_atr",
        "ema21_vs_50_atr",
        "ema50_vs_200_atr",
        "ema21_slope_atr",
        "ema50_slope_atr",
        "trend_efficiency",
        "displacement_atr",
        "pct_with_trend",
    )
    structure = trend + (
        "hh_count",
        "hl_count",
        "lh_count",
        "ll_count",
        "structure_bias",
        "last_swing_dir",
        "bars_since_swing",
        "swing_amp_atr",
        "dist_to_support_atr",
        "dist_to_resistance_atr",
        "support_touch_count",
        "resistance_touch_count",
        "range_width_atr",
        "dist_to_resistance_pct",
        "dist_to_support_pct",
    )
    chart = structure + (
        "peak_diff_atr",
        "trough_diff_atr",
        "peak_sep_bars",
        "trough_sep_bars",
        "dist_neck_atr",
        "high_slope_atr",
        "low_slope_atr",
        "convergence",
        "width_now_atr",
        "pole_disp_atr",
        "flag_width_atr",
        "flag_slope_atr",
        "breakout_size_atr",
        "breakout_vol_ratio",
        "pre_breakout_comp",
        "bars_since_breakout",
        "retest_dist_atr",
        "failed_break",
    )
    return {
        "A": tuple(c for c in ohlcv if c in all_cols),
        "B": tuple(c for c in geometry if c in all_cols),
        "C": tuple(c for c in trend if c in all_cols),
        "D": tuple(c for c in structure if c in all_cols),
        "E": tuple(c for c in chart if c in all_cols),
        "F": all_cols,
    }


## Derivatives features

In [ ]:
"""Funding and OI derivative features scaled per resolution."""

from __future__ import annotations

import numpy as np
import pandas as pd

_EPS = 1e-9

def compute_funding_derivatives(
    fund_rate: pd.Series,
    ret_2: pd.Series,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """Funding z-score, momentum, and rate-of-change on native TF bars."""
    z_window = scale_period(16, resolution_minutes)
    roc_diff = max(1, scale_period(1, resolution_minutes))
    fz_mean = fund_rate.rolling(z_window, min_periods=5).mean()
    fz_std = fund_rate.rolling(z_window, min_periods=5).std().replace(0, _EPS)
    funding_zscore = ((fund_rate - fz_mean) / fz_std).fillna(0.0).clip(-4.0, 4.0)
    funding_mom = funding_zscore * ret_2
    diff = fund_rate.diff(roc_diff)
    roc_mu = diff.rolling(z_window, min_periods=5).mean()
    roc_std = diff.rolling(z_window, min_periods=5).std().replace(0, _EPS)
    funding_rate_roc = ((diff - roc_mu) / roc_std).fillna(0.0).clip(-4.0, 4.0)
    return pd.DataFrame(
        {
            "funding_zscore": funding_zscore,
            "funding_mom": funding_mom.fillna(0.0),
            "funding_rate_roc": funding_rate_roc,
        }
    )

def compute_oi_derivatives(
    primary: pd.DataFrame,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """OI-derived features on native TF bars."""
    n = len(primary)
    zero = pd.DataFrame(
        {
            "oi_change_2": np.zeros(n),
            "oi_delta_z": np.zeros(n),
            "oi_price_divergence": np.zeros(n),
            "oi_acceleration": np.zeros(n),
            "oi_zscore": np.zeros(n),
        },
        index=primary.index,
    )
    if "open_interest" not in primary.columns:
        return zero

    z_window = scale_period(16, resolution_minutes)
    change_window = max(1, scale_period(2, resolution_minutes))

    oi_s = primary["open_interest"].astype(float)
    if oi_s.notna().sum() == 0 or float(oi_s.max()) < _EPS:
        return zero

    oi_mu = oi_s.rolling(z_window, min_periods=max(2, z_window // 4)).mean()
    oi_std = oi_s.rolling(z_window, min_periods=max(2, z_window // 4)).std().clip(
        lower=_EPS
    )
    oi_zscore = ((oi_s - oi_mu) / oi_std).fillna(0.0).clip(-4.0, 4.0)

    oi_lagged = oi_s.shift(change_window)
    oi_change_2 = (
        ((oi_s - oi_lagged) / (oi_lagged.abs() + _EPS)).fillna(0.0).clip(-0.05, 0.05)
    )

    close_s = primary["close"].astype(float)
    close_lagged = close_s.shift(change_window)
    ret_2 = ((close_s - close_lagged) / (close_lagged.abs() + _EPS)).fillna(0.0)
    oi_price_divergence = (
        np.sign(oi_change_2.values) * -np.sign(ret_2.values)
    ).astype(np.float32)
    oi_acceleration = oi_change_2.diff().fillna(0.0).clip(-0.02, 0.02)

    oi_delta_1 = oi_s.diff(1).fillna(0.0)
    oi_delta_mu = oi_delta_1.rolling(z_window, min_periods=max(2, z_window // 4)).mean()
    oi_delta_std = oi_delta_1.rolling(z_window, min_periods=max(2, z_window // 4)).std().clip(
        lower=_EPS
    )
    oi_delta_z = ((oi_delta_1 - oi_delta_mu) / oi_delta_std).fillna(0.0).clip(-4.0, 4.0)

    return pd.DataFrame(
        {
            "oi_zscore": oi_zscore,
            "oi_change_2": oi_change_2,
            "oi_price_divergence": pd.Series(oi_price_divergence).fillna(0.0),
            "oi_acceleration": oi_acceleration,
            "oi_delta_z": oi_delta_z,
        },
        index=primary.index,
    ).replace([np.inf, -np.inf], 0.0).fillna(0.0)


## Market structure

In [ ]:
"""Causal OHLCV market-structure and chart-geometry features.

All values at bar t use information available at or before t. Swings are ATR
ZigZag confirmations (not future-looking fractals). Donchian/breakout ranges
exclude bar t. Higher-TF context is merged from fully closed resampled bars.
"""

from __future__ import annotations

from collections import deque
from typing import Deque, List, Tuple

import numpy as np
import pandas as pd

_EPS = 1e-9
_ATR_CLIP = 8.0
_SLOPE_CLIP = 2.0
_ZZ_TAU = 1.5
_ZZ_EVENT_M = 6
_ZZ_EPS_ATR = 0.15
_SWING_K = 20
_ER_N = 32
_SLOPE_LAG = 8
_DONCHIAN_N = 32
_RANGE_PCTILE_W = 256
_TOUCH_W = 64
_SR_DELTA_ATR = 0.5
_POLE_BARS = 12
_FLAG_BARS = 16
_BREAKOUT_THRESH = 0.25
_FAILED_BREAK_BARS = 8
_CHANNEL_K = 3
_ATR_SHORT = 14
_ATR_LONG = 56
_HTF_NATIVE_MINUTES = 5

_HTF_NATIVE_MAP = {
    "structure_bias": "structure_bias",
    "trend_efficiency": "trend_efficiency",
    "ema21_slope_atr": "ema21_slope_atr",
    "dist_support_atr": "dist_to_support_atr",
    "dist_resistance_atr": "dist_to_resistance_atr",
    "range_width_atr": "range_width_atr",
}

def _structure_safe_atr(atr: np.ndarray, close: np.ndarray, i: int) -> float:
    val = float(atr[i]) if np.isfinite(atr[i]) else float("nan")
    if not np.isfinite(val) or val <= 0:
        c = float(close[i]) if np.isfinite(close[i]) else 0.0
        return max(abs(c) * 0.01, _EPS)
    return val

def _clip_atr(val: float) -> float:
    if not np.isfinite(val):
        return 0.0
    return float(np.clip(val, -_ATR_CLIP, _ATR_CLIP))

def _ols_slope(x: np.ndarray, y: np.ndarray) -> float:
    n = float(len(x))
    if n < 2:
        return 0.0
    sx = float(x.sum())
    sy = float(y.sum())
    sxy = float((x * y).sum())
    sx2 = float((x * x).sum())
    den = n * sx2 - sx * sx
    if abs(den) < 1e-12:
        return 0.0
    return float((n * sxy - sx * sy) / den)

def _ensure_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:
    out = df.copy()
    if "atr" in out.columns and out["atr"].notna().any():
        return out
    prev_close = out["close"].shift(1)
    tr = pd.concat(
        [
            out["high"] - out["low"],
            (out["high"] - prev_close).abs(),
            (out["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    out["atr"] = tr.rolling(period, min_periods=1).mean()
    return out

def _vectorized_trend_and_ma(out: pd.DataFrame) -> pd.DataFrame:
    """ER, displacement, MA distances/slopes, Donchian compression, flag geometry."""
    close = out["close"].astype(float)
    high = out["high"].astype(float)
    low = out["low"].astype(float)
    volume = out["volume"].astype(float) if "volume" in out.columns else pd.Series(
        0.0, index=out.index
    )
    atr = out["atr"].astype(float) if "atr" in out.columns else pd.Series(
        close.abs() * 0.01, index=out.index
    )
    atr_safe = atr.replace(0, np.nan).fillna(close.abs() * 0.01 + _EPS)

    abs_move = (close - close.shift(_ER_N)).abs()
    path = close.diff().abs().rolling(_ER_N, min_periods=2).sum()
    out["trend_efficiency"] = (abs_move / (path + _EPS)).clip(0.0, 1.0).fillna(0.0)
    out["displacement_atr"] = (abs_move / (atr_safe + _EPS)).clip(
        0.0, _ATR_CLIP
    ).fillna(0.0)

    window_dir = np.sign(close - close.shift(_ER_N))
    up_frac = (close.diff() > 0).astype(float).rolling(_ER_N, min_periods=1).mean()
    pct = np.where(window_dir >= 0, up_frac, 1.0 - up_frac)
    out["pct_with_trend"] = pd.Series(pct, index=out.index).fillna(0.0)

    ema9 = close.ewm(span=9, adjust=False).mean()
    ema21 = close.ewm(span=21, adjust=False).mean()
    ema50 = close.ewm(span=50, adjust=False).mean()
    ema200 = close.ewm(span=200, adjust=False).mean()
    out["price_vs_ema9_atr"] = ((close - ema9) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["price_vs_ema21_atr"] = ((close - ema21) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["price_vs_ema50_atr"] = ((close - ema50) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["price_vs_ema200_atr"] = ((close - ema200) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["ema9_vs_21_atr"] = ((ema9 - ema21) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["ema21_vs_50_atr"] = ((ema21 - ema50) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    out["ema50_vs_200_atr"] = ((ema50 - ema200) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    )
    lag = float(_SLOPE_LAG)
    out["ema21_slope_atr"] = (
        (ema21 - ema21.shift(_SLOPE_LAG)) / (atr_safe * lag + _EPS)
    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)
    out["ema50_slope_atr"] = (
        (ema50 - ema50.shift(_SLOPE_LAG)) / (atr_safe * lag + _EPS)
    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)

    prior_high = high.shift(1)
    prior_low = low.shift(1)
    donch_high = prior_high.rolling(_DONCHIAN_N, min_periods=1).max()
    donch_low = prior_low.rolling(_DONCHIAN_N, min_periods=1).min()
    range_width_atr = ((donch_high - donch_low) / (atr_safe + _EPS)).clip(
        0.0, _ATR_CLIP
    )
    out["range_width_atr"] = range_width_atr.fillna(0.0)
    out["range_width_pctile"] = range_width_atr.rolling(
        _RANGE_PCTILE_W, min_periods=8
    ).rank(pct=True).fillna(0.0)
    atr_short = atr_safe.rolling(_ATR_SHORT, min_periods=1).mean()
    atr_long = atr_safe.rolling(_ATR_LONG, min_periods=1).mean()
    out["atr_contraction"] = (atr_short / (atr_long + _EPS)).clip(0.0, _ATR_CLIP).fillna(
        1.0
    )

    out["breakout_size_atr"] = ((close - donch_high) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    ).fillna(0.0)
    vol_mean = volume.shift(1).rolling(_DONCHIAN_N, min_periods=1).mean()
    out["breakout_vol_ratio"] = (volume / (vol_mean + _EPS)).clip(0.0, _ATR_CLIP).fillna(
        0.0
    )
    out["pre_breakout_comp"] = out["range_width_pctile"].shift(1).fillna(0.0)

    pole_end = close.shift(_FLAG_BARS)
    pole_start = close.shift(_POLE_BARS + _FLAG_BARS)
    out["pole_disp_atr"] = ((pole_end - pole_start) / (atr_safe + _EPS)).clip(
        -_ATR_CLIP, _ATR_CLIP
    ).fillna(0.0)
    flag_high = high.rolling(_FLAG_BARS, min_periods=1).max()
    flag_low = low.rolling(_FLAG_BARS, min_periods=1).min()
    out["flag_width_atr"] = ((flag_high - flag_low) / (atr_safe + _EPS)).clip(
        0.0, _ATR_CLIP
    ).fillna(0.0)
    flag_lag = float(_FLAG_BARS)
    out["flag_slope_atr"] = (
        (close - close.shift(_FLAG_BARS)) / (atr_safe * flag_lag + _EPS)
    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)
    return out

def _zigzag_structure_loop(out: pd.DataFrame) -> pd.DataFrame:
    """Single-pass ATR ZigZag: HH/HL counts, S/R, breakout events, pattern geometry."""
    n = len(out)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    close = out["close"].to_numpy(dtype=np.float64)
    atr = out["atr"].to_numpy(dtype=np.float64) if "atr" in out.columns else np.full(
        n, np.nan
    )
    prior_high = np.roll(high, 1)
    prior_high[0] = high[0]
    prior_low = np.roll(low, 1)
    prior_low[0] = low[0]
    donch_h = np.empty(n, dtype=np.float64)
    donch_l = np.empty(n, dtype=np.float64)
    max_q: Deque[int] = deque()
    min_q: Deque[int] = deque()
    for t in range(n):
        left = t - _DONCHIAN_N
        while max_q and max_q[0] <= left:
            max_q.popleft()
        while min_q and min_q[0] <= left:
            min_q.popleft()
        src_h = prior_high[t]
        src_l = prior_low[t]
        while max_q and prior_high[max_q[-1]] <= src_h:
            max_q.pop()
        while min_q and prior_low[min_q[-1]] >= src_l:
            min_q.pop()
        max_q.append(t)
        min_q.append(t)
        donch_h[t] = prior_high[max_q[0]]
        donch_l[t] = prior_low[min_q[0]]

    hh_count = np.zeros(n, dtype=np.float64)
    hl_count = np.zeros(n, dtype=np.float64)
    lh_count = np.zeros(n, dtype=np.float64)
    ll_count = np.zeros(n, dtype=np.float64)
    structure_bias = np.zeros(n, dtype=np.float64)
    last_swing_dir = np.zeros(n, dtype=np.float64)
    bars_since_swing = np.zeros(n, dtype=np.float64)
    swing_amp_atr = np.zeros(n, dtype=np.float64)
    unconfirmed_ext_atr = np.zeros(n, dtype=np.float64)
    dist_to_support_atr = np.zeros(n, dtype=np.float64)
    dist_to_resistance_atr = np.zeros(n, dtype=np.float64)
    support_touch = np.zeros(n, dtype=np.float64)
    resistance_touch = np.zeros(n, dtype=np.float64)
    bars_since_breakout = np.zeros(n, dtype=np.float64)
    retest_dist_atr = np.zeros(n, dtype=np.float64)
    failed_break = np.zeros(n, dtype=np.float64)
    peak_diff_atr = np.zeros(n, dtype=np.float64)
    trough_diff_atr = np.zeros(n, dtype=np.float64)
    peak_sep_bars = np.zeros(n, dtype=np.float64)
    trough_sep_bars = np.zeros(n, dtype=np.float64)
    dist_neck_atr = np.zeros(n, dtype=np.float64)
    high_slope_atr = np.zeros(n, dtype=np.float64)
    low_slope_atr = np.zeros(n, dtype=np.float64)
    convergence = np.zeros(n, dtype=np.float64)
    width_now_atr = np.zeros(n, dtype=np.float64)

    direction = 1
    e_idx = 0
    e_price = high[0] if n else 0.0
    last_high_price = float("nan")
    last_low_price = float("nan")
    last_high_idx = -1
    last_low_idx = -1
    last_swing_idx = -1
    last_dir = 0.0
    highs: List[Tuple[int, float]] = []
    lows: List[Tuple[int, float]] = []
    events: Deque[str] = deque(maxlen=_ZZ_EVENT_M)
    last_bo_idx = -1
    last_bo_level = float("nan")
    last_bo_side = 0

    for t in range(n):
        atr_t = _structure_safe_atr(atr, close, t)
        thresh = _ZZ_TAU * atr_t
        eps = _ZZ_EPS_ATR * atr_t

        if direction == 1:
            if high[t] >= e_price:
                e_price = high[t]
                e_idx = t
            elif (e_price - low[t]) >= thresh:
                highs.append((e_idx, e_price))
                if len(highs) > _SWING_K:
                    highs = highs[-_SWING_K:]
                if np.isfinite(last_high_price):
                    if e_price > last_high_price + eps:
                        events.append("HH")
                    elif e_price < last_high_price - eps:
                        events.append("LH")
                last_high_price = e_price
                last_high_idx = e_idx
                last_swing_idx = e_idx
                last_dir = 1.0
                direction = -1
                e_price = low[t]
                e_idx = t
        else:
            if low[t] <= e_price:
                e_price = low[t]
                e_idx = t
            elif (high[t] - e_price) >= thresh:
                lows.append((e_idx, e_price))
                if len(lows) > _SWING_K:
                    lows = lows[-_SWING_K:]
                if np.isfinite(last_low_price):
                    if e_price > last_low_price + eps:
                        events.append("HL")
                    elif e_price < last_low_price - eps:
                        events.append("LL")
                last_low_price = e_price
                last_low_idx = e_idx
                last_swing_idx = e_idx
                last_dir = -1.0
                direction = 1
                e_price = high[t]
                e_idx = t

        n_hh = sum(1 for e in events if e == "HH")
        n_hl = sum(1 for e in events if e == "HL")
        n_lh = sum(1 for e in events if e == "LH")
        n_ll = sum(1 for e in events if e == "LL")
        n_ev = max(len(events), 1)
        hh_count[t] = float(n_hh)
        hl_count[t] = float(n_hl)
        lh_count[t] = float(n_lh)
        ll_count[t] = float(n_ll)
        structure_bias[t] = (n_hh + n_hl - n_lh - n_ll) / float(n_ev)
        last_swing_dir[t] = last_dir
        bars_since_swing[t] = float(t - last_swing_idx) if last_swing_idx >= 0 else float(t)
        if np.isfinite(last_high_price) and np.isfinite(last_low_price):
            swing_amp_atr[t] = _clip_atr(abs(last_high_price - last_low_price) / atr_t)
        unconfirmed_ext_atr[t] = _clip_atr((e_price - close[t]) / atr_t)

        swing_lows = [p for _, p in lows if p <= close[t]]
        swing_highs = [p for _, p in highs if p >= close[t]]
        support = max(swing_lows) if swing_lows else float(donch_l[t])
        resistance = min(swing_highs) if swing_highs else float(donch_h[t])
        dist_to_support_atr[t] = _clip_atr((close[t] - support) / atr_t)
        dist_to_resistance_atr[t] = _clip_atr((resistance - close[t]) / atr_t)
        delta = _SR_DELTA_ATR * atr_t
        if low[t] <= support + delta and close[t] >= support - delta:
            support_touch[t] = 1.0
        if high[t] >= resistance - delta and close[t] <= resistance + delta:
            resistance_touch[t] = 1.0

        up_ext = (close[t] - donch_h[t]) / atr_t
        dn_ext = (donch_l[t] - close[t]) / atr_t
        if up_ext >= _BREAKOUT_THRESH:
            last_bo_idx = t
            last_bo_level = float(donch_h[t])
            last_bo_side = 1
        elif dn_ext >= _BREAKOUT_THRESH:
            last_bo_idx = t
            last_bo_level = float(donch_l[t])
            last_bo_side = -1
        if last_bo_idx >= 0:
            bars_since_breakout[t] = float(t - last_bo_idx)
            retest_dist_atr[t] = _clip_atr((close[t] - last_bo_level) / atr_t)
            if last_bo_side > 0 and close[t] < last_bo_level:
                failed_break[t] = 1.0
            elif last_bo_side < 0 and close[t] > last_bo_level:
                failed_break[t] = 1.0
        else:
            bars_since_breakout[t] = float(t)

        if len(highs) >= 2:
            (i1, p1), (i2, p2) = highs[-2], highs[-1]
            peak_diff_atr[t] = _clip_atr((p2 - p1) / atr_t)
            peak_sep_bars[t] = float(i2 - i1)
            between = [lp for li, lp in lows if i1 < li < i2]
            if between:
                neck = min(between)
                dist_neck_atr[t] = _clip_atr((close[t] - neck) / atr_t)
        if len(lows) >= 2:
            (j1, q1), (j2, q2) = lows[-2], lows[-1]
            trough_diff_atr[t] = _clip_atr((q2 - q1) / atr_t)
            trough_sep_bars[t] = float(j2 - j1)

        if len(highs) >= _CHANNEL_K:
            hx = np.array([idx for idx, _ in highs[-_CHANNEL_K:]], dtype=np.float64)
            hy = np.array([px for _, px in highs[-_CHANNEL_K:]], dtype=np.float64)
            hs = _ols_slope(hx, hy)
            high_slope_atr[t] = float(np.clip(hs / atr_t, -_SLOPE_CLIP, _SLOPE_CLIP))
        if len(lows) >= _CHANNEL_K:
            lx = np.array([idx for idx, _ in lows[-_CHANNEL_K:]], dtype=np.float64)
            ly = np.array([px for _, px in lows[-_CHANNEL_K:]], dtype=np.float64)
            ls = _ols_slope(lx, ly)
            low_slope_atr[t] = float(np.clip(ls / atr_t, -_SLOPE_CLIP, _SLOPE_CLIP))
        convergence[t] = float(
            np.clip(high_slope_atr[t] - low_slope_atr[t], -_SLOPE_CLIP, _SLOPE_CLIP)
        )
        if len(highs) >= 1 and len(lows) >= 1:
            h_line = highs[-1][1] + high_slope_atr[t] * atr_t * (t - highs[-1][0])
            l_line = lows[-1][1] + low_slope_atr[t] * atr_t * (t - lows[-1][0])
            width_now_atr[t] = _clip_atr((h_line - l_line) / atr_t)

    touch_sup = pd.Series(support_touch).rolling(_TOUCH_W, min_periods=1).sum()
    touch_res = pd.Series(resistance_touch).rolling(_TOUCH_W, min_periods=1).sum()

    out["hh_count"] = hh_count
    out["hl_count"] = hl_count
    out["lh_count"] = lh_count
    out["ll_count"] = ll_count
    out["structure_bias"] = structure_bias
    out["last_swing_dir"] = last_swing_dir
    out["bars_since_swing"] = bars_since_swing
    out["swing_amp_atr"] = swing_amp_atr
    out["unconfirmed_ext_atr"] = unconfirmed_ext_atr
    out["dist_to_support_atr"] = dist_to_support_atr
    out["dist_to_resistance_atr"] = dist_to_resistance_atr
    out["support_touch_count"] = touch_sup.to_numpy(dtype=np.float64)
    out["resistance_touch_count"] = touch_res.to_numpy(dtype=np.float64)
    out["bars_since_breakout"] = bars_since_breakout
    out["retest_dist_atr"] = retest_dist_atr
    out["failed_break"] = failed_break
    out["peak_diff_atr"] = peak_diff_atr
    out["trough_diff_atr"] = trough_diff_atr
    out["peak_sep_bars"] = peak_sep_bars
    out["trough_sep_bars"] = trough_sep_bars
    out["dist_neck_atr"] = dist_neck_atr
    out["high_slope_atr"] = high_slope_atr
    out["low_slope_atr"] = low_slope_atr
    out["convergence"] = convergence
    out["width_now_atr"] = width_now_atr
    out[CHART_PATTERN_COL] = _chart_pattern_ids(
        failed_break=failed_break,
        bars_since_breakout=bars_since_breakout,
        breakout_size_atr=(
            out["breakout_size_atr"].to_numpy(dtype=np.float64)
            if "breakout_size_atr" in out.columns
            else np.zeros(n)
        ),
        peak_diff_atr=peak_diff_atr,
        trough_diff_atr=trough_diff_atr,
        peak_sep_bars=peak_sep_bars,
        trough_sep_bars=trough_sep_bars,
        dist_neck_atr=dist_neck_atr,
        high_slope_atr=high_slope_atr,
        low_slope_atr=low_slope_atr,
        convergence=convergence,
        width_now_atr=width_now_atr,
        pole_disp_atr=(
            out["pole_disp_atr"].to_numpy(dtype=np.float64)
            if "pole_disp_atr" in out.columns
            else np.zeros(n)
        ),
        flag_width_atr=(
            out["flag_width_atr"].to_numpy(dtype=np.float64)
            if "flag_width_atr" in out.columns
            else np.zeros(n)
        ),
        flag_slope_atr=(
            out["flag_slope_atr"].to_numpy(dtype=np.float64)
            if "flag_slope_atr" in out.columns
            else np.zeros(n)
        ),
        structure_bias=structure_bias,
    )
    return out

def _chart_pattern_ids(
    *,
    failed_break: np.ndarray,
    bars_since_breakout: np.ndarray,
    breakout_size_atr: np.ndarray,
    peak_diff_atr: np.ndarray,
    trough_diff_atr: np.ndarray,
    peak_sep_bars: np.ndarray,
    trough_sep_bars: np.ndarray,
    dist_neck_atr: np.ndarray,
    high_slope_atr: np.ndarray,
    low_slope_atr: np.ndarray,
    convergence: np.ndarray,
    width_now_atr: np.ndarray,
    pole_disp_atr: np.ndarray,
    flag_width_atr: np.ndarray,
    flag_slope_atr: np.ndarray,
    structure_bias: np.ndarray,
) -> np.ndarray:
    """Causal discrete chart-pattern id at bar t (0=NONE ... 8=FAILED_BREAK).

    FAILED_BREAK is a rising-edge event inside ``_FAILED_BREAK_BARS`` of the
    last Donchian breakout. It does not overwrite FLAG/TRIANGLE/DOUBLE/CHANNEL
    and does not stay on for the whole post-breakout regime. BREAKOUT is the
    event bar only and also does not overwrite those named geometries.
    """
    _ = breakout_size_atr
    n = int(failed_break.shape[0])
    ids = np.zeros(n, dtype=np.int64)
    channel = (np.abs(high_slope_atr - low_slope_atr) < 0.15) & (width_now_atr > 0.6)
    triangle = (np.abs(convergence) >= 0.08) & (width_now_atr < 2.5)
    double_top = (
        (np.abs(peak_diff_atr) < 0.45)
        & (peak_sep_bars >= 4)
        & (dist_neck_atr > 0.15)
    )
    double_bottom = (
        (np.abs(trough_diff_atr) < 0.45)
        & (trough_sep_bars >= 4)
        & (dist_neck_atr < -0.15)
    )
    flag_bull = (
        (pole_disp_atr > 1.0)
        & (flag_width_atr < 2.0)
        & (flag_slope_atr < 0.0)
        & (structure_bias > 0.15)
    )
    flag_bear = (
        (pole_disp_atr < -1.0)
        & (flag_width_atr < 2.0)
        & (flag_slope_atr > 0.0)
        & (structure_bias < -0.15)
    )
    # Event bar only. abs(size vs Donchian high) is not a two-sided breakout
    # and was tagging most 5m bars as BREAKOUT.
    fresh_break = bars_since_breakout <= 0.5
    failed_now = (failed_break >= 0.5) & (
        bars_since_breakout <= float(_FAILED_BREAK_BARS)
    )
    failed_prev = np.empty_like(failed_now)
    failed_prev[0] = False
    if n > 1:
        failed_prev[1:] = failed_now[:-1]
    failed_pulse = failed_now & ~failed_prev
    ids = np.where(channel, 6, ids)
    ids = np.where(triangle, 3, ids)
    ids = np.where(flag_bull, 1, ids)
    ids = np.where(flag_bear, 2, ids)
    ids = np.where(double_bottom, 5, ids)
    ids = np.where(double_top, 4, ids)
    named = np.isin(ids, [1, 2, 3, 4, 5, 6])
    ids = np.where(fresh_break & ~named, 7, ids)
    ids = np.where(failed_pulse & ~named, 8, ids)
    return ids

def add_market_structure_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add causal native-TF structure, trend, S/R, breakout, and geometry columns.

    Args:
        df: OHLCV frame with atr (computed if missing).

    Returns:
        Same frame with ``NATIVE_STRUCTURE_COLS`` assigned and finite-filled.
    """
    out = _ensure_atr(df)
    out = _vectorized_trend_and_ma(out)
    out = _zigzag_structure_loop(out)
    for col in NATIVE_STRUCTURE_COLS:
        if col not in out.columns:
            out[col] = 0.0
        out[col] = out[col].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if CHART_PATTERN_COL not in out.columns:
        out[CHART_PATTERN_COL] = 0
    out[CHART_PATTERN_COL] = (
        pd.to_numeric(out[CHART_PATTERN_COL], errors="coerce").fillna(0).astype(np.int64)
    )
    return out

def _resample_closed_ohlcv(
    df: pd.DataFrame,
    htf_minutes: int,
    native_minutes: int = _HTF_NATIVE_MINUTES,
) -> pd.DataFrame:
    """Resample native bars to a higher TF, keeping only complete buckets."""
    expected = max(1, int(round(htf_minutes / native_minutes)))
    src = df[["time", "open", "high", "low", "close"]].copy()
    if "volume" in df.columns:
        src["volume"] = df["volume"]
    else:
        src["volume"] = 0.0
    src["time"] = pd.to_datetime(src["time"], utc=True)
    grouped = src.set_index("time")
    rule = f"{int(htf_minutes)}min"
    agg = grouped.resample(rule, label="right", closed="right").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample(rule, label="right", closed="right").count()
    complete = agg.loc[counts >= expected].dropna(subset=["open", "high", "low", "close"])
    if complete.empty:
        return complete
    complete = complete.reset_index()
    return complete

def add_htf_structure_features(df: pd.DataFrame) -> pd.DataFrame:
    """Merge closed 15m/30m/1h/2h structure onto a 5m frame (backward asof)."""
    out = df.copy()
    out["time"] = pd.to_datetime(out["time"], utc=True)
    out = out.sort_values("time").reset_index(drop=True)

    for tf_name in HTF_SOURCE_TFS:
        minutes = RESOLUTION_MINUTES[tf_name]
        htf_raw = _resample_closed_ohlcv(out, minutes, _HTF_NATIVE_MINUTES)
        prefixed = {f"htf_{tf_name}_{field}": 0.0 for field in HTF_FEATURE_FIELDS}
        if htf_raw.empty:
            for col, val in prefixed.items():
                out[col] = val
            continue
        htf_feat = add_market_structure_features(_ensure_atr(htf_raw))
        merge_cols = {"time": htf_feat["time"]}
        for field in HTF_FEATURE_FIELDS:
            src = _HTF_NATIVE_MAP[field]
            merge_cols[f"htf_{tf_name}_{field}"] = htf_feat[src].to_numpy(dtype=np.float64)
        htf_df = pd.DataFrame(merge_cols).sort_values("time")
        out = pd.merge_asof(
            out.sort_values("time"),
            htf_df,
            on="time",
            direction="backward",
        )

    for col in HTF_STRUCTURE_COLS:
        if col not in out.columns:
            out[col] = 0.0
        out[col] = out[col].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return out


## Feature engineering

In [ ]:
"""Causal per-TF feature engineering for BTCUSD transformers."""

from __future__ import annotations

from typing import Optional, Sequence

import numpy as np
import pandas as pd

WICK_NEGLIGIBLE = 0.05
WICK_BALANCE_MAX = 0.25
FLAT_ATR_MULT = 1e-4
_RATIO_COLS = ("body_ratio", "upper_wick_ratio", "lower_wick_ratio")
_STRUCTURE_EPS = 1e-9
_ATR_NORM_CLIP = 8.0

def assemble_raw_frame(
    df: pd.DataFrame,
    *,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """Merge OHLCV with funding/OI using the same path as live inference."""
    out = df.copy()
    if "timestamp" in out.columns and "time" not in out.columns:
        out["time"] = pd.to_datetime(out["timestamp"], utc=True)
    elif "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"], utc=True)
    else:
        raise ValueError("OHLCV frame requires timestamp or time column")

    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"OHLCV frame missing columns: {missing}")

    out = out.sort_values("time").reset_index(drop=True)

    if funding_df is not None and not funding_df.empty:
        fund = funding_df.copy()
        if "timestamp" in fund.columns:
            fund["time"] = pd.to_datetime(fund["timestamp"], utc=True)
        rate_col = "funding_rate" if "funding_rate" in fund.columns else "close"
        fund = fund[["time", rate_col]].rename(columns={rate_col: "funding_rate"})
        out = pd.merge_asof(
            out.sort_values("time"),
            fund.sort_values("time"),
            on="time",
            direction="backward",
        )
    elif "funding_rate" not in out.columns:
        out["funding_rate"] = np.nan

    if oi_df is not None and not oi_df.empty:
        oi = oi_df.copy()
        if "timestamp" in oi.columns:
            oi["time"] = pd.to_datetime(oi["timestamp"], utc=True)
        oi_col = None
        for candidate in ("open_interest", "oi_contracts", "close"):
            if candidate in oi.columns:
                oi_col = candidate
                break
        if oi_col is not None:
            oi = oi[["time", oi_col]].rename(columns={oi_col: "open_interest"})
            out = pd.merge_asof(
                out.sort_values("time"),
                oi.sort_values("time"),
                on="time",
                direction="backward",
            )
    elif "open_interest" not in out.columns:
        out["open_interest"] = np.nan

    if "funding_rate" in out.columns:
        out["funding_rate"] = out["funding_rate"].ffill()
    if "open_interest" in out.columns:
        out["open_interest"] = out["open_interest"].ffill()

    return out

def prepare_raw_frame(
    df: pd.DataFrame,
    *,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """Alias for assemble_raw_frame (agent inference entry point)."""
    return assemble_raw_frame(df, funding_df=funding_df, oi_df=oi_df)

def classify_candle_shape(out: pd.DataFrame, *, sr_window: int) -> pd.Series:
    """Assign mutually exclusive single-bar candle class ids 0..12.

    Uses bar-i OHLC geometry plus rolling body-size quantiles up to bar i.
    First matching ``np.select`` condition wins.

    Args:
        out: Frame with open/high/low/close, atr, and wick/body ratio columns.
        sr_window: Rolling lookback for doji/marubozu quantiles.

    Returns:
        int64 Series of class ids aligned with ``out.index``.
    """
    rng_raw = (out["high"] - out["low"]).astype(float)
    atr = out["atr"] if "atr" in out.columns else pd.Series(0.0, index=out.index)
    is_flat = (rng_raw.fillna(0) <= 0) | (
        rng_raw.fillna(0) < atr.fillna(0) * FLAT_ATR_MULT
    )

    body_abs = out["body_ratio"].abs()
    upper = out["upper_wick_ratio"]
    lower = out["lower_wick_ratio"]
    doji_thresh = body_abs.rolling(sr_window, min_periods=20).quantile(0.15)
    marubozu_thresh = body_abs.rolling(sr_window, min_periods=20).quantile(0.85)
    body_median = body_abs.rolling(sr_window, min_periods=20).median()

    wick_sum = upper + lower
    wick_imbalance = (upper - lower).abs() / (wick_sum + 1e-9)
    cond_wicks_balanced = (wick_sum > 2 * WICK_NEGLIGIBLE) & (
        wick_imbalance < WICK_BALANCE_MAX
    )

    cond_doji = ~is_flat & (body_abs < doji_thresh)
    cond_dragonfly = (
        cond_doji
        & (upper < WICK_NEGLIGIBLE)
        & (lower > 2 * WICK_NEGLIGIBLE)
    )
    cond_gravestone = (
        cond_doji
        & (lower < WICK_NEGLIGIBLE)
        & (upper > 2 * WICK_NEGLIGIBLE)
    )
    cond_doji_standard = cond_doji
    cond_marubozu_bull = (
        ~is_flat
        & ~cond_doji
        & (out["body_ratio"] > marubozu_thresh)
        & (upper < WICK_NEGLIGIBLE)
        & (lower < WICK_NEGLIGIBLE)
    )
    cond_marubozu_bear = (
        ~is_flat
        & ~cond_doji
        & (-out["body_ratio"] > marubozu_thresh)
        & (upper < WICK_NEGLIGIBLE)
        & (lower < WICK_NEGLIGIBLE)
    )
    cond_hammer = (
        ~is_flat
        & ~cond_doji
        & (lower > 2 * body_abs)
        & (upper < body_abs)
    )
    cond_inv_hammer = (
        ~is_flat
        & ~cond_doji
        & (upper > 2 * body_abs)
        & (lower < body_abs)
    )
    cond_spinning = (
        ~is_flat
        & ~cond_doji
        & (body_abs < body_median)
        & cond_wicks_balanced
    )
    cond_belt_bull = (
        ~is_flat
        & ~cond_doji
        & (out["close"] > out["open"])
        & (lower < WICK_NEGLIGIBLE)
        & (upper > WICK_NEGLIGIBLE)
    )
    cond_belt_bear = (
        ~is_flat
        & ~cond_doji
        & (out["close"] < out["open"])
        & (upper < WICK_NEGLIGIBLE)
        & (lower > WICK_NEGLIGIBLE)
    )
    cond_standard_bull = ~is_flat & (out["close"] > out["open"])
    cond_standard_bear = ~is_flat & (out["close"] < out["open"])

    conditions = [
        is_flat,
        cond_dragonfly,
        cond_gravestone,
        cond_doji_standard,
        cond_marubozu_bull,
        cond_marubozu_bear,
        cond_hammer,
        cond_inv_hammer,
        cond_spinning,
        cond_belt_bull,
        cond_belt_bear,
        cond_standard_bull,
        cond_standard_bear,
    ]
    choices = list(range(CANDLE_CLASS_CARDINALITY))
    ids = np.select(conditions, choices, default=3).astype(np.int64)
    return pd.Series(ids, index=out.index, dtype="int64")

def summarize_candle_class_distribution(
    feat_df: pd.DataFrame,
    *,
    high_frac: float = 0.40,
    low_frac: float = 0.001,
) -> pd.Series:
    """Return class fractions and print warnings for extreme imbalance.

    Args:
        feat_df: Feature frame containing ``candle_class_id``.
        high_frac: Warn if any class exceeds this share of bars.
        low_frac: Warn if any present class is below this share.

    Returns:
        Normalized value counts indexed by class id.
    """
    counts = feat_df[CANDLE_CLASS_COL].value_counts(normalize=True).sort_index()
    for cid, frac in counts.items():
        if float(frac) > high_frac or float(frac) < low_frac:
            print(
                f"WARNING: candle class {int(cid)} fraction {float(frac):.4f} "
                f"(thresholds {low_frac:.4f} / {high_frac:.2f})"
            )
    return counts

def add_candle_structure_features(out: pd.DataFrame) -> pd.DataFrame:
    """Add causal bar-geometry and bar-to-bar relation features.

    All columns use OHLCV and ATR at or before bar t. Shift-based values on
    the first bar fill to 0.

    Args:
        out: Frame with open/high/low/close and atr.

    Returns:
        The same frame with seven structure columns assigned.
    """
    rng_raw = (out["high"] - out["low"]).astype(float)
    atr = out["atr"] if "atr" in out.columns else pd.Series(0.0, index=out.index)
    is_flat = rng_raw.fillna(0) <= 0

    close_loc = (out["close"] - out["low"]) / (rng_raw + _STRUCTURE_EPS)
    out["close_loc"] = close_loc.where(~is_flat, 0.5).fillna(0.5)

    out["range_atr"] = (rng_raw / (atr + _STRUCTURE_EPS)).clip(
        -_ATR_NORM_CLIP, _ATR_NORM_CLIP
    )
    out["body_atr"] = (
        (out["close"] - out["open"]).abs() / (atr + _STRUCTURE_EPS)
    ).clip(-_ATR_NORM_CLIP, _ATR_NORM_CLIP)
    out["gap_atr"] = (
        (out["open"] - out["close"].shift(1)) / (atr + _STRUCTURE_EPS)
    ).clip(-_ATR_NORM_CLIP, _ATR_NORM_CLIP)

    prev_high = out["high"].shift(1)
    prev_low = out["low"].shift(1)
    out["inside_bar"] = (
        (out["high"] <= prev_high) & (out["low"] >= prev_low)
    ).astype(np.float32)
    out["outside_bar"] = (
        (out["high"] >= prev_high) & (out["low"] <= prev_low)
    ).astype(np.float32)

    prior_body = out["close"].shift(1) - out["open"].shift(1)
    cur_body = out["close"] - out["open"]
    opposite = np.sign(cur_body) != np.sign(prior_body)
    prior_nonzero = prior_body.abs() > _STRUCTURE_EPS
    penetration = (out["close"] - out["open"].shift(1)) / (
        prior_body.abs() + _STRUCTURE_EPS
    )
    out["engulf_score"] = (
        penetration.clip(-2.0, 2.0)
        * np.sign(cur_body)
        * opposite.astype(np.float64)
        * prior_nonzero.astype(np.float64)
    )

    shift_fill_cols = ("gap_atr", "inside_bar", "outside_bar", "engulf_score")
    for col in shift_fill_cols:
        out[col] = out[col].fillna(0.0)
    return out

def add_features(
    df: pd.DataFrame,
    *,
    resolution_minutes: int = 15,
    atr_period: int = 14,
    rsi_period: int = 14,
    adx_period: int = 14,
    ema_period: int = 50,
    macd_fast: int = 12,
    macd_slow: int = 26,
    macd_signal: int = 9,
    include_htf: bool = True,
) -> pd.DataFrame:
    """Compute causal features on a native TF grid.

    Args:
        include_htf: When True (default), 5m frames also merge resampled HTF
            structure. The fused multi-TF model passes False and encodes each
            timeframe from independently sampled OHLCV instead.
    """
    out = df.copy()
    rv_short = scale_period(16, resolution_minutes)
    rv_long = scale_period(96, resolution_minutes)
    sr_window = scale_period(96, resolution_minutes)
    obv_window = scale_period(96, resolution_minutes)
    vol_window = scale_period(96, resolution_minutes)
    ret2_bars = max(1, scale_period(2, resolution_minutes))

    atr_period = scale_period(atr_period, resolution_minutes)
    rsi_period = scale_period(rsi_period, resolution_minutes)
    adx_period = scale_period(adx_period, resolution_minutes)
    ema_period = scale_period(ema_period, resolution_minutes)
    macd_fast = scale_period(macd_fast, resolution_minutes)
    macd_slow = scale_period(macd_slow, resolution_minutes)
    macd_signal = scale_period(macd_signal, resolution_minutes)

    out["ret_1"] = np.log(out["close"] / out["close"].shift(1))

    prev_close = out["close"].shift(1)
    tr = pd.concat(
        [
            out["high"] - out["low"],
            (out["high"] - prev_close).abs(),
            (out["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    out["atr"] = tr.rolling(atr_period).mean()

    out["rv_16"] = out["ret_1"].rolling(rv_short).std()
    out["rv_96"] = out["ret_1"].rolling(rv_long).std()

    out["ema50"] = out["close"].ewm(span=ema_period, adjust=False).mean()
    out["ema50_dist_pct"] = (out["close"] - out["ema50"]) / out["ema50"]

    ema_fast_s = out["close"].ewm(span=macd_fast, adjust=False).mean()
    ema_slow_s = out["close"].ewm(span=macd_slow, adjust=False).mean()
    macd_line = ema_fast_s - ema_slow_s
    macd_signal_line = macd_line.ewm(span=macd_signal, adjust=False).mean()
    out["macd_hist"] = macd_line - macd_signal_line

    delta = out["close"].diff()
    gain = delta.clip(lower=0).rolling(rsi_period).mean()
    loss = (-delta.clip(upper=0)).rolling(rsi_period).mean()
    rs = gain / (loss + 1e-9)
    out["rsi_14"] = 100 - (100 / (1 + rs))

    up_move = out["high"].diff()
    down_move = -out["low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    atr_for_di = tr.rolling(adx_period).mean()
    plus_di = 100 * pd.Series(plus_dm, index=out.index).rolling(adx_period).mean() / (
        atr_for_di + 1e-9
    )
    minus_di = 100 * pd.Series(minus_dm, index=out.index).rolling(adx_period).mean() / (
        atr_for_di + 1e-9
    )
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-9)
    out["adx_14"] = dx.rolling(adx_period).mean()

    obv_raw = (np.sign(out["close"].diff()) * out["volume"]).fillna(0).cumsum()
    out["obv_z"] = (obv_raw - obv_raw.rolling(obv_window).mean()) / (
        obv_raw.rolling(obv_window).std() + 1e-9
    )

    out["vol_z"] = (out["volume"] - out["volume"].rolling(vol_window).mean()) / (
        out["volume"].rolling(vol_window).std() + 1e-9
    )

    rng_raw = (out["high"] - out["low"]).astype(float)
    is_flat = (rng_raw.fillna(0) <= 0) | (
        rng_raw.fillna(0) < out["atr"].fillna(0) * FLAT_ATR_MULT
    )
    rng = rng_raw.replace(0, np.nan)
    out["body_ratio"] = (out["close"] - out["open"]) / rng
    out["upper_wick_ratio"] = (
        out["high"] - out[["open", "close"]].max(axis=1)
    ) / rng
    out["lower_wick_ratio"] = (
        out[["open", "close"]].min(axis=1) - out["low"]
    ) / rng
    out.loc[is_flat, list(_RATIO_COLS)] = 0.0
    out[list(_RATIO_COLS)] = out[list(_RATIO_COLS)].fillna(0.0)

    out["hour"] = out["time"].dt.hour
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["dow"] = out["time"].dt.dayofweek
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7)

    rolling_high = out["high"].rolling(sr_window).max()
    rolling_low = out["low"].rolling(sr_window).min()
    out["dist_to_resistance_pct"] = (rolling_high - out["close"]) / out["close"]
    out["dist_to_support_pct"] = (out["close"] - rolling_low) / out["close"]

    if "funding_rate" not in out.columns:
        out["funding_rate"] = np.nan

    if "open_interest" in out.columns:
        oi_z_window = scale_period(96, resolution_minutes)
        out["oi_z"] = (out["open_interest"] - out["open_interest"].rolling(oi_z_window).mean()) / (
            out["open_interest"].rolling(oi_z_window).std() + 1e-9
        )
    else:
        out["oi_z"] = np.nan

    ret_2 = out["close"].pct_change(ret2_bars)
    fund_rate = out["funding_rate"].fillna(0.0)
    funding_deriv = compute_funding_derivatives(
        fund_rate, ret_2, resolution_minutes=resolution_minutes
    )
    for col in funding_deriv.columns:
        out[col] = funding_deriv[col].values

    oi_deriv = compute_oi_derivatives(out, resolution_minutes=resolution_minutes)
    out["oi_change_2"] = oi_deriv["oi_change_2"]
    out["oi_delta_z"] = oi_deriv["oi_delta_z"]
    out["oi_price_divergence"] = oi_deriv["oi_price_divergence"]
    out["oi_acceleration"] = oi_deriv["oi_acceleration"]
    out["funding_x_oi"] = out["funding_zscore"] * oi_deriv["oi_zscore"]

    out = add_candle_structure_features(out)
    out[CANDLE_CLASS_COL] = classify_candle_shape(out, sr_window=sr_window)
    out = add_market_structure_features(out)
    if include_htf and int(resolution_minutes) == 5:
        out = add_htf_structure_features(out)

    return out

def build_feature_matrix(
    df: pd.DataFrame,
    *,
    resolution_minutes: int = 15,
    atr_period: int = 14,
    dropna: bool = True,
) -> pd.DataFrame:
    """Return feature columns ready for windowing."""
    feat = add_features(df, resolution_minutes=resolution_minutes, atr_period=atr_period)
    if dropna:
        feat = feat.dropna().reset_index(drop=True)
    return feat

def latest_closed_feature_row(feat_df: pd.DataFrame) -> pd.Series:
    """Closed-bar row used for diagnostics (second-to-last after dropna)."""
    if len(feat_df) < 2:
        raise ValueError("Need at least 2 feature rows for closed-bar semantics")
    return feat_df.iloc[-2]

def validate_feature_columns(
    feat_df: pd.DataFrame,
    *,
    require_finite_closed_bar: bool = False,
    feature_cols: Optional[Sequence[str]] = None,
) -> None:
    cols = tuple(feature_cols) if feature_cols is not None else FEATURE_COLS
    missing = [c for c in cols if c not in feat_df.columns]
    if missing:
        raise ValueError(f"Feature matrix missing columns: {missing}")
    if CANDLE_CLASS_COL not in feat_df.columns:
        raise ValueError(f"Feature matrix missing {CANDLE_CLASS_COL}")
    ids = feat_df[CANDLE_CLASS_COL]
    if ((ids < 0) | (ids > CANDLE_CLASS_CARDINALITY - 1)).any():
        raise ValueError(
            f"{CANDLE_CLASS_COL} out of range [0, {CANDLE_CLASS_CARDINALITY - 1}]"
        )
    if require_finite_closed_bar and len(feat_df) >= 2:
        closed = latest_closed_feature_row(feat_df)
        for col in cols:
            val = closed[col]
            if not np.isfinite(float(val)):
                raise ValueError(f"Non-finite closed-bar value for {col}: {val}")
        cid = int(closed[CANDLE_CLASS_COL])
        if cid < 0 or cid > CANDLE_CLASS_CARDINALITY - 1:
            raise ValueError(f"Closed-bar {CANDLE_CLASS_COL} out of range: {cid}")


## Inference helpers

In [ ]:
"""Window building and label un-standardization for per-TF transformer inference."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, Mapping, Sequence, Tuple

import numpy as np

def load_feature_config(path: Path) -> Dict[str, Any]:
    raw = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(raw, dict):
        raise ValueError(f"{path} must contain a JSON object")
    return raw

def resolve_feature_config(bundle_dir: Path) -> Dict[str, Any]:
    cfg_path = bundle_dir / TRANSFORMER_FEATURE_CONFIG_FILENAME
    if not cfg_path.is_file():
        raise FileNotFoundError(
            f"Missing {TRANSFORMER_FEATURE_CONFIG_FILENAME} in {bundle_dir}"
        )
    return load_feature_config(cfg_path)

def zscore_window(window: np.ndarray) -> np.ndarray:
    """Per-window z-score (matches Colab WindowDataset)."""
    mu = window.mean(axis=0, keepdims=True)
    sd = window.std(axis=0, keepdims=True) + 1e-6
    return ((window - mu) / sd).astype(np.float32)

def build_continuous_window(
    feat_values: np.ndarray,
    *,
    window_len: int,
    feature_cols: Sequence[str] = FEATURE_COLS,
) -> np.ndarray:
    """Build a z-scored (1, window_len, n_features) tensor from continuous cols."""
    if feat_values.shape[0] < window_len:
        raise ValueError(
            f"Need at least {window_len} feature rows, got {feat_values.shape[0]}"
        )
    window = feat_values[-window_len:, :].astype(np.float32)
    if not np.isfinite(window).all():
        raise ValueError("Feature window contains non-finite values")
    normed = zscore_window(window)
    return normed[np.newaxis, :, :]

def build_candle_class_window(
    class_ids: np.ndarray,
    *,
    window_len: int,
    max_class_id: int = CANDLE_CLASS_CARDINALITY - 1,
) -> np.ndarray:
    """Build a raw (1, window_len) int64 tensor of candle class ids."""
    ids = np.asarray(class_ids).reshape(-1)
    if ids.shape[0] < window_len:
        raise ValueError(
            f"Need at least {window_len} candle class rows, got {ids.shape[0]}"
        )
    window = ids[-window_len:].astype(np.int64)
    if np.any((window < 0) | (window > max_class_id)):
        raise ValueError(
            f"{CANDLE_CLASS_COL} out of range [0, {max_class_id}]"
        )
    return window[np.newaxis, :]

def build_inference_window(
    feat_values: np.ndarray,
    *,
    window_len: int,
    feature_cols: Sequence[str] = FEATURE_COLS,
) -> np.ndarray:
    """Build a single (1, window_len, n_features) tensor from feature matrix values."""
    return build_continuous_window(
        feat_values, window_len=window_len, feature_cols=feature_cols
    )

def unstandardize_continuous(
    pred_z: np.ndarray,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    *,
    label_cols: Sequence[str] = CONTINUOUS_LABEL_COLS,
) -> Dict[str, float]:
    """Map standardized ONNX output back to real units."""
    mean = np.asarray(label_mean, dtype=np.float64)
    std = np.asarray(label_std, dtype=np.float64)
    flat = np.asarray(pred_z, dtype=np.float64).reshape(-1)
    if flat.shape[0] != len(label_cols):
        raise ValueError(
            f"Expected {len(label_cols)} continuous outputs, got {flat.shape[0]}"
        )
    real = flat * std + mean
    return {str(col): float(val) for col, val in zip(label_cols, real)}

def softmax(logits: np.ndarray) -> np.ndarray:
    x = np.asarray(logits, dtype=np.float64).reshape(-1)
    x = x - x.max()
    exp = np.exp(x)
    return exp / (exp.sum() + 1e-12)

def parse_regime_prediction(
    regime_logits: np.ndarray,
    regime_names: Mapping[str, str],
) -> Tuple[int, str, Dict[str, float]]:
    probs = softmax(regime_logits)
    idx = int(np.argmax(probs))
    name = regime_names.get(str(idx), regime_names.get(idx, f"CLASS_{idx}"))
    prob_map = {
        regime_names.get(str(i), f"CLASS_{i}"): float(probs[i])
        for i in range(len(probs))
    }
    return idx, str(name), prob_map

def require_onnx_output_names(
    output_names: Sequence[str],
    *,
    contract_version: str | None = None,
    resolution: str = "15m",
) -> None:
    """Reject bundles that lack the heads required by the given contract."""
    have = {str(name) for name in output_names}
    ver = str(contract_version or FEATURE_CONTRACT_VERSION_V6)
    required = onnx_output_names_for_contract(ver, resolution=resolution)
    known = (
        FEATURE_CONTRACT_VERSION,
        FEATURE_CONTRACT_VERSION_V11,
        FEATURE_CONTRACT_VERSION_V10,
        FEATURE_CONTRACT_VERSION_V8,
        FEATURE_CONTRACT_VERSION_V7,
        FEATURE_CONTRACT_VERSION_V6,
    )
    if ver not in known and not required:
        required = ONNX_OUTPUT_NAMES_V6
    missing = [name for name in required if name not in have]
    if missing:
        raise RuntimeError(
            f"ONNX bundle is not {ver}: missing outputs {missing}. Retrain all TFs."
        )

def feature_config_from_training_export(
    *,
    feature_cols: Sequence[str],
    window_len: int,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    q_edges: Sequence[float],
    config: Mapping[str, Any],
    contract_version: str | None = None,
    onnx_output_names: Sequence[str] | None = None,
    continuous_label_cols: Sequence[str] | None = None,
) -> Dict[str, Any]:
    """Build feature_config.json payload written by Colab export cell."""
    version = str(contract_version or FEATURE_CONTRACT_VERSION_V6)
    if onnx_output_names is not None:
        names = list(onnx_output_names)
    elif version == FEATURE_CONTRACT_VERSION_V11:
        names = list(ONNX_OUTPUT_NAMES_V11)
    elif version == FEATURE_CONTRACT_VERSION_V10:
        names = list(ONNX_OUTPUT_NAMES_V10)
    elif version == FEATURE_CONTRACT_VERSION:
        names = list(ONNX_OUTPUT_NAMES_V9)
    elif version == FEATURE_CONTRACT_VERSION_V8:
        names = list(ONNX_OUTPUT_NAMES_V8)
    elif version == FEATURE_CONTRACT_VERSION_V7:
        names = list(ONNX_OUTPUT_NAMES_V7)
    else:
        names = list(ONNX_OUTPUT_NAMES)
    if continuous_label_cols is not None:
        label_cols = list(continuous_label_cols)
    elif version in (FEATURE_CONTRACT_VERSION, FEATURE_CONTRACT_VERSION_V8):
        label_cols = list(V8_CONTINUOUS_LABEL_COLS)
    else:
        label_cols = list(CONTINUOUS_LABEL_COLS)
    return {
        "feature_contract_version": version,
        "feature_cols": list(feature_cols),
        "categorical_cols": [CANDLE_CLASS_COL],
        "categorical_cardinality": {CANDLE_CLASS_COL: CANDLE_CLASS_CARDINALITY},
        "candle_class_names": {str(k): v for k, v in CANDLE_CLASS_NAMES.items()},
        "window_len": int(window_len),
        "continuous_label_cols": label_cols,
        "label_mean": [float(x) for x in label_mean],
        "label_std": [float(x) for x in label_std],
        "regime_names": {"0": "LOW", "1": "NORMAL", "2": "HIGH", "3": "EXTREME"},
        "structure_outcome_names": {
            str(k): v for k, v in STRUCTURE_OUTCOME_NAMES.items()
        },
        "onnx_output_names": names,
        "vol_regime_quantile_edges": [float(x) for x in q_edges],
        "config": dict(config),
    }

def metadata_from_training_export(
    *,
    resolution: str,
    label_mean: Sequence[float],
    label_std: Sequence[float],
    config: Mapping[str, Any],
    test_metrics: Mapping[str, Any] | None = None,
    export_quality: Mapping[str, Any] | None = None,
    onnx_output_names: Sequence[str] | None = None,
) -> Dict[str, Any]:
    """Build metadata_transformer.json for a per-TF bundle."""
    res = resolution.strip().lower()
    cfg = dict(config)
    names = list(onnx_output_names or ONNX_OUTPUT_NAMES)
    meta: Dict[str, Any] = {
        "version": "transformer_per_tf_v1",
        "model_name": f"jacksparrow_transformer_BTCUSD_{res}",
        "model_family": model_family_for_resolution(res),
        "symbol": str(cfg.get("symbol") or "BTCUSD"),
        "resolution": res,
        "resolution_minutes": int(cfg.get("resolution_minutes") or 15),
        "onnx_filename": onnx_filename_for_resolution(res),
        "feature_config_filename": TRANSFORMER_FEATURE_CONFIG_FILENAME,
        "path_label_horizon_bars": int(cfg.get("path_label_horizon_bars") or 8),
        "atr_period": int(cfg.get("atr_period") or 14),
        "default_threshold": float(cfg.get("default_threshold") or 0.005),
        "primary_signal_mode": "path_edge",
        "onnx_output_names": names,
        "label_mean": [float(x) for x in label_mean],
        "label_std": [float(x) for x in label_std],
        "test_metrics": dict(test_metrics or {}),
        "training_config": cfg,
    }
    if export_quality:
        meta["export_quality"] = dict(export_quality)
    return meta

def training_config_for_resolution(resolution: str) -> Dict[str, Any]:
    return default_training_config(resolution)


## Delta Exchange India data

In [ ]:
"""Public Delta India history fetch for Colab transformer training."""

from __future__ import annotations

import time
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional

import pandas as pd
import requests

_RESOLUTION_MINUTES = {
    "1m": 1,
    "3m": 3,
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "1d": 1440,
}

# Documented candles cap is 2000; live India API has returned 4000 (newest-in-range
# when the window is larger). Request windows stay at the documented 2000. The loop
# walks backward from the oldest timestamp actually returned so a lower cap cannot
# open gaps. Production loaders use 500 for the same reason.
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95

def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(
            f"Unsupported resolution {resolution!r}; "
            f"use one of {sorted(_RESOLUTION_MINUTES)}"
        )
    return int(_RESOLUTION_MINUTES[resolution]) * 60

def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    """Normalize Delta history/candles payloads to a list of candle dicts."""
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    rows: List[Dict[str, Any]] = []
    for row in result:
        if isinstance(row, dict) and "time" in row:
            rows.append(row)
    return rows

def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None

def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []

def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df

def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    """Pull OHLC-style candles with response-driven backward pagination.

    Each request covers at most ``page_bars`` of wall-clock. The next ``end`` is the
    oldest timestamp actually returned minus one second, so a server cap below the
    requested window cannot skip bars.

    Args:
        symbol: Delta symbol (e.g. ``BTCUSD``, ``FUNDING:BTCUSD``).
        resolution: Candle resolution key in ``_RESOLUTION_MINUTES``.
        start_ts: Inclusive Unix seconds.
        end_ts: Exclusive-ish Unix seconds (API treats the range as a window).
        base_url: Exchange origin, without a trailing path.
        page_bars: Max bars per request window (default documented 2000).

    Returns:
        Deduplicated OHLCV frame sorted by ``time`` ascending.

    Raises:
        ValueError: Unknown resolution.
        RuntimeError: No candles in the requested range.
    """
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")

    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)

    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)

    if not all_rows:
        raise RuntimeError(
            f"No candle data returned for symbol={symbol}; "
            "check symbol/resolution/date range."
        )
    return _rows_to_frame(all_rows)

def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    """Measure gap rate on a native-TF OHLCV frame.

    Completeness is ``1 - gaps / (n - 1)`` where a gap is a bar-to-bar delta greater
    than ``1.5 * bar_seconds``. This catches sawtooth holes from truncated pages.

    Args:
        df: Frame with a ``time`` column (datetime or Unix seconds).
        resolution: Candle resolution key.
        min_completeness: Raise if the gap-free fraction is below this.
        symbol: Optional label for the error message.

    Returns:
        Report with ``rows``, ``gaps``, ``completeness``, and ``span_bars``.

    Raises:
        ValueError: Empty frame or completeness below ``min_completeness``.
    """
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    if pd.api.types.is_datetime64_any_dtype(time_col):
        epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    else:
        epochs = pd.to_numeric(time_col, errors="coerce").dropna().astype(int).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")

    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    span_bars = int((epochs[-1] - epochs[0]) // bar_seconds) + 1
    report: Dict[str, Any] = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
        "span_bars": span_bars,
        "expected_bars": span_bars,
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'} ({gaps} gaps, {len(epochs)} rows, "
            f"span {span_bars} bars). Refetch with response-driven pagination."
        )
    return report

def validate_derivatives_coverage(
    df: pd.DataFrame,
    *,
    min_coverage: float = 0.5,
    warn_coverage: float = 0.9,
) -> Dict[str, Any]:
    """Check funding/OI non-null coverage over the OHLCV timeline."""
    n = len(df)
    if n == 0:
        raise ValueError("Cannot validate derivatives coverage on empty frame")

    report: Dict[str, Any] = {"rows": n, "columns": {}}
    for col in ("funding_rate", "open_interest"):
        if col not in df.columns:
            report["columns"][col] = {"coverage": 0.0, "status": "missing"}
            continue
        coverage = float(df[col].notna().mean())
        status = "ok"
        if coverage < min_coverage:
            status = "error"
        elif coverage < warn_coverage:
            status = "warn"
        report["columns"][col] = {
            "coverage": round(coverage, 4),
            "status": status,
        }

    worst = min(v["coverage"] for v in report["columns"].values())
    report["worst_coverage"] = worst
    if worst < min_coverage:
        raise ValueError(
            "Derivatives coverage below minimum "
            f"({worst:.1%} < {min_coverage:.0%}): {report['columns']}"
        )
    return report

def fetch_history_bundle(
    *,
    symbol: str,
    resolution: str,
    history_days: int,
    base_url: str,
    min_derivatives_coverage: float = 0.5,
    derivatives_coverage_warn: float = 0.9,
    min_ohlcv_completeness: float = MIN_OHLCV_COMPLETENESS,
) -> pd.DataFrame:
    """Fetch OHLCV + funding + OI merged frame (notebook-compatible)."""
    end_dt = datetime.now(timezone.utc)
    start_dt = end_dt - timedelta(days=history_days)
    start_ts, end_ts = int(start_dt.timestamp()), int(end_dt.timestamp())

    raw_df = fetch_candles(symbol, resolution, start_ts, end_ts, base_url)
    ohlcv_report = validate_ohlcv_completeness(
        raw_df,
        resolution,
        min_completeness=min_ohlcv_completeness,
        symbol=symbol,
    )
    print(
        f"OHLCV {symbol} {resolution}: {ohlcv_report['rows']} bars, "
        f"completeness {ohlcv_report['completeness']:.1%}, "
        f"gaps={ohlcv_report['gaps']}"
    )

    try:
        funding_df = fetch_candles(
            f"FUNDING:{symbol}", resolution, start_ts, end_ts, base_url
        )
        funding_df = funding_df[["time", "close"]].rename(columns={"close": "funding_rate"})
    except Exception:
        funding_df = pd.DataFrame({"time": [], "funding_rate": []})

    try:
        oi_df = fetch_candles(f"OI:{symbol}", resolution, start_ts, end_ts, base_url)
        oi_df = oi_df[["time", "close"]].rename(columns={"close": "open_interest"})
    except Exception:
        oi_df = pd.DataFrame({"time": [], "open_interest": []})

    raw_df = assemble_raw_frame(raw_df, funding_df=funding_df, oi_df=oi_df)

    report = validate_derivatives_coverage(
        raw_df,
        min_coverage=min_derivatives_coverage,
        warn_coverage=derivatives_coverage_warn,
    )
    for col, info in report["columns"].items():
        if info["status"] == "warn":
            print(
                f"WARNING: {col} coverage {info['coverage']:.1%} "
                f"below recommended {derivatives_coverage_warn:.0%}"
            )
        elif info["status"] == "ok":
            print(f"{col} coverage: {info['coverage']:.1%}")

    return raw_df


## Pattern geometry utils

In [ ]:
"""Shared utilities for pattern feature computation."""

from dataclasses import dataclass

import pandas as pd

@dataclass
class CandleGeometry:
    """Pre-computed geometry for a single candle — avoids redundant calculations."""

    body: float
    upper_wick: float
    lower_wick: float
    total_range: float
    body_ratio: float
    upper_ratio: float
    lower_ratio: float
    is_bullish: bool

    @classmethod
    def from_row(cls, row: pd.Series) -> "CandleGeometry":
        body = abs(row["close"] - row["open"])
        upper_wick = row["high"] - max(row["open"], row["close"])
        lower_wick = min(row["open"], row["close"]) - row["low"]
        total_range = row["high"] - row["low"]
        safe_range = total_range if total_range > 1e-10 else 1e-10
        return cls(
            body=body,
            upper_wick=upper_wick,
            lower_wick=lower_wick,
            total_range=total_range,
            body_ratio=body / safe_range,
            upper_ratio=upper_wick / safe_range,
            lower_ratio=lower_wick / safe_range,
            is_bullish=row["close"] >= row["open"],
        )

def compute_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:
    """Compute Average True Range."""
    high_low = df["high"] - df["low"]
    high_close = (df["high"] - df["close"].shift()).abs()
    low_close = (df["low"] - df["close"].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()


## Native candlestick encodings

In [ ]:
"""
Candlestick pattern detection for ML features.

Detects single-candle, two-candle, and three-candle patterns.
Produces ML-ready feature columns aligned with input DataFrame index.
"""

import pandas as pd

# Feature names for registration
CANDLESTICK_FEATURES = [
    "cdl_doji",
    "cdl_long_legged_doji",
    "cdl_dragonfly_doji",
    "cdl_gravestone_doji",
    "cdl_hammer",
    "cdl_inv_hammer",
    "cdl_hanging_man",
    "cdl_shooting_star",
    "cdl_bull_marubozu",
    "cdl_bear_marubozu",
    "cdl_spinning_top",
    "cdl_bull_engulfing",
    "cdl_bear_engulfing",
    "cdl_bull_harami",
    "cdl_bear_harami",
    "cdl_piercing",
    "cdl_dark_cloud",
    "cdl_tweezer_top",
    "cdl_tweezer_bottom",
    "cdl_bull_kicker",
    "cdl_bear_kicker",
    "cdl_morning_star",
    "cdl_evening_star",
    "cdl_three_white_soldiers",
    "cdl_three_black_crows",
    "cdl_three_inside_up",
    "cdl_three_inside_down",
    "cdl_abandoned_baby_bull",
    "cdl_abandoned_baby_bear",
    "cdl_bull_score",
    "cdl_bear_score",
    "cdl_net_score",
    "cdl_reversal_signal",
    "cdl_indecision_score",
    "cdl_body_ratio",
    "cdl_upper_wick_ratio",
    "cdl_lower_wick_ratio",
    "cdl_consecutive_bull",
    "cdl_consecutive_bear",
]

class CandlestickPatternEngine:
    """
    Detects candlestick patterns and produces ML-ready feature columns.
    All methods operate on a full OHLCV DataFrame and return a feature DataFrame
    aligned by index — safe for both training (batch) and live (last-row) use.

    Hammer vs hanging man (and inverted hammer vs shooting star) share geometry;
    they are disambiguated only by prior trend context, never by candle color alone.
    """

    # Bars of prior closes (ending at t-1) used for trend context.
    TREND_LOOKBACK = 5

    BULL_PATTERN_WEIGHTS = {
        "cdl_hammer": 0.8,
        "cdl_dragonfly_doji": 0.6,
        "cdl_bull_engulfing": 0.95,
        "cdl_bull_harami": 0.6,
        "cdl_piercing": 0.75,
        "cdl_morning_star": 0.9,
        "cdl_three_white_soldiers": 0.85,
        "cdl_bull_marubozu": 0.7,
        "cdl_tweezer_bottom": 0.65,
        "cdl_three_inside_up": 0.7,
        "cdl_abandoned_baby_bull": 0.95,
        "cdl_bull_kicker": 0.9,
        "cdl_inv_hammer": 0.5,
    }

    BEAR_PATTERN_WEIGHTS = {
        "cdl_shooting_star": 0.8,
        "cdl_gravestone_doji": 0.6,
        "cdl_bear_engulfing": 0.95,
        "cdl_bear_harami": 0.6,
        "cdl_dark_cloud": 0.75,
        "cdl_evening_star": 0.9,
        "cdl_three_black_crows": 0.85,
        "cdl_bear_marubozu": 0.7,
        "cdl_tweezer_top": 0.65,
        "cdl_three_inside_down": 0.7,
        "cdl_abandoned_baby_bear": 0.95,
        "cdl_bear_kicker": 0.9,
        "cdl_hanging_man": 0.7,
    }

    def compute_all(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Main entry point. Returns DataFrame of all candlestick pattern features.
        Aligned with input df index.
        """
        required = ["open", "high", "low", "close"]
        for col in required:
            if col not in df.columns:
                raise ValueError(f"Missing required column: {col}")

        geo = list(df.apply(CandleGeometry.from_row, axis=1))
        out = pd.DataFrame(index=df.index)

        # Single-candle patterns
        out["cdl_doji"] = self._doji(df, geo)
        out["cdl_long_legged_doji"] = self._long_legged_doji(df, geo)
        out["cdl_dragonfly_doji"] = self._dragonfly_doji(df, geo)
        out["cdl_gravestone_doji"] = self._gravestone_doji(df, geo)
        out["cdl_hammer"] = self._hammer(df, geo)
        out["cdl_inv_hammer"] = self._inverted_hammer(df, geo)
        out["cdl_hanging_man"] = self._hanging_man(df, geo)
        out["cdl_shooting_star"] = self._shooting_star(df, geo)
        out["cdl_bull_marubozu"] = self._bull_marubozu(df, geo)
        out["cdl_bear_marubozu"] = self._bear_marubozu(df, geo)
        out["cdl_spinning_top"] = self._spinning_top(df, geo)

        # Two-candle patterns
        out["cdl_bull_engulfing"] = self._bull_engulfing(df, geo)
        out["cdl_bear_engulfing"] = self._bear_engulfing(df, geo)
        out["cdl_bull_harami"] = self._bull_harami(df, geo)
        out["cdl_bear_harami"] = self._bear_harami(df, geo)
        out["cdl_piercing"] = self._piercing(df, geo)
        out["cdl_dark_cloud"] = self._dark_cloud(df, geo)
        out["cdl_tweezer_top"] = self._tweezer_top(df, geo)
        out["cdl_tweezer_bottom"] = self._tweezer_bottom(df, geo)
        out["cdl_bull_kicker"] = self._bull_kicker(df, geo)
        out["cdl_bear_kicker"] = self._bear_kicker(df, geo)

        # Three-candle patterns
        out["cdl_morning_star"] = self._morning_star(df, geo)
        out["cdl_evening_star"] = self._evening_star(df, geo)
        out["cdl_three_white_soldiers"] = self._three_white_soldiers(df, geo)
        out["cdl_three_black_crows"] = self._three_black_crows(df, geo)
        out["cdl_three_inside_up"] = self._three_inside_up(df, geo)
        out["cdl_three_inside_down"] = self._three_inside_down(df, geo)
        out["cdl_abandoned_baby_bull"] = self._abandoned_baby(df, geo, direction="bull")
        out["cdl_abandoned_baby_bear"] = self._abandoned_baby(df, geo, direction="bear")

        # Composite scores
        out["cdl_bull_score"] = self._bull_composite(out)
        out["cdl_bear_score"] = self._bear_composite(out)
        out["cdl_net_score"] = out["cdl_bull_score"] - out["cdl_bear_score"]
        out["cdl_reversal_signal"] = out["cdl_net_score"].clip(-1, 1)
        out["cdl_indecision_score"] = (
            out["cdl_doji"] * 0.6
            + out["cdl_long_legged_doji"] * 0.8
            + out["cdl_spinning_top"] * 0.4
        ).clip(0, 1)

        # Raw geometry features
        out["cdl_body_ratio"] = [g.body_ratio for g in geo]
        out["cdl_upper_wick_ratio"] = [g.upper_ratio for g in geo]
        out["cdl_lower_wick_ratio"] = [g.lower_ratio for g in geo]
        out["cdl_consecutive_bull"] = self._consecutive_direction(df, direction="bull")
        out["cdl_consecutive_bear"] = self._consecutive_direction(df, direction="bear")

        return out.fillna(0)

    def _doji(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [1 if g.body_ratio < 0.10 and g.total_range > 0 else 0 for g in geo],
            index=df.index,
        )

    def _long_legged_doji(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [
                1
                if g.body_ratio < 0.10 and g.upper_ratio > 0.30 and g.lower_ratio > 0.30
                else 0
                for g in geo
            ],
            index=df.index,
        )

    def _dragonfly_doji(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [
                1
                if g.body_ratio < 0.10 and g.upper_ratio < 0.05 and g.lower_ratio > 0.60
                else 0
                for g in geo
            ],
            index=df.index,
        )

    def _gravestone_doji(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [
                1
                if g.body_ratio < 0.10 and g.lower_ratio < 0.05 and g.upper_ratio > 0.60
                else 0
                for g in geo
            ],
            index=df.index,
        )

    def _prior_trend(self, df: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
        """Causal prior up/down trend from closes ending at the previous bar.

        Uses prior close vs its trailing SMA over ``TREND_LOOKBACK`` bars.
        Flat or insufficient history yields neither up nor down.
        """
        prior_close = df["close"].astype(float).shift(1)
        sma = prior_close.rolling(
            self.TREND_LOOKBACK, min_periods=self.TREND_LOOKBACK
        ).mean()
        prior_up = (prior_close > sma).fillna(False)
        prior_down = (prior_close < sma).fillna(False)
        return prior_up, prior_down

    @staticmethod
    def _atr_proxy(df: pd.DataFrame) -> pd.Series:
        atr = df["close"].diff().abs().rolling(14).mean()
        fallback = df["close"].astype(float) * 0.01
        return atr.where(atr.notna() & (atr > 0), fallback)

    @staticmethod
    def _is_hammer_shape(g: CandleGeometry, atr_val: float) -> bool:
        """Long lower wick, short upper wick, meaningful body and range."""
        return (
            g.lower_wick >= 2.0 * g.body
            and g.upper_ratio < 0.20
            and g.body_ratio > 0.05
            and g.total_range > float(atr_val) * 0.5
        )

    @staticmethod
    def _is_inverted_hammer_shape(g: CandleGeometry) -> bool:
        """Long upper wick, short lower wick, meaningful body."""
        return (
            g.upper_wick >= 2.0 * g.body
            and g.lower_ratio < 0.20
            and g.body_ratio > 0.05
        )

    def _hammer(self, df: pd.DataFrame, geo: list) -> pd.Series:
        """Bullish hammer: hammer shape after a prior downtrend."""
        atr = self._atr_proxy(df)
        _, prior_down = self._prior_trend(df)
        result = [
            1
            if self._is_hammer_shape(g, float(atr.iloc[i])) and bool(prior_down.iloc[i])
            else 0
            for i, g in enumerate(geo)
        ]
        return pd.Series(result, index=df.index)

    def _hanging_man(self, df: pd.DataFrame, geo: list) -> pd.Series:
        """Bearish hanging man: same shape as hammer after a prior uptrend."""
        atr = self._atr_proxy(df)
        prior_up, _ = self._prior_trend(df)
        result = [
            1
            if self._is_hammer_shape(g, float(atr.iloc[i])) and bool(prior_up.iloc[i])
            else 0
            for i, g in enumerate(geo)
        ]
        return pd.Series(result, index=df.index)

    def _inverted_hammer(self, df: pd.DataFrame, geo: list) -> pd.Series:
        """Bullish inverted hammer: inverted shape after a prior downtrend."""
        _, prior_down = self._prior_trend(df)
        result = [
            1
            if self._is_inverted_hammer_shape(g) and bool(prior_down.iloc[i])
            else 0
            for i, g in enumerate(geo)
        ]
        return pd.Series(result, index=df.index)

    def _shooting_star(self, df: pd.DataFrame, geo: list) -> pd.Series:
        """Bearish shooting star: same shape as inverted hammer after uptrend."""
        prior_up, _ = self._prior_trend(df)
        result = [
            1
            if self._is_inverted_hammer_shape(g) and bool(prior_up.iloc[i])
            else 0
            for i, g in enumerate(geo)
        ]
        return pd.Series(result, index=df.index)

    def _bull_marubozu(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [1 if g.is_bullish and g.body_ratio > 0.95 else 0 for g in geo],
            index=df.index,
        )

    def _bear_marubozu(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [1 if not g.is_bullish and g.body_ratio > 0.95 else 0 for g in geo],
            index=df.index,
        )

    def _spinning_top(self, df: pd.DataFrame, geo: list) -> pd.Series:
        return pd.Series(
            [
                1
                if 0.10 < g.body_ratio < 0.40
                and g.upper_ratio > 0.20
                and g.lower_ratio > 0.20
                else 0
                for g in geo
            ],
            index=df.index,
        )

    def _bull_engulfing(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            if (
                not prev.is_bullish
                and curr.is_bullish
                and df["open"].iloc[i] <= df["close"].iloc[i - 1]
                and df["close"].iloc[i] >= df["open"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _bear_engulfing(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            if (
                prev.is_bullish
                and not curr.is_bullish
                and df["open"].iloc[i] >= df["close"].iloc[i - 1]
                and df["close"].iloc[i] <= df["open"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _bull_harami(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            if (
                not prev.is_bullish
                and curr.is_bullish
                and curr.body < prev.body * 0.5
                and df["open"].iloc[i] > df["close"].iloc[i - 1]
                and df["close"].iloc[i] < df["open"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _bear_harami(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            if (
                prev.is_bullish
                and not curr.is_bullish
                and curr.body < prev.body * 0.5
                and df["open"].iloc[i] < df["close"].iloc[i - 1]
                and df["close"].iloc[i] > df["open"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _piercing(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            midpoint = (df["open"].iloc[i - 1] + df["close"].iloc[i - 1]) / 2
            if (
                not prev.is_bullish
                and curr.is_bullish
                and df["open"].iloc[i] < df["close"].iloc[i - 1]
                and df["close"].iloc[i] > midpoint
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _dark_cloud(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            prev, curr = geo[i - 1], geo[i]
            midpoint = (df["open"].iloc[i - 1] + df["close"].iloc[i - 1]) / 2
            if (
                prev.is_bullish
                and not curr.is_bullish
                and df["open"].iloc[i] > df["close"].iloc[i - 1]
                and df["close"].iloc[i] < midpoint
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _tweezer_top(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            high_diff = abs(df["high"].iloc[i] - df["high"].iloc[i - 1])
            atr = df["close"].iloc[max(0, i - 14) : i].diff().abs().mean()
            atr_val = atr if not pd.isna(atr) and atr > 0 else df["close"].iloc[i] * 0.01
            if not geo[i].is_bullish and high_diff < atr_val * 0.1:
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _tweezer_bottom(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            low_diff = abs(df["low"].iloc[i] - df["low"].iloc[i - 1])
            atr = df["close"].iloc[max(0, i - 14) : i].diff().abs().mean()
            atr_val = atr if not pd.isna(atr) and atr > 0 else df["close"].iloc[i] * 0.01
            if geo[i].is_bullish and low_diff < atr_val * 0.1:
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _bull_kicker(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            if (
                not geo[i - 1].is_bullish
                and geo[i].is_bullish
                and df["open"].iloc[i] > df["open"].iloc[i - 1]
                and geo[i].body_ratio > 0.5
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _bear_kicker(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(1, len(df)):
            if (
                geo[i - 1].is_bullish
                and not geo[i].is_bullish
                and df["open"].iloc[i] < df["open"].iloc[i - 1]
                and geo[i].body_ratio > 0.5
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _morning_star(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            mid_a = (df["open"].iloc[i - 2] + df["close"].iloc[i - 2]) / 2
            if (
                not a.is_bullish
                and a.body_ratio > 0.4
                and b.body_ratio < 0.3
                and c.is_bullish
                and df["close"].iloc[i] > mid_a
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _evening_star(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            mid_a = (df["open"].iloc[i - 2] + df["close"].iloc[i - 2]) / 2
            if (
                a.is_bullish
                and a.body_ratio > 0.4
                and b.body_ratio < 0.3
                and not c.is_bullish
                and df["close"].iloc[i] < mid_a
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _three_white_soldiers(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            if (
                a.is_bullish
                and b.is_bullish
                and c.is_bullish
                and df["close"].iloc[i] > df["close"].iloc[i - 1] > df["close"].iloc[i - 2]
                and a.body_ratio > 0.5
                and b.body_ratio > 0.5
                and c.body_ratio > 0.5
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _three_black_crows(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            if (
                not a.is_bullish
                and not b.is_bullish
                and not c.is_bullish
                and df["close"].iloc[i] < df["close"].iloc[i - 1] < df["close"].iloc[i - 2]
                and a.body_ratio > 0.5
                and b.body_ratio > 0.5
                and c.body_ratio > 0.5
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _three_inside_up(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            if (
                not a.is_bullish
                and b.is_bullish
                and c.is_bullish
                and b.body < a.body
                and df["close"].iloc[i] > df["close"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _three_inside_down(self, df: pd.DataFrame, geo: list) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            if (
                a.is_bullish
                and not b.is_bullish
                and not c.is_bullish
                and b.body < a.body
                and df["close"].iloc[i] < df["close"].iloc[i - 1]
            ):
                result[i] = 1
        return pd.Series(result, index=df.index)

    def _abandoned_baby(
        self, df: pd.DataFrame, geo: list, direction: str
    ) -> pd.Series:
        result = [0] * len(df)
        for i in range(2, len(df)):
            a, b, c = geo[i - 2], geo[i - 1], geo[i]
            is_doji_b = b.body_ratio < 0.10
            if direction == "bull":
                gap1 = df["low"].iloc[i - 1] < df["low"].iloc[i - 2]
                gap2 = df["low"].iloc[i] > df["high"].iloc[i - 1]
                if not a.is_bullish and is_doji_b and c.is_bullish and gap1 and gap2:
                    result[i] = 1
            else:
                gap1 = df["high"].iloc[i - 1] > df["high"].iloc[i - 2]
                gap2 = df["high"].iloc[i] < df["low"].iloc[i - 1]
                if a.is_bullish and is_doji_b and not c.is_bullish and gap1 and gap2:
                    result[i] = 1
        return pd.Series(result, index=df.index)

    def _bull_composite(self, out: pd.DataFrame) -> pd.Series:
        score = pd.Series(0.0, index=out.index)
        for col, weight in self.BULL_PATTERN_WEIGHTS.items():
            if col in out.columns:
                score += out[col] * weight
        total_weight = sum(self.BULL_PATTERN_WEIGHTS.values())
        return (score / total_weight).clip(0, 1)

    def _bear_composite(self, out: pd.DataFrame) -> pd.Series:
        score = pd.Series(0.0, index=out.index)
        for col, weight in self.BEAR_PATTERN_WEIGHTS.items():
            if col in out.columns:
                score += out[col] * weight
        total_weight = sum(self.BEAR_PATTERN_WEIGHTS.values())
        return (score / total_weight).clip(0, 1)

    def _consecutive_direction(
        self, df: pd.DataFrame, direction: str
    ) -> pd.Series:
        is_bull = (df["close"] >= df["open"]).astype(int)
        values = is_bull if direction == "bull" else (1 - is_bull)
        result = []
        count = 0
        for v in values:
            count = count + 1 if v == 1 else 0
            result.append(count)
        return pd.Series(result, index=df.index)


## Native chart encodings

In [ ]:
"""
Chart pattern detection for ML features.

Detects support/resistance, trendlines, flags, triangles, reversals, and breakouts.
Produces ML-ready feature columns aligned with input DataFrame index.
"""

import numpy as np
import pandas as pd
from scipy.signal import argrelextrema
from scipy.stats import linregress

# Feature names for registration
CHART_PATTERN_FEATURES = [
    "sr_support_dist_pct",
    "sr_resistance_dist_pct",
    "sr_at_support",
    "sr_at_resistance",
    "sr_support_strength",
    "sr_resistance_strength",
    "sr_range_position",
    "tl_uptrend_detected",
    "tl_downtrend_detected",
    "tl_trend_slope",
    "tl_dist_to_trendline",
    "tl_near_trendline",
    "tl_breakout_up",
    "tl_breakout_down",
    "chp_bull_flag",
    "chp_bear_flag",
    "chp_bull_flag_strength",
    "chp_bear_flag_strength",
    "chp_asc_triangle",
    "chp_desc_triangle",
    "chp_sym_triangle",
    "chp_triangle_apex_dist",
    "chp_double_top",
    "chp_double_bottom",
    "chp_double_top_dist",
    "chp_double_bottom_dist",
    "chp_hs_detected",
    "chp_ihs_detected",
    "bo_at_high",
    "bo_at_low",
    "bo_volume_confirmation",
    "bo_breakout_score",
]

def _chart_safe_atr(atr_value: object, price: float) -> float:
    """Positive ATR for ratios. Zero ATR appears on flat / warmup bars."""
    try:
        a = float(atr_value)
    except (TypeError, ValueError):
        a = float("nan")
    if not np.isfinite(a) or a <= 0.0:
        return max(abs(float(price)) * 0.01, 1e-9)
    return a

class ChartPatternEngine:
    """
    Detects chart patterns and computes structural market features.
    All methods return full-length DataFrames aligned with input index.
    """

    def compute_all(self, df: pd.DataFrame, atr_period: int = 14) -> pd.DataFrame:
        """Main entry point — computes all chart pattern features."""
        atr = self._compute_atr(df, atr_period)
        out = pd.DataFrame(index=df.index)

        sr = self._compute_support_resistance(df, atr)
        out = pd.concat([out, sr], axis=1)

        tl = self._compute_trendlines(df, atr)
        out = pd.concat([out, tl], axis=1)

        flags = self._compute_flags(df, atr)
        out = pd.concat([out, flags], axis=1)

        tri = self._compute_triangles(df, atr)
        out = pd.concat([out, tri], axis=1)

        rev = self._compute_reversal_patterns(df, atr)
        out = pd.concat([out, rev], axis=1)

        bo = self._compute_breakouts(df, atr)
        out = pd.concat([out, bo], axis=1)

        return out.replace([np.inf, -np.inf], 0).fillna(0)

    def _compute_atr(self, df: pd.DataFrame, period: int = 14) -> pd.Series:
        high_low = df["high"] - df["low"]
        high_close = (df["high"] - df["close"].shift()).abs()
        low_close = (df["low"] - df["close"].shift()).abs()
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        return tr.rolling(period).mean()

    def _compute_support_resistance(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        window: int = 5,
        lookback: int = 60,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        highs = df["high"].values
        lows = df["low"].values
        closes = df["close"].values

        high_idx = argrelextrema(highs, np.greater, order=window)[0]
        low_idx = argrelextrema(lows, np.less, order=window)[0]

        sr_support_dist = np.full(len(df), np.nan)
        sr_resistance_dist = np.full(len(df), np.nan)
        sr_at_support = np.zeros(len(df))
        sr_at_resistance = np.zeros(len(df))
        sr_support_strength = np.zeros(len(df))
        sr_resistance_strength = np.zeros(len(df))
        sr_range_position = np.full(len(df), 0.5)

        for i in range(lookback, len(df)):
            current_price = closes[i]
            current_atr = _chart_safe_atr(atr.iloc[i], float(current_price))

            recent_highs_idx = high_idx[(high_idx >= i - lookback) & (high_idx < i)]
            recent_lows_idx = low_idx[(low_idx >= i - lookback) & (low_idx < i)]

            if len(recent_highs_idx) > 0:
                resistance_levels = highs[recent_highs_idx]
                above = resistance_levels[resistance_levels > current_price]
                if len(above) > 0:
                    nearest_res = above.min()
                    sr_resistance_dist[i] = (
                        (nearest_res - current_price) / current_price * 100
                    )
                    sr_at_resistance[i] = (
                        1 if abs(nearest_res - current_price) < current_atr else 0
                    )
                    sr_resistance_strength[i] = np.sum(
                        np.abs(resistance_levels - nearest_res) < current_atr * 0.5
                    )

            if len(recent_lows_idx) > 0:
                support_levels = lows[recent_lows_idx]
                below = support_levels[support_levels < current_price]
                if len(below) > 0:
                    nearest_sup = below.max()
                    sr_support_dist[i] = (
                        (current_price - nearest_sup) / current_price * 100
                    )
                    sr_at_support[i] = (
                        1 if abs(current_price - nearest_sup) < current_atr else 0
                    )
                    sr_support_strength[i] = np.sum(
                        np.abs(support_levels - nearest_sup) < current_atr * 0.5
                    )

            sup_d = sr_support_dist[i] if not np.isnan(sr_support_dist[i]) else 0
            res_d = sr_resistance_dist[i] if not np.isnan(sr_resistance_dist[i]) else 0
            total = sup_d + res_d
            if total > 0:
                sr_range_position[i] = res_d / total

        out["sr_support_dist_pct"] = sr_support_dist
        out["sr_resistance_dist_pct"] = sr_resistance_dist
        out["sr_at_support"] = sr_at_support
        out["sr_at_resistance"] = sr_at_resistance
        out["sr_support_strength"] = sr_support_strength
        out["sr_resistance_strength"] = sr_resistance_strength
        out["sr_range_position"] = sr_range_position

        return out

    def _compute_trendlines(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        min_touches: int = 3,
        lookback: int = 50,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        highs = df["high"].values
        lows = df["low"].values
        closes = df["close"].values

        high_idx = argrelextrema(highs, np.greater, order=5)[0]
        low_idx = argrelextrema(lows, np.less, order=5)[0]

        tl_uptrend = np.zeros(len(df))
        tl_downtrend = np.zeros(len(df))
        tl_slope = np.zeros(len(df))
        tl_dist = np.zeros(len(df))
        tl_near = np.zeros(len(df))
        tl_break_up = np.zeros(len(df))
        tl_break_down = np.zeros(len(df))

        for i in range(lookback, len(df)):
            current_atr = _chart_safe_atr(atr.iloc[i], float(closes[i]))

            ul_idx = low_idx[(low_idx >= i - lookback) & (low_idx < i)]
            if len(ul_idx) >= min_touches:
                x = ul_idx.astype(float)
                y = lows[ul_idx]
                slope, intercept, r, _, _ = linregress(x, y)
                if slope > 0 and r > 0.7:
                    tl_uptrend[i] = 1
                    trendline_val = slope * i + intercept
                    dist = (closes[i] - trendline_val) / current_atr
                    tl_slope[i] = slope / (closes[i] * 0.0001)
                    tl_dist[i] = max(0, dist)
                    tl_near[i] = 1 if abs(dist) < 1.0 else 0
                    if closes[i] < trendline_val and closes[i - 1] >= (
                        slope * (i - 1) + intercept
                    ):
                        tl_break_down[i] = 1

            dl_idx = high_idx[(high_idx >= i - lookback) & (high_idx < i)]
            if len(dl_idx) >= min_touches:
                x = dl_idx.astype(float)
                y = highs[dl_idx]
                slope, intercept, r, _, _ = linregress(x, y)
                if slope < 0 and r > 0.7:
                    tl_downtrend[i] = 1
                    trendline_val = slope * i + intercept
                    if closes[i] > trendline_val and closes[i - 1] <= (
                        slope * (i - 1) + intercept
                    ):
                        tl_break_up[i] = 1

        out["tl_uptrend_detected"] = tl_uptrend
        out["tl_downtrend_detected"] = tl_downtrend
        out["tl_trend_slope"] = tl_slope
        out["tl_dist_to_trendline"] = tl_dist
        out["tl_near_trendline"] = tl_near
        out["tl_breakout_up"] = tl_break_up
        out["tl_breakout_down"] = tl_break_down

        return out

    def _compute_flags(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        pole_bars: int = 10,
        flag_bars: int = 20,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        closes = df["close"].values
        n = len(df)

        bull_flag = np.zeros(n)
        bear_flag = np.zeros(n)
        bull_flag_strength = np.zeros(n)
        bear_flag_strength = np.zeros(n)

        for i in range(pole_bars + flag_bars, n):
            current_atr = _chart_safe_atr(atr.iloc[i], float(closes[i]))

            pole_start = i - pole_bars - flag_bars
            pole_end = i - flag_bars
            pole_move = closes[pole_end] - closes[pole_start]
            pole_magnitude = abs(pole_move) / (current_atr * pole_bars)

            if pole_magnitude > 0.5:
                flag_closes = closes[pole_end:i]
                flag_range = np.max(flag_closes) - np.min(flag_closes)
                flag_drift = closes[i - 1] - closes[pole_end]

                if pole_move > 0:
                    if flag_range < abs(pole_move) * 0.5 and flag_drift < 0:
                        bull_flag[i] = 1
                        bull_flag_strength[i] = pole_magnitude
                else:
                    if flag_range < abs(pole_move) * 0.5 and flag_drift > 0:
                        bear_flag[i] = 1
                        bear_flag_strength[i] = pole_magnitude

        out["chp_bull_flag"] = bull_flag
        out["chp_bear_flag"] = bear_flag
        out["chp_bull_flag_strength"] = bull_flag_strength
        out["chp_bear_flag_strength"] = bear_flag_strength

        return out

    def _compute_triangles(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        lookback: int = 40,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        highs = df["high"].values
        lows = df["low"].values
        n = len(df)

        asc_tri = np.zeros(n)
        desc_tri = np.zeros(n)
        sym_tri = np.zeros(n)
        apex_dist = np.zeros(n)

        for i in range(lookback, n):
            window_highs = highs[i - lookback : i]
            window_lows = lows[i - lookback : i]
            x = np.arange(lookback, dtype=float)

            slope_h, _, _, _, _ = linregress(x, window_highs)
            slope_l, _, _, _, _ = linregress(x, window_lows)

            atr_val = _chart_safe_atr(atr.iloc[i], float(highs[i]))

            if abs(slope_h) < 0.1 * atr_val and slope_l > 0:
                asc_tri[i] = 1
            elif slope_h < 0 and abs(slope_l) < 0.1 * atr_val:
                desc_tri[i] = 1
            elif slope_h < 0 and slope_l > 0:
                sym_tri[i] = 1
                price_range = window_highs[-1] - window_lows[-1]
                if abs(slope_h - slope_l) > 1e-10:
                    bars_to_apex = price_range / abs(slope_h - slope_l)
                    apex_dist[i] = bars_to_apex / lookback

        out["chp_asc_triangle"] = asc_tri
        out["chp_desc_triangle"] = desc_tri
        out["chp_sym_triangle"] = sym_tri
        out["chp_triangle_apex_dist"] = apex_dist

        return out

    def _compute_reversal_patterns(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        lookback: int = 60,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        highs = df["high"].values
        lows = df["low"].values
        closes = df["close"].values
        n = len(df)

        double_top = np.zeros(n)
        double_bottom = np.zeros(n)
        double_top_dist = np.zeros(n)
        double_bottom_dist = np.zeros(n)
        hs_detected = np.zeros(n)
        ihs_detected = np.zeros(n)

        high_idx = argrelextrema(highs, np.greater, order=5)[0]
        low_idx = argrelextrema(lows, np.less, order=5)[0]

        for i in range(lookback, n):
            current_atr = _chart_safe_atr(atr.iloc[i], float(closes[i]))

            recent_hi = high_idx[(high_idx >= i - lookback) & (high_idx < i)]
            if len(recent_hi) >= 2:
                h1, h2 = highs[recent_hi[-2]], highs[recent_hi[-1]]
                separation = recent_hi[-1] - recent_hi[-2]
                if abs(h1 - h2) < current_atr * 0.5 and 5 <= separation <= lookback // 2:
                    double_top[i] = 1
                    double_top_dist[i] = (closes[i] - min(h1, h2)) / current_atr

            recent_lo = low_idx[(low_idx >= i - lookback) & (low_idx < i)]
            if len(recent_lo) >= 2:
                l1, l2 = lows[recent_lo[-2]], lows[recent_lo[-1]]
                separation = recent_lo[-1] - recent_lo[-2]
                if abs(l1 - l2) < current_atr * 0.5 and 5 <= separation <= lookback // 2:
                    double_bottom[i] = 1
                    double_bottom_dist[i] = (max(l1, l2) - closes[i]) / current_atr

            if len(recent_hi) >= 3:
                p1, p2, p3 = (
                    highs[recent_hi[-3]],
                    highs[recent_hi[-2]],
                    highs[recent_hi[-1]],
                )
                if p2 > p1 and p2 > p3 and abs(p1 - p3) < current_atr:
                    hs_detected[i] = 1

            if len(recent_lo) >= 3:
                t1, t2, t3 = (
                    lows[recent_lo[-3]],
                    lows[recent_lo[-2]],
                    lows[recent_lo[-1]],
                )
                if t2 < t1 and t2 < t3 and abs(t1 - t3) < current_atr:
                    ihs_detected[i] = 1

        out["chp_double_top"] = double_top
        out["chp_double_bottom"] = double_bottom
        out["chp_double_top_dist"] = double_top_dist
        out["chp_double_bottom_dist"] = double_bottom_dist
        out["chp_hs_detected"] = hs_detected
        out["chp_ihs_detected"] = ihs_detected

        return out

    def _compute_breakouts(
        self,
        df: pd.DataFrame,
        atr: pd.Series,
        lookback: int = 20,
    ) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        closes = df["close"].values
        n = len(df)

        bo_at_high = np.zeros(n)
        bo_at_low = np.zeros(n)
        bo_volume_conf = np.zeros(n)
        bo_score = np.zeros(n)

        for i in range(lookback, n):
            window_closes = closes[i - lookback : i]
            range_high = np.max(window_closes)
            range_low = np.min(window_closes)
            current = closes[i]
            current_atr = _chart_safe_atr(atr.iloc[i], float(current))

            if current >= range_high - current_atr * 0.3:
                bo_at_high[i] = 1

            if current <= range_low + current_atr * 0.3:
                bo_at_low[i] = 1

            if "volume" in df.columns:
                vol = df["volume"].values
                avg_vol = np.mean(vol[i - lookback : i])
                if vol[i] > avg_vol * 1.5:
                    bo_volume_conf[i] = 1

            bo_score[i] = float(bo_at_high[i]) * 0.5 + float(bo_volume_conf[i]) * 0.5

        out["bo_at_high"] = bo_at_high
        out["bo_at_low"] = bo_at_low
        out["bo_volume_confirmation"] = bo_volume_conf
        out["bo_breakout_score"] = bo_score

        return out


## Independent MTF frames

In [ ]:
"""Independent multi-timeframe OHLCV construction and causal as-of alignment.

10m is built from two consecutive closed 5m bars outside the model. Higher TFs
are never resampled inside the encoder; they are joined with merge_asof on
fully closed bar close-time.
"""

from __future__ import annotations

from typing import Any, Dict, Mapping, Optional, Sequence

import numpy as np
import pandas as pd

def _ensure_time_column(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with UTC ``time`` plus original columns."""
    out = df.copy()
    if "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"], utc=True)
    elif "timestamp" in out.columns:
        ts = out["timestamp"]
        if pd.api.types.is_numeric_dtype(ts):
            out["time"] = pd.to_datetime(ts, unit="s", utc=True)
        else:
            out["time"] = pd.to_datetime(ts, utc=True)
    else:
        raise ValueError("OHLCV frame requires time or timestamp")
    return out.sort_values("time").reset_index(drop=True)

def bar_close_time(open_time: pd.Series, resolution_minutes: int) -> pd.Series:
    """Candle close time from open time (Delta bars are labeled at open)."""
    return pd.to_datetime(open_time, utc=True) + pd.Timedelta(
        minutes=int(resolution_minutes)
    )

def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Construct an independent 10m OHLCV series from two closed 5m bars.

    Aggregation is exchange-aligned (left-closed, left-labeled): each 10m bar
    covers ``[T, T+10m)`` and uses the two 5m opens at T and T+5m. High/low
    are the true extrema of those source bars. This is done *outside* the
    model — the encoder never resamples 5m on the forward pass.
    """
    src = _ensure_time_column(df5m)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()

    grouped = src.set_index("time")
    rule = "10min"
    agg = grouped.resample(rule, label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample(rule, label="left", closed="left").count()
    # Keep complete 2-bar buckets only so a partial last 10m is not a feature.
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    out = out.rename(columns={"time": "time"})
    out["timestamp"] = out["time"]
    return out.sort_values("time").reset_index(drop=True)

def closed_bars_asof(
    tf_df: pd.DataFrame,
    decision_time: pd.Timestamp,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """Rows of ``tf_df`` whose candle close time is ``<= decision_time``.

    A 1h bar that opened at 10:00 is still forming at 10:17 and is excluded.
    """
    frame = _ensure_time_column(tf_df)
    if frame.empty:
        return frame
    close_t = bar_close_time(frame["time"], resolution_minutes)
    t = pd.Timestamp(decision_time)
    if t.tzinfo is None:
        t = t.tz_localize("UTC")
    else:
        t = t.tz_convert("UTC")
    return frame.loc[close_t <= t].reset_index(drop=True)

def asof_join_last_closed(
    decision_times: Sequence[pd.Timestamp],
    tf_df: pd.DataFrame,
    *,
    resolution_minutes: int,
) -> pd.DataFrame:
    """For each decision time, attach the last fully closed TF bar (as-of)."""
    left = pd.DataFrame(
        {"decision_time": pd.to_datetime(list(decision_times), utc=True)}
    ).sort_values("decision_time")
    right = _ensure_time_column(tf_df)
    if right.empty:
        empty = left.copy()
        empty["time"] = pd.NaT
        return empty
    right = right.copy()
    right["close_time"] = bar_close_time(right["time"], resolution_minutes)
    right = right.sort_values("close_time")
    merged = pd.merge_asof(
        left,
        right,
        left_on="decision_time",
        right_on="close_time",
        direction="backward",
        allow_exact_matches=True,
    )
    # Guard: never keep a bar that closed after the decision clock.
    late = merged["close_time"].notna() & (merged["close_time"] > merged["decision_time"])
    if late.any():
        merged.loc[late, right.columns] = np.nan
    return merged

def last_n_closed_bars(
    tf_df: pd.DataFrame,
    decision_time: pd.Timestamp,
    *,
    resolution_minutes: int,
    window_len: int,
) -> pd.DataFrame:
    """Last ``window_len`` fully closed native bars at ``decision_time``."""
    closed = closed_bars_asof(
        tf_df, decision_time, resolution_minutes=resolution_minutes
    )
    if len(closed) < int(window_len):
        return closed
    return closed.iloc[-int(window_len) :].reset_index(drop=True)

def normalize_mtf_frames(
    frames: Mapping[str, pd.DataFrame],
) -> Dict[str, pd.DataFrame]:
    """Ensure every fusion TF frame has UTC ``time`` (build 10m if missing)."""
    out: Dict[str, pd.DataFrame] = {}
    for res, df in frames.items():
        if df is None or (isinstance(df, pd.DataFrame) and df.empty):
            continue
        out[str(res)] = _ensure_time_column(df)
    if "10m" not in out and "5m" in out:
        out["10m"] = build_10m_ohlcv_from_5m(out["5m"])
    return out

def fusion_frames_from_fetch(
    df5m: pd.DataFrame,
    df30m: pd.DataFrame,
    df1h: pd.DataFrame,
    df2h: pd.DataFrame,
    *,
    df10m: Optional[pd.DataFrame] = None,
) -> Dict[str, pd.DataFrame]:
    """Assemble the five independent OHLCV datasets used by the fused model."""
    frames: Dict[str, pd.DataFrame] = {
        "5m": df5m,
        "30m": df30m,
        "1h": df1h,
        "2h": df2h,
    }
    if df10m is not None and not df10m.empty:
        frames["10m"] = df10m
    return normalize_mtf_frames(frames)

def assert_no_lookahead(
    tf_df: pd.DataFrame,
    decision_time: pd.Timestamp,
    *,
    resolution_minutes: int,
) -> None:
    """Raise if any bar close time is after the decision clock."""
    closed = closed_bars_asof(
        tf_df, decision_time, resolution_minutes=resolution_minutes
    )
    if closed.empty:
        return
    close_t = bar_close_time(closed["time"], resolution_minutes)
    t = pd.Timestamp(decision_time)
    if t.tzinfo is None:
        t = t.tz_localize("UTC")
    else:
        t = t.tz_convert("UTC")
    if bool((close_t > t).any()):
        raise AssertionError(
            f"Lookahead: {resolution_minutes}m bar close after {t.isoformat()}"
        )

def resolution_minutes(resolution: str) -> int:
    res = str(resolution).strip().lower()
    if res not in RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(RESOLUTION_MINUTES[res])

def required_fusion_resolutions() -> Sequence[str]:
    return FUSION_INPUT_RESOLUTIONS

def frames_have_fusion_inputs(frames: Mapping[str, Any]) -> bool:
    """True when every fusion TF is present and non-empty."""
    for res in FUSION_INPUT_RESOLUTIONS:
        df = frames.get(res)
        if not isinstance(df, pd.DataFrame) or df.empty:
            return False
    return True


## Fusion 2-class labels

In [ ]:
"""Fusion labels: per-horizon ATR dead zone, 2-class BEAR/BULL for training.

Labels live on the 5m decision clock. Features at t are causal. Targets use
close[t+k] only. No MFE/MAE path heads. NEUTRAL is ignore_index (-1), not a class.
h30m/h1h use 0.5 ATR; h2h uses 0.75 ATR so weak chop is ignored.
"""

from __future__ import annotations

from typing import Dict, Tuple

import numpy as np
import pandas as pd

def _ensure_time(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"], utc=True)
    elif "timestamp" in out.columns:
        ts = out["timestamp"]
        if pd.api.types.is_numeric_dtype(ts):
            out["time"] = pd.to_datetime(ts, unit="s", utc=True)
        else:
            out["time"] = pd.to_datetime(ts, utc=True)
    else:
        raise ValueError("Label frame requires time or timestamp")
    return out.sort_values("time").reset_index(drop=True)

def horizon_atr_weak(horizon_key: str) -> float:
    """Per-head ATR dead-zone width. Unknown keys fall back to 0.5 ATR."""
    return float(FUSION_HORIZON_ATR_WEAK.get(str(horizon_key), HORIZON_DIR_ATR_WEAK))

def three_class_direction(move: float, atr: float) -> int:
    """Map ATR-normalized close-to-close move to BEAR/NEUTRAL/BULL (0/1/2)."""
    ratio = float(move) / max(float(atr), 1e-9)
    if ratio >= float(HORIZON_DIR_ATR_WEAK):
        return 2
    if ratio <= -float(HORIZON_DIR_ATR_WEAK):
        return 0
    return 1

def three_class_to_train_label(class_id: float) -> int:
    """Map internal 3-class id to 2-class train id. NEUTRAL/NaN → ignore_index."""
    if not np.isfinite(class_id):
        return int(FUSION_IGNORE_INDEX)
    cid = int(class_id)
    if cid == 0:
        return 0
    if cid == 2:
        return 1
    return int(FUSION_IGNORE_INDEX)

def compute_fusion_horizon_labels(df5m: pd.DataFrame) -> pd.DataFrame:
    """Label +30m/+1h/+2h position on the 5m grid (internal 3-class 0/1/2).

    ``y_h`` at bar t uses ``close[t+k] - close[t]`` over ATR at t.
    The last ``MAX_FUSION_HORIZON_BARS`` rows are NaN (insufficient future).
    Training maps NEUTRAL to ignore_index via ``fusion_label_matrix``.
    """
    out = _ensure_time(df5m)
    n = len(out)
    close = out["close"].to_numpy(dtype=np.float64)
    if "atr" in out.columns:
        atr = out["atr"].to_numpy(dtype=np.float64)
    else:
        prev = np.concatenate([[close[0]], close[:-1]])
        tr = np.maximum(
            out["high"].to_numpy(dtype=np.float64) - out["low"].to_numpy(dtype=np.float64),
            np.maximum(np.abs(out["high"].to_numpy() - prev), np.abs(out["low"].to_numpy() - prev)),
        )
        atr = pd.Series(tr).rolling(14, min_periods=1).mean().to_numpy(dtype=np.float64)
    max_k = int(MAX_FUSION_HORIZON_BARS)
    n_valid = max(n - max_k, 0)
    atr_valid = atr[:n_valid].copy()
    atr_valid = np.where(np.isfinite(atr_valid), atr_valid, 0.0)
    denom = np.maximum(atr_valid, 1e-9)
    for key, col, k in zip(FUSION_HORIZON_KEYS, FUSION_DIR_COLS, FUSION_HORIZON_BARS_5M):
        weak = horizon_atr_weak(str(key))
        labels = np.full(n, np.nan, dtype=np.float64)
        if n_valid > 0:
            move = close[int(k) : int(k) + n_valid] - close[:n_valid]
            ratio = move / denom
            cls = np.ones(n_valid, dtype=np.float64)
            cls[ratio >= weak] = 2.0
            cls[ratio <= -weak] = 0.0
            labels[:n_valid] = cls
        out[col] = labels
    return out

def trim_fusion_label_tail(df: pd.DataFrame) -> pd.DataFrame:
    """Drop the last 24 five-minute bars that cannot form a +2h live label."""
    if len(df) <= int(MAX_FUSION_HORIZON_BARS):
        return df.iloc[0:0].copy()
    return df.iloc[: -int(MAX_FUSION_HORIZON_BARS)].reset_index(drop=True)

def trim_label_v2_tail(df: pd.DataFrame) -> pd.DataFrame:
    """Drop the last 12 five-minute bars that cannot form a +1h Label V2 target."""
    if len(df) <= int(MAX_LABEL_V2_HORIZON_BARS):
        return df.iloc[0:0].copy()
    return df.iloc[: -int(MAX_LABEL_V2_HORIZON_BARS)].reset_index(drop=True)

def fusion_label_matrix(df: pd.DataFrame) -> np.ndarray:
    """(n, 3) int64 train labels: BEAR=0, BULL=1, NEUTRAL/NaN=-1."""
    cols = list(FUSION_DIR_COLS)
    arr = df.loc[:, cols].to_numpy(dtype=np.float64)
    out = np.full(arr.shape, int(FUSION_IGNORE_INDEX), dtype=np.int64)
    finite = np.isfinite(arr)
    mapped = np.full(arr.shape, int(FUSION_IGNORE_INDEX), dtype=np.int64)
    mapped[arr == 0] = 0
    mapped[arr == 2] = 1
    out[finite] = mapped[finite]
    return out

def fusion_future_leak_cols() -> frozenset:
    """Columns that must never appear as model inputs."""
    return frozenset(FUSION_DIR_COLS) | frozenset(FUSION_RETIRED_DIR_COLS)

def embargo_bars() -> int:
    return int(FUSION_EMBARGO_BARS)

def direction_name(class_id: int) -> str:
    cid = int(class_id)
    if cid < 0:
        return "IGNORED"
    return FUSION_DIRECTION_NAMES.get(cid, "IGNORED")

def horizon_key_to_col() -> Dict[str, str]:
    return {key: f"{key}_dir" for key in FUSION_HORIZON_KEYS}

def label_class_mix(y: np.ndarray) -> Dict[str, Dict[str, float]]:
    """Per-horizon 2-class counts plus ignore rate for reports."""
    report: Dict[str, Dict[str, float]] = {}
    for j, key in enumerate(FUSION_HORIZON_KEYS):
        col = y[:, j] if y.ndim == 2 else y
        n = int(len(col))
        ignored = int(np.sum(col < 0))
        counts: Dict[str, float] = {
            "BEAR": float(np.sum(col == 0)),
            "BULL": float(np.sum(col == 1)),
            "IGNORED": float(ignored),
            "ignore_rate": float(ignored) / float(max(n, 1)),
        }
        report[key] = counts
    return report

def valid_label_mask(y: np.ndarray) -> np.ndarray:
    """True where at least one horizon is a 2-class BEAR/BULL label."""
    if y.ndim == 1:
        return (y == 0) | (y == 1)
    return np.any((y == 0) | (y == 1), axis=1)

def split_hint() -> Tuple[int, int]:
    """(embargo_bars, max_horizon_bars) for purged splits."""
    return int(FUSION_EMBARGO_BARS), int(MAX_FUSION_HORIZON_BARS)


## Label V2 path targets

In [ ]:
"""Label V2 path targets: ATR-normalized return and side-aware MFE/MAE.

Research-only. Live v11 fusion remains 2-class close-to-close with ignore_index
NEUTRAL. Path windows are t+1..t+k; High_t / Low_t never enter the target.
"""

from __future__ import annotations

from typing import Any, Dict, Iterable, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

_EPS = 1e-9
_QUANTILES: Tuple[float, ...] = (0.10, 0.25, 0.50, 0.75, 0.90)
_BEAR = 0
_NEUTRAL = 1
_BULL = 2

def _ensure_ohlcv_time(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"], utc=True)
    elif "timestamp" in out.columns:
        ts = out["timestamp"]
        if pd.api.types.is_numeric_dtype(ts):
            out["time"] = pd.to_datetime(ts, unit="s", utc=True)
        else:
            out["time"] = pd.to_datetime(ts, utc=True)
    else:
        raise ValueError("Label frame requires time or timestamp")
    return out.sort_values("time").reset_index(drop=True)

def _atr_series(df: pd.DataFrame) -> np.ndarray:
    """Causal ATR at t. Falls back to 14-period mean true range."""
    n = len(df)
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    if n == 0:
        return np.zeros(0, dtype=np.float64)
    return pd.Series(tr).rolling(14, min_periods=1).mean().to_numpy(dtype=np.float64)

def _forward_stack(arr: np.ndarray, n_valid: int, k: int) -> np.ndarray:
    """Stack arr[i+1:i+k+1] rows for i in [0, n_valid). Shape (n_valid, k)."""
    kk = int(k)
    if n_valid <= 0 or kk <= 0:
        return np.zeros((max(n_valid, 0), max(kk, 0)), dtype=np.float64)
    return np.stack([arr[j : j + n_valid] for j in range(1, kk + 1)], axis=1)

def direction_from_return(r: float, theta: float) -> int:
    """Map ATR-normalized return to BEAR/NEUTRAL/BULL (0/1/2).

    Args:
        r: Forward return in ATR units.
        theta: Dead-zone half-width in ATR units. Must be positive.

    Returns:
        0 BEAR, 1 NEUTRAL, 2 BULL.

    Raises:
        ValueError: If ``r`` is not finite.
    """
    if not np.isfinite(r):
        raise ValueError("return must be finite")
    th = abs(float(theta))
    val = float(r)
    if val >= th:
        return _BULL
    if val <= -th:
        return _BEAR
    return _NEUTRAL

def direction_from_return_array(r: np.ndarray, theta: float) -> np.ndarray:
    """Vectorized :func:`direction_from_return`. Non-finite → -1."""
    arr = np.asarray(r, dtype=np.float64)
    th = abs(float(theta))
    out = np.ones(arr.shape, dtype=np.int64)
    finite = np.isfinite(arr)
    out[finite & (arr >= th)] = _BULL
    out[finite & (arr <= -th)] = _BEAR
    out[~finite] = -1
    return out

def _normalize_side(side: str) -> str:
    s = str(side or "LONG").strip().upper()
    if s in ("SHORT", "SELL", "BEAR", "STRONG_SELL"):
        return "SHORT"
    return "LONG"

def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk a future OHLC path and return TP_FIRST / SL_FIRST / NEITHER / AMBIGUOUS.

    Same-bar both-sides breach is AMBIGUOUS (not a forced winner).

    Args:
        fwd_high: Highs for bars t+1..t+k.
        fwd_low: Lows for bars t+1..t+k.
        entry: Close at t.
        atr: ATR at t.
        tp_mult: Take-profit distance in ATR.
        sl_mult: Stop-loss distance in ATR.
        side: ``LONG`` or ``SHORT``.

    Returns:
        Integer code from ``LABEL_V2_TP_SL_*``.
    """
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(LABEL_V2_TP_SL_NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if _normalize_side(side) == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(LABEL_V2_TP_SL_AMBIGUOUS)
        if hit_tp:
            return int(LABEL_V2_TP_SL_TP_FIRST)
        if hit_sl:
            return int(LABEL_V2_TP_SL_SL_FIRST)
    return int(LABEL_V2_TP_SL_NEITHER)

def tp_sl_from_windows(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: np.ndarray,
    atr: np.ndarray,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> np.ndarray:
    """Vectorized TP/SL outcome for stacked paths. Shape (n,)."""
    high = np.asarray(fwd_high, dtype=np.float64)
    low = np.asarray(fwd_low, dtype=np.float64)
    if high.ndim != 2 or low.shape != high.shape:
        raise ValueError("fwd_high and fwd_low must be (n, k)")
    n, k = high.shape
    out = np.full(n, int(LABEL_V2_TP_SL_NEITHER), dtype=np.int64)
    if n == 0 or k == 0:
        return out
    denom = np.maximum(np.asarray(atr, dtype=np.float64).reshape(n), _EPS)
    ent = np.asarray(entry, dtype=np.float64).reshape(n)
    if _normalize_side(side) == "SHORT":
        tp_hit = low <= (ent - float(tp_mult) * denom)[:, None]
        sl_hit = high >= (ent + float(sl_mult) * denom)[:, None]
    else:
        tp_hit = high >= (ent + float(tp_mult) * denom)[:, None]
        sl_hit = low <= (ent - float(sl_mult) * denom)[:, None]
    tp_idx = np.where(tp_hit.any(axis=1), tp_hit.argmax(axis=1), k)
    sl_idx = np.where(sl_hit.any(axis=1), sl_hit.argmax(axis=1), k)
    both = (tp_idx < k) & (sl_idx < k) & (tp_idx == sl_idx)
    out[(tp_idx < sl_idx) & (tp_idx < k)] = int(LABEL_V2_TP_SL_TP_FIRST)
    out[(sl_idx < tp_idx) & (sl_idx < k)] = int(LABEL_V2_TP_SL_SL_FIRST)
    out[both] = int(LABEL_V2_TP_SL_AMBIGUOUS)
    return out

def state_persistence(
    fwd_close: np.ndarray,
    entry: float,
    atr: float,
    theta: float,
) -> float:
    """Fraction of path closes matching the endpoint 3-class state."""
    path = np.asarray(fwd_close, dtype=np.float64).ravel()
    if path.size == 0:
        return 0.0
    denom = max(float(atr), _EPS)
    r_path = (path - float(entry)) / denom
    dirs = direction_from_return_array(r_path, theta)
    end = int(dirs[-1])
    if end < 0:
        return 0.0
    return float(np.mean(dirs == end))

def label_v2_future_leak_cols() -> frozenset:
    """Columns that must never appear as model inputs."""
    return frozenset(LABEL_V2_COLS)

def fusion_label_v2_matrices(
    df: pd.DataFrame,
    *,
    thetas: Optional[Mapping[str, float]] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """Build (n, H) 3-class dirs and (n, H, 5) path targets.

    Path last-dim order matches ``LABEL_V2_REG_FIELDS``:
    ret, long_mfe, long_mae, short_mfe, short_mae.
    Direction uses frozen per-horizon theta. Non-finite return maps to -1.
    """
    theta_map = dict(LABEL_V2_THETA_FROZEN)
    if thetas is not None:
        theta_map.update({str(k): float(v) for k, v in thetas.items()})
    n = int(len(df))
    n_h = len(LABEL_V2_HORIZON_KEYS)
    y_dir = np.full((n, n_h), -1, dtype=np.int64)
    y_reg = np.full((n, n_h, len(LABEL_V2_REG_FIELDS)), np.nan, dtype=np.float32)
    for j, key in enumerate(LABEL_V2_HORIZON_KEYS):
        ret = df[f"{key}_ret"].to_numpy(dtype=np.float64)
        y_dir[:, j] = direction_from_return_array(ret, float(theta_map[str(key)]))
        for k, field in enumerate(LABEL_V2_REG_FIELDS):
            col = f"{key}_{field}"
            if col not in df.columns:
                continue
            y_reg[:, j, k] = df[col].to_numpy(dtype=np.float32)
    return y_dir, y_reg

def valid_label_v2_mask(y_dir: np.ndarray, y_reg: np.ndarray) -> np.ndarray:
    """Keep rows with at least one finite path return (any horizon)."""
    del y_dir
    ret = np.asarray(y_reg, dtype=np.float32)[:, :, 0]
    return np.any(np.isfinite(ret), axis=1)

def label_v2_class_mix(y_dir: np.ndarray) -> Dict[str, Dict[str, float]]:
    """Per-horizon 3-class counts. Invalid dirs (<0) are unlabeled."""
    report: Dict[str, Dict[str, float]] = {}
    mat = np.asarray(y_dir, dtype=np.int64)
    for j, key in enumerate(LABEL_V2_HORIZON_KEYS):
        col = mat[:, j] if mat.ndim == 2 else mat
        n = int(len(col))
        unlabeled = int(np.sum(col < 0))
        counts: Dict[str, float] = {
            name: float(np.sum(col == int(cid)))
            for cid, name in LABEL_V2_DIRECTION_NAMES.items()
        }
        counts["UNLABELED"] = float(unlabeled)
        counts["n"] = float(n)
        counts["unlabeled_rate"] = float(unlabeled) / float(max(n, 1))
        report[str(key)] = counts
    return report

def compute_fusion_path_targets(df5m: pd.DataFrame) -> pd.DataFrame:
    """ATR-normalized path targets on the 5m decision clock.

    For each fusion horizon k, at bar t:

    * ``ret`` = (Close[t+k] - Close[t]) / ATR[t]
    * ``long_mfe`` = max(0, (max High[t+1:t+k] - Close[t]) / ATR[t])
    * ``short_mfe`` = max(0, (Close[t] - min Low[t+1:t+k]) / ATR[t])
    * ``long_mae`` = max(0, (Close[t] - min Low[t+1:t_long_mfe]) / ATR[t])
    * ``short_mae`` = max(0, (max High[t+1:t_short_mfe] - Close[t]) / ATR[t])
    * ``persist`` = mean(sign(Close[t+j] - Close[t])) for j=1..k
    * ``t_long_mfe`` / ``t_short_mfe`` = 1-based first max-high / min-low bar

    MAE windows are inclusive of the MFE bar (same-bar opposite wick counts).
    The last ``MAX_LABEL_V2_HORIZON_BARS`` rows are NaN (insufficient +1h future).
    """
    out = _ensure_ohlcv_time(df5m)
    n = len(out)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = _atr_series(out)
    max_k = int(MAX_LABEL_V2_HORIZON_BARS)
    n_valid = max(n - max_k, 0)
    denom = np.maximum(atr[:n_valid], _EPS) if n_valid else np.zeros(0)
    entry = close[:n_valid] if n_valid else np.zeros(0)

    for field in LABEL_V2_PATH_FIELDS:
        for key in LABEL_V2_HORIZON_KEYS:
            out[f"{key}_{field}"] = np.full(n, np.nan, dtype=np.float64)

    if n_valid == 0:
        return out

    for key, k in zip(LABEL_V2_HORIZON_KEYS, LABEL_V2_HORIZON_BARS_5M):
        kk = int(k)
        fwd_close = _forward_stack(close, n_valid, kk)
        fwd_high = _forward_stack(high, n_valid, kk)
        fwd_low = _forward_stack(low, n_valid, kk)
        t_long = fwd_high.argmax(axis=1)
        t_short = fwd_low.argmin(axis=1)
        bar_idx = np.arange(kk, dtype=np.int64)
        long_mask = bar_idx[None, :] <= t_long[:, None]
        short_mask = bar_idx[None, :] <= t_short[:, None]
        min_low_until = np.where(long_mask, fwd_low, np.inf).min(axis=1)
        max_high_until = np.where(short_mask, fwd_high, -np.inf).max(axis=1)
        long_mfe = np.maximum((fwd_high.max(axis=1) - entry) / denom, 0.0)
        short_mfe = np.maximum((entry - fwd_low.min(axis=1)) / denom, 0.0)
        long_mae = np.maximum((entry - min_low_until) / denom, 0.0)
        short_mae = np.maximum((max_high_until - entry) / denom, 0.0)
        ret = (fwd_close[:, -1] - entry) / denom
        delta = fwd_close - entry[:, None]
        persist = np.sign(delta).mean(axis=1)
        t_long_mfe = t_long.astype(np.float64) + 1.0
        t_short_mfe = t_short.astype(np.float64) + 1.0
        atr_ok = np.isfinite(atr[:n_valid]) & (atr[:n_valid] > _EPS)
        nan_bad = ~atr_ok
        for arr in (
            ret,
            long_mfe,
            long_mae,
            short_mfe,
            short_mae,
            persist,
            t_long_mfe,
            t_short_mfe,
        ):
            arr[nan_bad] = np.nan
        out.loc[: n_valid - 1, f"{key}_ret"] = ret
        out.loc[: n_valid - 1, f"{key}_long_mfe"] = long_mfe
        out.loc[: n_valid - 1, f"{key}_long_mae"] = long_mae
        out.loc[: n_valid - 1, f"{key}_short_mfe"] = short_mfe
        out.loc[: n_valid - 1, f"{key}_short_mae"] = short_mae
        out.loc[: n_valid - 1, f"{key}_persist"] = persist
        out.loc[: n_valid - 1, f"{key}_t_long_mfe"] = t_long_mfe
        out.loc[: n_valid - 1, f"{key}_t_short_mfe"] = t_short_mfe
    return out

def _py(value: Any) -> Any:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        val = float(value)
        return val if np.isfinite(val) else None
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, np.ndarray):
        return [_py(v) for v in value.tolist()]
    if isinstance(value, dict):
        return {str(k): _py(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_py(v) for v in value]
    return value

def _spearman(left: np.ndarray, right: np.ndarray) -> float:
    a = np.asarray(left, dtype=np.float64)
    b = np.asarray(right, dtype=np.float64)
    mask = np.isfinite(a) & np.isfinite(b)
    if int(mask.sum()) < 3:
        return float("nan")
    ra = pd.Series(a[mask]).rank()
    rb = pd.Series(b[mask]).rank()
    return float(ra.corr(rb))

def _quantile_map(arr: np.ndarray) -> Dict[str, Optional[float]]:
    finite = np.asarray(arr, dtype=np.float64)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return {f"p{int(q * 100)}": None for q in _QUANTILES}
    qs = np.quantile(finite, _QUANTILES)
    return {f"p{int(q * 100)}": _py(v) for q, v in zip(_QUANTILES, qs)}

def _outcome_mix(codes: np.ndarray) -> Dict[str, float]:
    n = int(len(codes))
    mix: Dict[str, float] = {}
    for cid, name in LABEL_V2_TP_SL_NAMES.items():
        mix[name] = float(np.sum(codes == int(cid))) / float(max(n, 1))
        mix[f"{name}_n"] = float(np.sum(codes == int(cid)))
    mix["n"] = float(n)
    return mix

def _expectancy_atr(
    codes: np.ndarray,
    ret: np.ndarray,
    tp_mult: float,
    sl_mult: float,
) -> Optional[float]:
    valid = codes != int(LABEL_V2_TP_SL_AMBIGUOUS)
    if not np.any(valid):
        return None
    pnl = np.full(codes.shape, np.nan, dtype=np.float64)
    pnl[codes == int(LABEL_V2_TP_SL_TP_FIRST)] = float(tp_mult)
    pnl[codes == int(LABEL_V2_TP_SL_SL_FIRST)] = -float(sl_mult)
    neither = codes == int(LABEL_V2_TP_SL_NEITHER)
    pnl[neither] = ret[neither]
    use = valid & np.isfinite(pnl)
    if not np.any(use):
        return None
    return float(np.mean(pnl[use]))

def _class_block(
    mask: np.ndarray,
    ret: np.ndarray,
    long_mfe: np.ndarray,
    short_mfe: np.ndarray,
    long_mae: np.ndarray,
    short_mae: np.ndarray,
    persist: np.ndarray,
    state_persist: np.ndarray,
    n_total: int,
) -> Dict[str, Any]:
    n = int(np.sum(mask))
    block: Dict[str, Any] = {
        "n": n,
        "rate": float(n) / float(max(n_total, 1)),
    }
    empty_q = _quantile_map(np.array([]))
    if n == 0:
        block.update(
            {
                "e_r": None,
                "r_q": empty_q,
                "mfe_q": empty_q,
                "mae_q": empty_q,
                "long_mae_q": empty_q,
                "short_mae_q": empty_q,
                "persist_mean": None,
                "state_persist_mean": None,
            }
        )
        return block
    block["e_r"] = _py(np.mean(ret[mask]))
    block["r_q"] = _quantile_map(ret[mask])
    block["mfe_q"] = _quantile_map(long_mfe[mask])
    block["mae_q"] = _quantile_map(short_mfe[mask])
    block["long_mae_q"] = _quantile_map(long_mae[mask])
    block["short_mae_q"] = _quantile_map(short_mae[mask])
    block["persist_mean"] = _py(np.mean(persist[mask]))
    block["state_persist_mean"] = _py(np.mean(state_persist[mask]))
    return block

def _theta_key(theta: float) -> str:
    return f"{float(theta):.2f}"

def _pair_key(sl_mult: float, tp_mult: float) -> str:
    return f"sl{float(sl_mult):.2f}_tp{float(tp_mult):.2f}"

def _gate_one_horizon(
    by_theta: Mapping[str, Mapping[str, Any]],
    live_pair_key: str,
) -> Dict[str, Any]:
    """Apply the five Label V2 promotion checks per theta."""
    ranked: list[Dict[str, Any]] = []
    for tkey, row in by_theta.items():
        flags: list[str] = []
        rates = {
            name: float((row.get("classes") or {}).get(name, {}).get("rate") or 0.0)
            for name in LABEL_V2_DIRECTION_NAMES.values()
        }
        minority = min(rates.values()) if rates else 0.0
        if minority < float(LABEL_V2_MINORITY_RATE):
            flags.append(f"minority_class<{LABEL_V2_MINORITY_RATE:.2f}")
        e_bull = (row.get("classes") or {}).get("BULL", {}).get("e_r")
        e_bear = (row.get("classes") or {}).get("BEAR", {}).get("e_r")
        theta = float(row.get("theta") or 0.0)
        separated = (
            e_bull is not None
            and e_bear is not None
            and float(e_bull) > 0.0
            and float(e_bear) < 0.0
            and (float(e_bull) - float(e_bear)) >= theta
        )
        if not separated:
            flags.append("return_classes_not_separated")
        disagree = float(row.get("disagreement_rate") or 0.0)
        if disagree < float(LABEL_V2_DISAGREE_MIN):
            flags.append("path_redundant_with_endpoint")
        tp_grid = row.get("tp_sl") or {}
        live = tp_grid.get(live_pair_key) or {}
        long_mix = live.get("long") or {}
        tp_n = float(long_mix.get("TP_FIRST_n") or 0.0)
        amb_n = float(long_mix.get("AMBIGUOUS_n") or 0.0)
        wins = tp_n + amb_n
        if wins > 0 and (amb_n / wins) >= 0.5:
            flags.append("ambiguous_majority_of_tp_wins")
        persist_vals = [
            (row.get("classes") or {}).get(name, {}).get("persist_mean")
            for name in LABEL_V2_DIRECTION_NAMES.values()
        ]
        finite_p = [float(v) for v in persist_vals if v is not None]
        persist_range = (max(finite_p) - min(finite_p)) if len(finite_p) >= 2 else 0.0
        if persist_range < float(LABEL_V2_PERSIST_RANGE_MIN):
            flags.append("persistence_flat")
        ranked.append(
            {
                "theta": theta,
                "theta_key": tkey,
                "pass": len(flags) == 0,
                "flags": flags,
                "minority_rate": minority,
                "disagreement_rate": disagree,
                "persist_range": persist_range,
            }
        )
    passing = [r for r in ranked if r["pass"]]
    chosen = None
    status = "no-go"
    if passing:
        prefer = [r for r in passing if abs(float(r["theta"]) - 0.50) < 1e-9]
        chosen = prefer[0] if prefer else passing[0]
        status = "go"
    return {
        "status": status,
        "chosen_theta": None if chosen is None else chosen["theta"],
        "candidates": ranked,
    }

def summarize_label_v2(
    df5m: pd.DataFrame,
    *,
    thetas: Sequence[float] = LABEL_V2_THETA_GRID,
    tp_sl_grid: Sequence[Tuple[float, float]] = LABEL_V2_TP_SL_GRID,
    already_labeled: bool = False,
) -> Dict[str, Any]:
    """θ and TP/SL distribution report. Does not train a model.

    Args:
        df5m: 5m OHLCV (and optional ATR). Path columns are computed unless
            ``already_labeled`` is true.
        thetas: Dead-zone grid.
        tp_sl_grid: ``(sl_mult, tp_mult)`` pairs.
        already_labeled: Skip :func:`compute_fusion_path_targets` when columns exist.

    Returns:
        Nested JSON-ready dict with per-horizon stats and a go/no-go gate.
    """
    labeled = df5m if already_labeled else compute_fusion_path_targets(df5m)
    labeled = _ensure_ohlcv_time(labeled)
    n_source = len(labeled)
    close_all = labeled["close"].to_numpy(dtype=np.float64)
    high_all = labeled["high"].to_numpy(dtype=np.float64)
    low_all = labeled["low"].to_numpy(dtype=np.float64)
    atr_all = _atr_series(labeled)
    trimmed = trim_label_v2_tail(labeled)
    n = len(trimmed)
    close = close_all[:n]
    atr = atr_all[:n]
    span: Dict[str, Any] = {"n": n, "n_source": n_source}
    if n and "time" in trimmed.columns:
        span["start"] = str(trimmed["time"].iloc[0])
        span["end"] = str(trimmed["time"].iloc[-1])

    horizons: Dict[str, Any] = {}
    stacks: Dict[str, Tuple[np.ndarray, np.ndarray, np.ndarray]] = {}
    for key, k in zip(LABEL_V2_HORIZON_KEYS, LABEL_V2_HORIZON_BARS_5M):
        stacks[str(key)] = (
            _forward_stack(high_all, n, int(k)),
            _forward_stack(low_all, n, int(k)),
            _forward_stack(close_all, n, int(k)),
        )

    for key, k in zip(LABEL_V2_HORIZON_KEYS, LABEL_V2_HORIZON_BARS_5M):
        kk = int(k)
        ret = trimmed[f"{key}_ret"].to_numpy(dtype=np.float64)
        long_mfe = trimmed[f"{key}_long_mfe"].to_numpy(dtype=np.float64)
        short_mfe = trimmed[f"{key}_short_mfe"].to_numpy(dtype=np.float64)
        long_mae = trimmed[f"{key}_long_mae"].to_numpy(dtype=np.float64)
        short_mae = trimmed[f"{key}_short_mae"].to_numpy(dtype=np.float64)
        persist = trimmed[f"{key}_persist"].to_numpy(dtype=np.float64)
        valid = (
            np.isfinite(ret)
            & np.isfinite(long_mfe)
            & np.isfinite(short_mfe)
            & np.isfinite(long_mae)
            & np.isfinite(short_mae)
        )
        n_valid = int(np.sum(valid))
        fwd_high, fwd_low, fwd_close = stacks[str(key)]
        entry = close
        denom = np.maximum(atr, _EPS)
        live_sl, live_tp = LABEL_V2_DURATION_ATR_MULT[str(key)]
        live_key = _pair_key(live_sl, live_tp)
        by_theta: Dict[str, Any] = {}
        stride = np.zeros(n, dtype=bool)
        if n:
            stride[::kk] = True

        for theta in thetas:
            dirs = direction_from_return_array(ret, float(theta))
            r_path = (fwd_close - entry[:, None]) / denom[:, None]
            path_dirs = direction_from_return_array(r_path, float(theta))
            end_dir = path_dirs[:, -1] if path_dirs.size else np.zeros(n, dtype=np.int64)
            state_p = np.mean(path_dirs == end_dir[:, None], axis=1) if n else np.zeros(0)
            long_live = tp_sl_from_windows(
                fwd_high, fwd_low, entry, atr, live_tp, live_sl, "LONG"
            )
            short_live = tp_sl_from_windows(
                fwd_high, fwd_low, entry, atr, live_tp, live_sl, "SHORT"
            )
            bear = (dirs == _BEAR) & valid
            bull = (dirs == _BULL) & valid
            n_dir = int(np.sum(bear | bull))
            disagree_n = int(
                np.sum(bear & (long_live == int(LABEL_V2_TP_SL_TP_FIRST)))
                + np.sum(bull & (short_live == int(LABEL_V2_TP_SL_TP_FIRST)))
            )
            classes = {}
            for cid, name in LABEL_V2_DIRECTION_NAMES.items():
                mask = (dirs == int(cid)) & valid
                classes[name] = _class_block(
                    mask,
                    ret,
                    long_mfe,
                    short_mfe,
                    long_mae,
                    short_mae,
                    persist,
                    state_p,
                    n_valid,
                )
                mask_stride = mask & stride
                n_stride = int(np.sum(valid & stride))
                classes[name]["nonoverlap"] = _class_block(
                    mask_stride,
                    ret,
                    long_mfe,
                    short_mfe,
                    long_mae,
                    short_mae,
                    persist,
                    state_p,
                    n_stride,
                )
            tp_sl_stats: Dict[str, Any] = {}
            for sl_m, tp_m in tp_sl_grid:
                long_c = tp_sl_from_windows(
                    fwd_high, fwd_low, entry, atr, tp_m, sl_m, "LONG"
                )
                short_c = tp_sl_from_windows(
                    fwd_high, fwd_low, entry, atr, tp_m, sl_m, "SHORT"
                )
                chosen = np.full(n, int(LABEL_V2_TP_SL_NEITHER), dtype=np.int64)
                chosen[bull] = long_c[bull]
                chosen[bear] = short_c[bear]
                tp_sl_stats[_pair_key(sl_m, tp_m)] = {
                    "sl_mult": float(sl_m),
                    "tp_mult": float(tp_m),
                    "live_bracket": abs(float(sl_m) - float(live_sl)) < 1e-12
                    and abs(float(tp_m) - float(live_tp)) < 1e-12,
                    "long": _outcome_mix(long_c[valid]),
                    "short": _outcome_mix(short_c[valid]),
                    "side_from_direction": _outcome_mix(chosen[bear | bull]),
                    "expectancy_atr": _expectancy_atr(
                        chosen[bear | bull],
                        ret[bear | bull],
                        float(tp_m),
                        float(sl_m),
                    ),
                }
            by_theta[_theta_key(float(theta))] = {
                "theta": float(theta),
                "n_valid": n_valid,
                "classes": classes,
                "corr_r_mfe": _py(_spearman(ret[valid], long_mfe[valid])),
                "corr_r_abs_mae": _py(_spearman(ret[valid], short_mfe[valid])),
                "corr_r_long_mae": _py(_spearman(ret[valid], long_mae[valid])),
                "corr_r_short_mae": _py(_spearman(ret[valid], short_mae[valid])),
                "disagreement_rate": float(disagree_n) / float(max(n_dir, 1)),
                "disagreement_n": disagree_n,
                "n_directional": n_dir,
                "bear_long_tp_first": float(
                    np.sum(bear & (long_live == int(LABEL_V2_TP_SL_TP_FIRST)))
                )
                / float(max(int(np.sum(bear)), 1)),
                "bull_short_tp_first": float(
                    np.sum(bull & (short_live == int(LABEL_V2_TP_SL_TP_FIRST)))
                )
                / float(max(int(np.sum(bull)), 1)),
                "tp_sl": tp_sl_stats,
            }

        cross: Dict[str, Any] = {}
        r10 = trimmed["h10m_ret"].to_numpy(dtype=np.float64)
        r15 = trimmed["h15m_ret"].to_numpy(dtype=np.float64)
        r1h = trimmed["h1h_ret"].to_numpy(dtype=np.float64)
        both = np.isfinite(r10) & np.isfinite(r15) & np.isfinite(r1h)
        for theta in thetas:
            d10 = direction_from_return_array(r10, float(theta))
            d15 = direction_from_return_array(r15, float(theta))
            d1h = direction_from_return_array(r1h, float(theta))
            n10 = int(np.sum(both & (d10 == _BULL)))
            cross[_theta_key(float(theta))] = {
                "p_10m_bull_and_15m_bull": float(
                    np.sum(both & (d10 == _BULL) & (d15 == _BULL))
                )
                / float(max(n10, 1)),
                "p_10m_bull_and_1h_not_bull": float(
                    np.sum(both & (d10 == _BULL) & (d1h != _BULL))
                )
                / float(max(n10, 1)),
                "n_10m_bull": n10,
            }

        gate = _gate_one_horizon(by_theta, live_key)
        horizons[str(key)] = {
            "bars": kk,
            "live_sl_atr": float(live_sl),
            "live_tp_atr": float(live_tp),
            "corr_r_mfe": _py(_spearman(ret[valid], long_mfe[valid])),
            "corr_r_abs_mae": _py(_spearman(ret[valid], short_mfe[valid])),
            "corr_r_long_mae": _py(_spearman(ret[valid], long_mae[valid])),
            "corr_r_short_mae": _py(_spearman(ret[valid], short_mae[valid])),
            "by_theta": by_theta,
            "cross_horizon": cross,
            "gate": gate,
        }

    overall = "go"
    chosen: Dict[str, Any] = {}
    for key, row in horizons.items():
        gate = row["gate"]
        chosen[key] = gate.get("chosen_theta")
        if gate.get("status") != "go":
            overall = "no-go"

    return _py(
        {
            "span": span,
            "thetas": [float(t) for t in thetas],
            "tp_sl_grid": [{"sl": a, "tp": b} for a, b in tp_sl_grid],
            "horizons": horizons,
            "chosen_theta": chosen,
            "overall": overall,
        }
    )

def format_label_v2_table(report: Mapping[str, Any]) -> str:
    """Compact stdout table for the Label V2 sweep."""
    lines = [
        f"overall={report.get('overall')}  n={((report.get('span') or {}).get('n'))}",
        f"{'h':<6} {'th':>5} {'BEAR':>6} {'NEU':>6} {'BULL':>6} "
        f"{'E[R+/-]':>11} {'disagree':>8} {'gate':<16}",
    ]
    for key in LABEL_V2_HORIZON_KEYS:
        hrow = (report.get("horizons") or {}).get(key) or {}
        by_theta = hrow.get("by_theta") or {}
        gate = hrow.get("gate") or {}
        chosen = gate.get("chosen_theta")
        for _tkey, row in by_theta.items():
            classes = row.get("classes") or {}
            e_b = (classes.get("BULL") or {}).get("e_r")
            e_s = (classes.get("BEAR") or {}).get("e_r")
            sep = ""
            if e_b is not None and e_s is not None:
                sep = f"{float(e_b):+.2f}/{float(e_s):+.2f}"
            chosen_hit = (
                chosen is not None
                and abs(float(row["theta"]) - float(chosen)) < 1e-9
            )
            mark = "*" if chosen_hit else " "
            status = str(gate.get("status") or "")
            lines.append(
                f"{mark}{key:<5} {float(row['theta']):5.2f} "
                f"{(classes.get('BEAR') or {}).get('rate', 0):6.1%} "
                f"{(classes.get('NEUTRAL') or {}).get('rate', 0):6.1%} "
                f"{(classes.get('BULL') or {}).get('rate', 0):6.1%} "
                f"{sep:>11} {float(row.get('disagreement_rate') or 0):8.1%} "
                f"{status:<16}"
            )
    lines.append("(* = chosen theta)")
    return "\n".join(lines)

def iter_theta_grid(
    thetas: Optional[Iterable[float]] = None,
) -> Tuple[float, ...]:
    if thetas is None:
        return tuple(float(t) for t in LABEL_V2_THETA_GRID)
    return tuple(float(t) for t in thetas)


## Per-TF native features

In [ ]:
"""Per-timeframe pattern encodings for the fused multi-TF model.

Each TF is featured on its *native* grid. Resampled HTF structure is never
added; 10m/30m/1h/2h representations come from independently sampled OHLCV.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

def fusion_feature_cols() -> Tuple[str, ...]:
    """Continuous columns in each TF window (native TA + pattern engines)."""
    return fusion_native_feature_cols() + tuple(CANDLESTICK_FEATURES) + tuple(
        CHART_PATTERN_FEATURES
    )

def fusion_feature_fingerprint() -> str:
    """Stable hash of fusion feature names for dataset-cache invalidation."""
    blob = "\0".join(fusion_feature_cols()).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()[:16]

def fusion_feature_fingerprint_v14() -> str:
    """Stable hash of the v14 continuous column list."""
    blob = "\0".join(fusion_feature_cols_v14()).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()[:16]

def featured_cache_manifest() -> Dict[str, Any]:
    """Identity for cached native-TF featured frames (independent of stride)."""
    cols = list(fusion_feature_cols())
    return {
        "feature_contract_version": FEATURE_CONTRACT_VERSION_V11,
        "n_features": len(cols),
        "feature_fingerprint": fusion_feature_fingerprint(),
    }

def try_load_featured_frames(cache_dir: Path) -> Optional[Dict[str, pd.DataFrame]]:
    """Load featured_{tf}.parquet when featured_manifest.json matches."""
    root = Path(cache_dir)
    man_path = root / "featured_manifest.json"
    if not man_path.is_file():
        return None
    try:
        stored = json.loads(man_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None
    expected = featured_cache_manifest()
    for key, value in expected.items():
        if stored.get(key) != value:
            return None
    out: Dict[str, pd.DataFrame] = {}
    for res in FUSION_INPUT_RESOLUTIONS:
        path = root / f"featured_{res}.parquet"
        if not path.is_file():
            return None
        out[str(res)] = pd.read_parquet(path)
    return out

def save_featured_frames(
    cache_dir: Path,
    featured: Mapping[str, pd.DataFrame],
) -> None:
    """Write featured parquets plus featured_manifest.json."""
    root = Path(cache_dir)
    root.mkdir(parents=True, exist_ok=True)
    for res in FUSION_INPUT_RESOLUTIONS:
        frame = featured.get(res)
        if frame is None:
            raise KeyError(f"Missing featured frame for {res}")
        frame.to_parquet(root / f"featured_{res}.parquet", index=False)
    (root / "featured_manifest.json").write_text(
        json.dumps(featured_cache_manifest(), indent=2),
        encoding="utf-8",
    )

def add_native_tf_features(
    df: pd.DataFrame,
    *,
    resolution: str,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
    atr_period: int = 14,
) -> pd.DataFrame:
    """Causal features on one independently sampled TF. No HTF resample."""
    res = str(resolution).strip().lower()
    minutes = int(RESOLUTION_MINUTES[res])
    raw = assemble_raw_frame(df, funding_df=funding_df, oi_df=oi_df)
    feat = add_features(
        raw,
        resolution_minutes=minutes,
        atr_period=atr_period,
        include_htf=False,
    )
    candle_engine = CandlestickPatternEngine()
    chart_engine = ChartPatternEngine()
    cdl = candle_engine.compute_all(feat)
    chart = chart_engine.compute_all(feat, atr_period=atr_period)
    extra = pd.concat([cdl, chart], axis=1)
    extra = extra.loc[:, ~extra.columns.duplicated()]
    overlap = [c for c in extra.columns if c in feat.columns]
    if overlap:
        extra = extra.drop(columns=overlap)
    if not extra.empty:
        feat = pd.concat([feat, extra], axis=1)
    missing = [c for c in fusion_feature_cols() if c not in feat.columns]
    if missing:
        zeros = pd.DataFrame(0.0, index=feat.index, columns=missing)
        feat = pd.concat([feat, zeros], axis=1)
    if CANDLE_CLASS_COL not in feat.columns:
        feat[CANDLE_CLASS_COL] = 0
    if CHART_PATTERN_COL not in feat.columns:
        feat[CHART_PATTERN_COL] = 0
    return feat

def _window_matrix(
    feat_df: pd.DataFrame,
    *,
    cols: Sequence[str],
    window_len: int,
) -> np.ndarray:
    values = feat_df.loc[:, list(cols)].to_numpy(dtype=np.float64)
    values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
    if len(values) < int(window_len):
        pad = np.zeros((int(window_len) - len(values), values.shape[1]), dtype=np.float64)
        values = np.vstack([pad, values]) if len(values) else pad
    else:
        values = values[-int(window_len) :]
    return values.astype(np.float32)

def encode_tf_window(
    tf_df: pd.DataFrame,
    decision_time: pd.Timestamp,
    *,
    resolution: str,
    window_len: Optional[int] = None,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
    featured: Optional[pd.DataFrame] = None,
    zscore: bool = True,
) -> np.ndarray:
    """Native closed-bar window for one TF at decision time T (closed bars only)."""
    minutes = int(RESOLUTION_MINUTES[str(resolution).strip().lower()])
    width = (
        int(window_len)
        if window_len is not None
        else fusion_window_len(str(resolution))
    )
    if featured is None:
        closed = last_n_closed_bars(
            tf_df,
            decision_time,
            resolution_minutes=minutes,
            window_len=max(int(width) * 4, 256),
        )
        featured = add_native_tf_features(
            closed,
            resolution=resolution,
            funding_df=funding_df,
            oi_df=oi_df,
        )
        featured = last_n_closed_bars(
            featured,
            decision_time,
            resolution_minutes=minutes,
            window_len=int(width),
        )
    else:
        featured = last_n_closed_bars(
            featured,
            decision_time,
            resolution_minutes=minutes,
            window_len=int(width),
        )
    cols = fusion_feature_cols()
    window = _window_matrix(featured, cols=cols, window_len=int(width))
    if zscore:
        window = zscore_window(window)
    return window

def encode_all_tf_windows(
    frames: Mapping[str, pd.DataFrame],
    decision_time: pd.Timestamp,
    *,
    window_len: Optional[int] = None,
    window_lens: Optional[Mapping[str, int]] = None,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
    featured_by_tf: Optional[Mapping[str, pd.DataFrame]] = None,
    zscore: bool = True,
) -> Dict[str, np.ndarray]:
    """Independent Z-ready windows for 5m/10m/30m/1h/2h at time T."""
    normalized = normalize_mtf_frames(frames)
    lens = resolve_fusion_window_lens(window_lens, window_len)
    out: Dict[str, np.ndarray] = {}
    for res in FUSION_INPUT_RESOLUTIONS:
        df = normalized.get(res)
        if df is None or df.empty:
            raise ValueError(f"Missing independent OHLCV for {res}")
        feat = None if featured_by_tf is None else featured_by_tf.get(res)
        out[res] = encode_tf_window(
            df,
            decision_time,
            resolution=res,
            window_len=int(lens[res]),
            funding_df=funding_df,
            oi_df=oi_df,
            featured=feat,
            zscore=zscore,
        )
    return out

def precompute_featured_frames(
    frames: Mapping[str, pd.DataFrame],
    *,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
) -> Dict[str, pd.DataFrame]:
    """Feature each TF once (training). Do not resample across TFs."""
    normalized = normalize_mtf_frames(frames)
    featured: Dict[str, pd.DataFrame] = {}
    for res in FUSION_INPUT_RESOLUTIONS:
        df = normalized.get(res)
        if df is None or df.empty:
            raise ValueError(f"Missing independent OHLCV for {res}")
        featured[res] = add_native_tf_features(
            df,
            resolution=res,
            funding_df=funding_df if res == "5m" else None,
            oi_df=oi_df if res == "5m" else None,
        )
    return featured

def stack_tf_windows(
    windows: Mapping[str, np.ndarray],
) -> np.ndarray:
    """Stack TF windows as (n_tf, window_len, n_features) in fusion order.

    Requires equal (window, feat) shapes. Per-TF equal-span windows differ in
    bar count, so callers should keep a dict or list instead of stacking.
    """
    mats: List[np.ndarray] = []
    for res in FUSION_INPUT_RESOLUTIONS:
        if res not in windows:
            raise KeyError(f"Missing window for {res}")
        mats.append(np.asarray(windows[res], dtype=np.float32))
    shapes = {tuple(mat.shape) for mat in mats}
    if len(shapes) != 1:
        raise ValueError(
            "stack_tf_windows requires equal window shapes; got "
            + ", ".join(
                f"{res}={tuple(mat.shape)}"
                for res, mat in zip(FUSION_INPUT_RESOLUTIONS, mats)
            )
        )
    return np.stack(mats, axis=0)

_FUSION_WINDOW_CHUNK = 4096
_V14_ATR_CLIP = 8.0
_V14_EPS = 1e-9

def scale_v14_feature_window(
    window: np.ndarray,
    *,
    feature_cols: Sequence[str],
    atr: Optional[np.ndarray] = None,
    resolution_minutes: int,
    zscore_returns: bool = True,
) -> np.ndarray:
    """Channel-specific scale for a (T, F) or (N, T, F) v14 window.

    RSI and ADX keep level via affine maps. MACD is ATR-normalized. Counts are
    divided by the structure lookback. ``ret_1`` is z-scored over time.
    Remaining columns pass through. Id sequences are not in this matrix.
    """
    out = np.array(window, dtype=np.float32, copy=True)
    cols = [str(c) for c in feature_cols]
    idx = {name: i for i, name in enumerate(cols)}
    if "rsi_14" in idx:
        out[..., idx["rsi_14"]] = (out[..., idx["rsi_14"]] - 50.0) / 50.0
    if "adx_14" in idx:
        out[..., idx["adx_14"]] = out[..., idx["adx_14"]] / 100.0
    if "macd_hist" in idx:
        if atr is None:
            atr_safe = np.ones(out.shape[:-1], dtype=np.float32)
        else:
            atr_arr = np.asarray(atr, dtype=np.float32)
            if atr_arr.ndim == out.ndim:
                atr_arr = np.squeeze(atr_arr, axis=-1)
            if atr_arr.shape != out.shape[:-1]:
                atr_arr = np.reshape(atr_arr, out.shape[:-1])
            atr_safe = np.maximum(np.abs(atr_arr), _V14_EPS)
        scaled = out[..., idx["macd_hist"]] / atr_safe
        out[..., idx["macd_hist"]] = np.clip(
            scaled, -_V14_ATR_CLIP, _V14_ATR_CLIP
        )
    lookback = float(scale_period(V14_STRUCTURE_LOOKBACK_BARS, resolution_minutes))
    lookback = max(lookback, 1.0)
    for col in V14_COUNT_COLS:
        if col in idx:
            out[..., idx[col]] = out[..., idx[col]] / lookback
    if zscore_returns and "ret_1" in idx:
        i = idx["ret_1"]
        ret = out[..., i]
        time_axis = 1 if out.ndim == 3 else 0
        mu = ret.mean(axis=time_axis, keepdims=True)
        sd = ret.std(axis=time_axis, keepdims=True) + 1e-6
        out[..., i] = (ret - mu) / sd
    return out

def _id_window_vector(
    feat_df: pd.DataFrame,
    *,
    col: str,
    window_len: int,
    max_id: int,
) -> np.ndarray:
    width = int(window_len)
    if col not in feat_df.columns:
        return np.zeros((width,), dtype=np.int64)
    values = pd.to_numeric(feat_df[col], errors="coerce").fillna(0).to_numpy()
    values = np.clip(values.astype(np.int64, copy=False), 0, int(max_id))
    if len(values) < width:
        pad = np.zeros((width - len(values),), dtype=np.int64)
        values = np.concatenate([pad, values]) if len(values) else pad
    else:
        values = values[-width:]
    return values.astype(np.int64, copy=False)

def encode_tf_window_v14(
    tf_df: pd.DataFrame,
    decision_time: pd.Timestamp,
    *,
    resolution: str,
    window_len: Optional[int] = None,
    funding_df: Optional[pd.DataFrame] = None,
    oi_df: Optional[pd.DataFrame] = None,
    featured: Optional[pd.DataFrame] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """v14 closed-bar window: scaled floats plus candle and chart ids."""
    minutes = int(RESOLUTION_MINUTES[str(resolution).strip().lower()])
    width = (
        int(window_len)
        if window_len is not None
        else fusion_window_len(str(resolution))
    )
    if featured is None:
        closed = last_n_closed_bars(
            tf_df,
            decision_time,
            resolution_minutes=minutes,
            window_len=max(int(width) * 4, 256),
        )
        featured = add_native_tf_features(
            closed,
            resolution=resolution,
            funding_df=funding_df,
            oi_df=oi_df,
        )
        featured = last_n_closed_bars(
            featured,
            decision_time,
            resolution_minutes=minutes,
            window_len=int(width),
        )
    else:
        featured = last_n_closed_bars(
            featured,
            decision_time,
            resolution_minutes=minutes,
            window_len=int(width),
        )
    cols = list(fusion_feature_cols_v14())
    window = _window_matrix(featured, cols=cols, window_len=int(width))
    if "atr" in featured.columns:
        atr = _window_matrix(featured, cols=("atr",), window_len=int(width))
        atr = atr[:, 0]
    else:
        atr = np.ones((int(width),), dtype=np.float32)
    scaled = scale_v14_feature_window(
        window,
        feature_cols=cols,
        atr=atr,
        resolution_minutes=minutes,
        zscore_returns=True,
    )
    candle_ids = _id_window_vector(
        featured,
        col=CANDLE_CLASS_COL,
        window_len=int(width),
        max_id=CANDLE_CLASS_CARDINALITY - 1,
    )
    chart_ids = _id_window_vector(
        featured,
        col=CHART_PATTERN_COL,
        window_len=int(width),
        max_id=CHART_PATTERN_CARDINALITY - 1,
    )
    return scaled, candle_ids, chart_ids

def _open_fusion_id_memmap(
    memmap_dir: Path,
    *,
    kind: str,
    resolution: str,
    shape: Tuple[int, int],
) -> np.ndarray:
    dest = Path(memmap_dir)
    dest.mkdir(parents=True, exist_ok=True)
    path = dest / f"windows_{kind}_{resolution}.npy"
    if path.exists():
        path.unlink()
    return np.lib.format.open_memmap(path, mode="w+", dtype=np.int64, shape=shape)

def _zscore_ret1_windows(mat: np.ndarray, ret_index: int) -> np.ndarray:
    """Per-window z-score of the ``ret_1`` column only."""
    ret = mat[:, :, ret_index]
    mu = ret.mean(axis=1, keepdims=True)
    sd = ret.std(axis=1, keepdims=True) + 1e-6
    mat[:, :, ret_index] = (ret - mu) / sd
    return mat

def collect_training_windows_v14(
    featured_by_tf: Mapping[str, pd.DataFrame],
    decision_times: Sequence[pd.Timestamp],
    *,
    window_len: Optional[int] = None,
    window_lens: Optional[Mapping[str, int]] = None,
    memmap_dir: Optional[Path] = None,
) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    """Build v14 (n, window, feat) arrays plus int64 id windows per TF."""
    n = len(decision_times)
    cols = list(fusion_feature_cols_v14())
    lens = resolve_fusion_window_lens(window_lens, window_len)
    decisions = pd.to_datetime(pd.Index(list(decision_times)), utc=True)
    dec_ns = decisions.asi8
    out: Dict[str, np.ndarray] = {}
    candle_out: Dict[str, np.ndarray] = {}
    chart_out: Dict[str, np.ndarray] = {}
    mmap_root = Path(memmap_dir) if memmap_dir is not None else None
    for res in FUSION_INPUT_RESOLUTIONS:
        if res not in featured_by_tf:
            raise KeyError(f"Missing featured frame for {res}")
        width = int(lens[res])
        shape = (int(n), width, len(cols))
        id_shape = (int(n), width)
        if mmap_root is not None:
            mat = _open_fusion_window_memmap(mmap_root, str(res), shape)
            candle_mat = _open_fusion_id_memmap(
                mmap_root, kind="candle", resolution=str(res), shape=id_shape
            )
            chart_mat = _open_fusion_id_memmap(
                mmap_root, kind="chart", resolution=str(res), shape=id_shape
            )
        else:
            mat = np.zeros(shape, dtype=np.float32)
            candle_mat = np.zeros(id_shape, dtype=np.int64)
            chart_mat = np.zeros(id_shape, dtype=np.int64)
        feat = featured_by_tf[res]
        minutes = int(RESOLUTION_MINUTES[str(res).strip().lower()])
        if feat is None or feat.empty or "time" not in feat.columns:
            out[res] = mat
            candle_out[res] = candle_mat
            chart_out[res] = chart_mat
            continue
        missing = [c for c in cols if c not in feat.columns]
        frame = feat
        if missing:
            zeros = pd.DataFrame(0.0, index=feat.index, columns=missing)
            frame = pd.concat([feat, zeros], axis=1)
        close_ns = pd.DatetimeIndex(
            pd.to_datetime(bar_close_time(frame["time"], minutes), utc=True)
        ).asi8
        values = frame.loc[:, cols].to_numpy(dtype=np.float64)
        values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
        values = values.astype(np.float32, copy=False)
        if "atr" in frame.columns:
            atr_col = frame["atr"].to_numpy(dtype=np.float64)
            atr_col = np.nan_to_num(atr_col, nan=1.0, posinf=1.0, neginf=1.0)
        else:
            atr_col = np.ones((len(frame),), dtype=np.float64)
        atr_vals = atr_col.astype(np.float32, copy=False).reshape(-1, 1)
        cdl_series = (
            pd.to_numeric(frame[CANDLE_CLASS_COL], errors="coerce").fillna(0)
            if CANDLE_CLASS_COL in frame.columns
            else pd.Series(0, index=frame.index)
        )
        chp_series = (
            pd.to_numeric(frame[CHART_PATTERN_COL], errors="coerce").fillna(0)
            if CHART_PATTERN_COL in frame.columns
            else pd.Series(0, index=frame.index)
        )
        cdl_vals = np.clip(
            cdl_series.to_numpy(dtype=np.int64), 0, CANDLE_CLASS_CARDINALITY - 1
        ).reshape(-1, 1).astype(np.float32)
        chp_vals = np.clip(
            chp_series.to_numpy(dtype=np.int64), 0, CHART_PATTERN_CARDINALITY - 1
        ).reshape(-1, 1).astype(np.float32)
        order = np.argsort(close_ns, kind="mergesort")
        close_sorted = close_ns[order]
        values = values[order]
        atr_vals = atr_vals[order]
        cdl_vals = cdl_vals[order]
        chp_vals = chp_vals[order]
        end_idx = np.searchsorted(close_sorted, dec_ns, side="right") - 1
        _gather_windows(values, end_idx, width, out=mat)
        atr_buf = np.zeros((int(n), width, 1), dtype=np.float32)
        _gather_windows(atr_vals, end_idx, width, out=atr_buf)
        atr_buf[atr_buf <= 0] = 1.0
        n_samples = int(mat.shape[0])
        for start in range(0, n_samples, _FUSION_WINDOW_CHUNK):
            stop = min(start + _FUSION_WINDOW_CHUNK, n_samples)
            sl = np.array(mat[start:stop], dtype=np.float32, copy=True)
            atr_sl = atr_buf[start:stop, :, 0]
            sl = scale_v14_feature_window(
                sl,
                feature_cols=cols,
                atr=atr_sl,
                resolution_minutes=minutes,
                zscore_returns=True,
            )
            mat[start:stop] = sl
        cdl_buf = np.zeros((int(n), width, 1), dtype=np.float32)
        chp_buf = np.zeros((int(n), width, 1), dtype=np.float32)
        _gather_windows(cdl_vals, end_idx, width, out=cdl_buf)
        _gather_windows(chp_vals, end_idx, width, out=chp_buf)
        candle_mat[:] = np.clip(
            np.rint(cdl_buf[:, :, 0]), 0, CANDLE_CLASS_CARDINALITY - 1
        ).astype(np.int64)
        chart_mat[:] = np.clip(
            np.rint(chp_buf[:, :, 0]), 0, CHART_PATTERN_CARDINALITY - 1
        ).astype(np.int64)
        del values, atr_vals, atr_buf, cdl_vals, chp_vals, cdl_buf, chp_buf
        if mmap_root is not None:
            mat.flush()
            candle_mat.flush()
            chart_mat.flush()
        out[res] = mat
        candle_out[res] = candle_mat
        chart_out[res] = chart_mat
    return out, candle_out, chart_out

def _zscore_windows(mat: np.ndarray) -> np.ndarray:
    """In-place per-window z-score over time. Matches ``zscore_window``."""
    mu = mat.mean(axis=1, keepdims=True)
    sd = mat.std(axis=1, keepdims=True) + 1e-6
    np.subtract(mat, mu, out=mat)
    np.divide(mat, sd, out=mat)
    return mat

def _zscore_windows_chunked(
    mat: np.ndarray,
    *,
    chunk_size: int = _FUSION_WINDOW_CHUNK,
) -> np.ndarray:
    """Z-score in sample chunks so a memmap is not pulled fully into RAM."""
    n = int(mat.shape[0])
    cs = max(int(chunk_size), 1)
    for start in range(0, n, cs):
        stop = min(start + cs, n)
        sl = np.array(mat[start:stop], dtype=np.float32, copy=True)
        _zscore_windows(sl)
        mat[start:stop] = sl
    return mat

def _open_fusion_window_memmap(
    memmap_dir: Path,
    resolution: str,
    shape: Tuple[int, int, int],
) -> np.ndarray:
    """Create a float32 .npy memmap for one TF's training windows."""
    dest = Path(memmap_dir)
    dest.mkdir(parents=True, exist_ok=True)
    path = dest / f"windows_{resolution}.npy"
    if path.exists():
        path.unlink()
    return np.lib.format.open_memmap(path, mode="w+", dtype=np.float32, shape=shape)

def _gather_windows(
    values: np.ndarray,
    end_idx: np.ndarray,
    window_len: int,
    *,
    out: Optional[np.ndarray] = None,
    chunk_size: int = _FUSION_WINDOW_CHUNK,
) -> np.ndarray:
    """Slice ``window_len`` rows ending at each inclusive ``end_idx`` (pad left).

    ``end_idx == -1`` means no closed bar yet and the row stays zeros.
    Fancy-index gathers run in chunks so peak RAM stays near one chunk.
    """
    n_samples = int(end_idx.shape[0])
    n_feat = int(values.shape[1]) if values.size else 0
    width = int(window_len)
    if out is None:
        out = np.zeros((n_samples, width, n_feat), dtype=np.float32)
    elif tuple(out.shape) != (n_samples, width, n_feat):
        raise ValueError(f"out shape {out.shape} != {(n_samples, width, n_feat)}")
    if values.size == 0 or n_feat == 0:
        return out
    n_bars = int(values.shape[0])
    cs = max(int(chunk_size), 1)
    for start in range(0, n_samples, cs):
        stop = min(start + cs, n_samples)
        out[start:stop] = 0
        idx = end_idx[start:stop]
        valid = (idx >= 0) & (idx < n_bars)
        full = valid & (idx >= width - 1)
        if np.any(full):
            ends = idx[full].astype(np.int64, copy=False)
            starts = ends - width + 1
            offsets = starts[:, None] + np.arange(width, dtype=np.int64)[None, :]
            dest = np.flatnonzero(full) + start
            out[dest] = values[offsets]
        for i in np.flatnonzero(valid & ~full):
            end = int(idx[i]) + 1
            sl = values[:end]
            out[start + int(i), width - len(sl) :, :] = sl
    return out

def collect_training_windows(
    featured_by_tf: Mapping[str, pd.DataFrame],
    decision_times: Sequence[pd.Timestamp],
    *,
    window_len: Optional[int] = None,
    window_lens: Optional[Mapping[str, int]] = None,
    zscore: bool = True,
    memmap_dir: Optional[Path] = None,
) -> Dict[str, np.ndarray]:
    """Build (n, window, feat) arrays per TF. Decision times are 5m closes.

    Uses ``searchsorted`` on bar close times so each TF is scanned once.
    Semantics match ``encode_tf_window(..., featured=frame)``.
    Each TF keeps its own ``window_lens[res]``; arrays are not padded to a
    shared max length.

    When ``memmap_dir`` is set, each TF array is a float32 memmap on disk so
    Colab does not hold five full window tensors in RAM.
    """
    n = len(decision_times)
    cols = list(fusion_feature_cols())
    lens = resolve_fusion_window_lens(window_lens, window_len)
    decisions = pd.to_datetime(pd.Index(list(decision_times)), utc=True)
    dec_ns = decisions.asi8
    out: Dict[str, np.ndarray] = {}
    mmap_root = Path(memmap_dir) if memmap_dir is not None else None
    for res in FUSION_INPUT_RESOLUTIONS:
        if res not in featured_by_tf:
            raise KeyError(f"Missing featured frame for {res}")
        width = int(lens[res])
        shape = (int(n), width, len(cols))
        if mmap_root is not None:
            mat = _open_fusion_window_memmap(mmap_root, str(res), shape)
        else:
            mat = np.zeros(shape, dtype=np.float32)
        feat = featured_by_tf[res]
        minutes = int(RESOLUTION_MINUTES[str(res).strip().lower()])
        if feat is None or feat.empty or "time" not in feat.columns:
            out[res] = mat
            continue
        missing = [c for c in cols if c not in feat.columns]
        frame = feat
        if missing:
            zeros = pd.DataFrame(0.0, index=feat.index, columns=missing)
            frame = pd.concat([feat, zeros], axis=1)
        close_ns = pd.DatetimeIndex(
            pd.to_datetime(bar_close_time(frame["time"], minutes), utc=True)
        ).asi8
        values = frame.loc[:, cols].to_numpy(dtype=np.float64)
        values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
        values = values.astype(np.float32, copy=False)
        order = np.argsort(close_ns, kind="mergesort")
        close_sorted = close_ns[order]
        values = values[order]
        end_idx = np.searchsorted(close_sorted, dec_ns, side="right") - 1
        _gather_windows(values, end_idx, width, out=mat)
        del values
        if zscore:
            _zscore_windows_chunked(mat)
        if mmap_root is not None:
            mat.flush()
        out[res] = mat
    return out

# Re-export so tests can assert 15m is not in the fusion input contract.
NATIVE_ONLY_FEATURE_COLS = FEATURE_COLS


## Fusion diagnostics

In [ ]:
"""v15 fusion diagnostics. No Transformer training.

Answers whether chance-level Colab direction skill is a label bug, a
lookahead leak, or a state that does not determine the frozen-theta class.
Walk-forward Transformer, Optuna, SHAP, and ONNX export stay out of this
module.
"""

from __future__ import annotations

from typing import Any, Callable, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

_EPS = 1e-9
_BEAR = 0
_NEUTRAL = 1
_BULL = 2

DIAG_THETA_GRID: Tuple[float, ...] = (
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60,
    0.75,
    0.80,
    1.00,
)
COST_ATR: Tuple[float, ...] = (0.0, 0.02, 0.05, 0.10)
BASELINE_MARGIN = 0.02
KNN_TV_NEAR = 0.10
TRANSFORMER_CHANCE_MAX = 0.40
# Validation balanced accuracy from the v14 Colab run this gate replaces.
PUBLISHED_TRANSFORMER_BALANCED_ACC: Dict[str, float] = {
    "h30m": 0.337,
    "h1h": 0.365,
}
MOMENTUM_COL = "rsi_14"
RETURN_COL = "ret_1"
FeatureFn = Callable[[pd.DataFrame], pd.DataFrame]

def _jsonify(value: Any) -> Any:
    """Cast numpy scalars so the report dumps as JSON."""
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        val = float(value)
        return val if np.isfinite(val) else None
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, np.ndarray):
        return [_jsonify(v) for v in value.tolist()]
    if isinstance(value, dict):
        return {str(k): _jsonify(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonify(v) for v in value]
    return value

def _causal_atr(df: pd.DataFrame) -> np.ndarray:
    """Causal ATR at t. Uses an ``atr`` column when present."""
    n = len(df)
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]]) if n else close
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    if n == 0:
        return np.zeros(0, dtype=np.float64)
    return (
        pd.Series(tr).rolling(14, min_periods=1).mean().to_numpy(dtype=np.float64)
    )

def _pair_key(sl_mult: float, tp_mult: float) -> str:
    return f"sl{float(sl_mult):.2f}_tp{float(tp_mult):.2f}"

def _theta_key(theta: float) -> str:
    return f"{float(theta):.2f}"

def purged_index_slices(
    n: int,
    *,
    train_frac: float = 0.70,
    val_frac: float = 0.15,
    embargo_bars: int = FUSION_EMBARGO_BARS,
) -> Dict[str, slice]:
    """Match ``split_purged_windows`` in ``mtf_fusion_research`` (no torch)."""
    train_end = int(n * float(train_frac))
    val_end = train_end + int(n * float(val_frac))
    embargo = int(embargo_bars)
    return {
        "train": slice(0, train_end),
        "val": slice(train_end + embargo, val_end),
        "test": slice(val_end + embargo, n),
    }

def walk_forward_slices(
    n: int,
    *,
    folds: int = 3,
    embargo: int = FUSION_EMBARGO_BARS,
) -> List[Tuple[slice, slice]]:
    """Match ``walk_forward_slices`` in ``mtf_fusion_research`` (no torch)."""
    folds_n = max(int(folds), 1)
    out: List[Tuple[slice, slice]] = []
    for i in range(folds_n):
        train_end = max(2, int(n * (0.50 + 0.10 * i)))
        val_end = min(n, train_end + max(int(n * 0.12), 8))
        gap = min(max(int(embargo), 0), max(0, val_end - train_end - 1))
        val_start = train_end + gap
        if val_start >= val_end:
            continue
        out.append((slice(0, train_end), slice(val_start, val_end)))
    return out

def balanced_accuracy(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_classes: int = LABEL_V2_DIRECTION_CARDINALITY,
) -> float:
    """Mean per-class recall. Empty classes are skipped."""
    scores: List[float] = []
    yt = np.asarray(y_true, dtype=np.int64)
    yp = np.asarray(y_pred, dtype=np.int64)
    for c in range(int(n_classes)):
        mask = yt == c
        if not np.any(mask):
            continue
        scores.append(float(np.mean(yp[mask] == c)))
    if not scores:
        return 0.0
    return float(np.mean(scores))

def raw_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Share of matching labels on rows with ``y_true >= 0``."""
    yt = np.asarray(y_true, dtype=np.int64)
    yp = np.asarray(y_pred, dtype=np.int64)
    valid = yt >= 0
    if not np.any(valid):
        return 0.0
    return float(np.mean(yp[valid] == yt[valid]))

def nonoverlap_indices(indices: np.ndarray, min_gap: int) -> np.ndarray:
    """Keep increasing indices at least ``min_gap`` bars apart."""
    ordered = np.asarray(indices, dtype=np.int64)
    if ordered.size == 0:
        return ordered
    ordered = np.sort(ordered)
    keep: List[int] = []
    last = None
    gap = max(int(min_gap), 1)
    for idx in ordered:
        if last is None or int(idx) - int(last) >= gap:
            keep.append(int(idx))
            last = int(idx)
    return np.asarray(keep, dtype=np.int64)

def horizon_bars_map() -> Dict[str, int]:
    """Horizon key to 5m bar count."""
    return {str(k): int(b) for k, b in zip(LABEL_V2_HORIZON_KEYS, LABEL_V2_HORIZON_BARS_5M)}

def recompute_horizon_return(
    close: np.ndarray,
    atr: np.ndarray,
    index: int,
    bars: int,
) -> float:
    """``(Close[t+k] - Close[t]) / ATR[t]``. Path starts at t+1; uses Close[t+k]."""
    i = int(index)
    k = int(bars)
    if i < 0 or k <= 0 or i + k >= len(close):
        return float("nan")
    denom = max(float(atr[i]), _EPS)
    return float((float(close[i + k]) - float(close[i])) / denom)

def audit_label_index(
    labeled: pd.DataFrame,
    *,
    sample_indices: Sequence[int],
    thetas: Optional[Mapping[str, float]] = None,
    atol: float = 1e-6,
) -> Dict[str, Any]:
    """Recompute stored ``{h}_ret`` and frozen-theta class at sample rows.

    Args:
        labeled: Output of ``compute_fusion_path_targets``.
        sample_indices: Row positions on the 5m clock.
        thetas: Per-horizon dead zone. Defaults to frozen theta.
        atol: Absolute tolerance on ATR-normalized return.

    Returns:
        JSON-ready audit with ``ok`` false on any mismatch.
    """
    theta_map = dict(LABEL_V2_THETA_FROZEN)
    if thetas is not None:
        theta_map.update({str(k): float(v) for k, v in thetas.items()})
    close = labeled["close"].to_numpy(dtype=np.float64)
    atr = _causal_atr(labeled)
    bars = horizon_bars_map()
    mismatches: List[Dict[str, Any]] = []
    checked = 0
    for idx in sample_indices:
        i = int(idx)
        if i < 0 or i >= len(labeled):
            continue
        for key, k in bars.items():
            stored = float(labeled.iloc[i][f"{key}_ret"])
            recomputed = recompute_horizon_return(close, atr, i, k)
            if not np.isfinite(stored) or not np.isfinite(recomputed):
                if np.isfinite(stored) != np.isfinite(recomputed):
                    mismatches.append(
                        {
                            "index": i,
                            "horizon": key,
                            "reason": "finite_mismatch",
                            "stored": stored,
                            "recomputed": recomputed,
                        }
                    )
                continue
            checked += 1
            if abs(stored - recomputed) > float(atol):
                mismatches.append(
                    {
                        "index": i,
                        "horizon": key,
                        "reason": "return_mismatch",
                        "stored": stored,
                        "recomputed": recomputed,
                    }
                )
                continue
            stored_dir = int(
                direction_from_return_array(np.array([stored]), theta_map[key])[0]
            )
            recompute_dir = int(
                direction_from_return_array(np.array([recomputed]), theta_map[key])[0]
            )
            if stored_dir != recompute_dir:
                mismatches.append(
                    {
                        "index": i,
                        "horizon": key,
                        "reason": "class_mismatch",
                        "stored_dir": stored_dir,
                        "recomputed_dir": recompute_dir,
                    }
                )
    return _jsonify(
        {
            "ok": len(mismatches) == 0,
            "checked": checked,
            "n_samples": int(len(sample_indices)),
            "mismatches": mismatches[:50],
            "n_mismatches": len(mismatches),
        }
    )

def class_intervals_disjoint(
    ret: np.ndarray,
    dirs: np.ndarray,
    *,
    theta: float,
    atol: float = 1e-9,
) -> Dict[str, Any]:
    """True when BEAR / NEUTRAL / BULL are disjoint cuts of ``ret``.

    Overlap is an indexing bug, not evidence the classes are noisy.
    """
    r = np.asarray(ret, dtype=np.float64)
    d = np.asarray(dirs, dtype=np.int64)
    th = abs(float(theta))
    by_class: Dict[str, Dict[str, Any]] = {}
    extrema: Dict[int, Tuple[float, float]] = {}
    for cid, name in LABEL_V2_DIRECTION_NAMES.items():
        vals = r[(d == int(cid)) & np.isfinite(r)]
        if vals.size == 0:
            by_class[name] = {"n": 0, "min": None, "max": None}
            continue
        lo = float(np.min(vals))
        hi = float(np.max(vals))
        extrema[int(cid)] = (lo, hi)
        by_class[name] = {"n": int(vals.size), "min": lo, "max": hi}
    flags: List[str] = []
    bear = extrema.get(_BEAR)
    neu = extrema.get(_NEUTRAL)
    bull = extrema.get(_BULL)
    if bear is not None and bear[1] > -th + float(atol):
        flags.append("bear_crosses_theta")
    if bull is not None and bull[0] < th - float(atol):
        flags.append("bull_crosses_theta")
    if neu is not None and (neu[0] < -th - float(atol) or neu[1] > th + float(atol)):
        flags.append("neutral_outside_deadzone")
    if bear is not None and neu is not None and bear[1] >= neu[0] - float(atol):
        flags.append("bear_overlaps_neutral")
    if neu is not None and bull is not None and neu[1] >= bull[0] - float(atol):
        flags.append("neutral_overlaps_bull")
    if bear is not None and bull is not None and bear[1] >= bull[0] - float(atol):
        flags.append("bear_overlaps_bull")
    return _jsonify(
        {
            "ok": len(flags) == 0,
            "theta": th,
            "classes": by_class,
            "flags": flags,
        }
    )

def audit_class_intervals(
    labeled: pd.DataFrame,
    *,
    thetas: Optional[Mapping[str, float]] = None,
) -> Dict[str, Any]:
    """Run :func:`class_intervals_disjoint` on each frozen-theta horizon."""
    theta_map = dict(LABEL_V2_THETA_FROZEN)
    if thetas is not None:
        theta_map.update({str(k): float(v) for k, v in thetas.items()})
    per: Dict[str, Any] = {}
    ok = True
    for key in LABEL_V2_HORIZON_KEYS:
        ret = labeled[f"{key}_ret"].to_numpy(dtype=np.float64)
        dirs = direction_from_return_array(ret, float(theta_map[str(key)]))
        row = class_intervals_disjoint(ret, dirs, theta=float(theta_map[str(key)]))
        per[str(key)] = row
        ok = ok and bool(row.get("ok"))
    return {"ok": ok, "horizons": per}

def extra_theta_mix(labeled: pd.DataFrame) -> Dict[str, Any]:
    """Class-mix table at DIAG_THETA_GRID. Does not train a model."""
    report = summarize_label_v2(
        labeled, thetas=DIAG_THETA_GRID, already_labeled=True
    )
    mix: Dict[str, Any] = {}
    for key in LABEL_V2_HORIZON_KEYS:
        by_theta = (report.get("horizons") or {}).get(key, {}).get("by_theta") or {}
        mix[str(key)] = {}
        for tkey, row in by_theta.items():
            classes = row.get("classes") or {}
            mix[str(key)][str(tkey)] = {
                name: {
                    "rate": (classes.get(name) or {}).get("rate"),
                    "e_r": (classes.get(name) or {}).get("e_r"),
                }
                for name in LABEL_V2_DIRECTION_NAMES.values()
            }
    return {
        "overall": report.get("overall"),
        "thetas": list(DIAG_THETA_GRID),
        "mix": mix,
        "table": report,
    }

def expectancy_minus_cost(
    label_report: Mapping[str, Any],
    *,
    costs: Sequence[float] = COST_ATR,
    thetas: Optional[Mapping[str, float]] = None,
) -> Dict[str, Any]:
    """Live-bracket ``expectancy_atr`` minus stated cost in ATR units.

    Does not block later diagnostics. Negative after-cost expectancy means
    classifying this label cannot be a trading system at that cost.
    """
    theta_map = dict(LABEL_V2_THETA_FROZEN)
    if thetas is not None:
        theta_map.update({str(k): float(v) for k, v in thetas.items()})
    out: Dict[str, Any] = {}
    for key in LABEL_V2_HORIZON_KEYS:
        sl_m, tp_m = LABEL_V2_DURATION_ATR_MULT[str(key)]
        live_key = _pair_key(sl_m, tp_m)
        tkey = _theta_key(float(theta_map[str(key)]))
        hrow = (label_report.get("horizons") or {}).get(key) or {}
        tp_sl = (
            ((hrow.get("by_theta") or {}).get(tkey) or {}).get("tp_sl") or {}
        )
        exp = (tp_sl.get(live_key) or {}).get("expectancy_atr")
        after = {}
        for cost in costs:
            if exp is None:
                after[str(cost)] = None
            else:
                after[str(cost)] = float(exp) - float(cost)
        out[str(key)] = {
            "theta": float(theta_map[str(key)]),
            "live_bracket": {"sl": float(sl_m), "tp": float(tp_m)},
            "expectancy_atr": exp,
            "after_cost": after,
        }
    return _jsonify(out)

def compare_row_values(
    full_row: Mapping[str, Any],
    truncated_row: Mapping[str, Any],
    cols: Sequence[str],
    *,
    atol: float = 1e-5,
) -> List[str]:
    """Column names whose truncated last row disagrees with the full series."""
    mismatches: List[str] = []
    for col in cols:
        left = full_row.get(col)
        right = truncated_row.get(col)
        if left is None and right is None:
            continue
        try:
            lv = float(left) if left is not None else float("nan")
            rv = float(right) if right is not None else float("nan")
        except (TypeError, ValueError):
            if left != right:
                mismatches.append(str(col))
            continue
        if np.isfinite(lv) and np.isfinite(rv):
            if abs(lv - rv) > float(atol):
                mismatches.append(str(col))
            continue
        if np.isfinite(lv) != np.isfinite(rv):
            mismatches.append(str(col))
    return mismatches

def default_native_feature_fn(frame: pd.DataFrame) -> pd.DataFrame:
    """Causal 5m native features for truncation audits."""
    return add_native_tf_features(frame, resolution="5m")

def audit_feature_truncation(
    frame: pd.DataFrame,
    *,
    sample_indices: Sequence[int],
    feature_fn: FeatureFn,
    cols: Sequence[str],
    atol: float = 1e-5,
    full_featured: Optional[pd.DataFrame] = None,
) -> Dict[str, Any]:
    """Rebuild features on ``df[: t]`` and compare the last row to the full series.

    Args:
        frame: Time-sorted OHLCV (or a synthetic frame with the audited columns).
        sample_indices: Decision rows. Each uses the prefix through that row.
        feature_fn: Maps a prefix frame to a featured frame.
        cols: Columns that must match.
        atol: Absolute tolerance.
        full_featured: Precomputed ``feature_fn(frame)``. Computed when omitted.

    Returns:
        ``ok`` is false when any sampled last row mismatches.
    """
    full_feat = full_featured if full_featured is not None else feature_fn(frame)
    mismatches: List[Dict[str, Any]] = []
    n_ok = 0
    for idx in sample_indices:
        i = int(idx)
        if i < 0 or i >= len(frame):
            continue
        prefix = frame.iloc[: i + 1].copy()
        trunc_feat = feature_fn(prefix)
        if trunc_feat.empty or full_feat.empty:
            mismatches.append({"index": i, "cols": ["empty_features"]})
            continue
        full_row = full_feat.iloc[min(i, len(full_feat) - 1)]
        trunc_row = trunc_feat.iloc[-1]
        bad = compare_row_values(full_row, trunc_row, cols, atol=atol)
        if bad:
            mismatches.append({"index": i, "cols": bad[:20]})
        else:
            n_ok += 1
    return _jsonify(
        {
            "ok": len(mismatches) == 0,
            "n_ok": n_ok,
            "n_checked": int(len(sample_indices)),
            "mismatches": mismatches[:20],
            "n_mismatches": len(mismatches),
        }
    )

def audit_10m_truncation(
    df5m: pd.DataFrame,
    *,
    sample_indices: Sequence[int],
) -> Dict[str, Any]:
    """Last complete 10m bar on a 5m prefix must match the full 10m series."""
    full_10 = build_10m_ohlcv_from_5m(df5m)
    mismatches = 0
    checked = 0
    for idx in sample_indices:
        i = int(idx)
        if i < 1 or i >= len(df5m):
            continue
        prefix = df5m.iloc[: i + 1]
        trunc_10 = build_10m_ohlcv_from_5m(prefix)
        if trunc_10.empty:
            continue
        checked += 1
        last_time = trunc_10["time"].iloc[-1]
        full_hit = full_10.loc[full_10["time"] == last_time]
        if full_hit.empty:
            mismatches += 1
            continue
        for col in ("open", "high", "low", "close", "volume"):
            if col not in trunc_10.columns or col not in full_hit.columns:
                continue
            if not np.isclose(
                float(trunc_10[col].iloc[-1]),
                float(full_hit[col].iloc[0]),
                atol=1e-8,
                equal_nan=True,
            ):
                mismatches += 1
                break
    return _jsonify(
        {
            "ok": mismatches == 0,
            "checked": checked,
            "mismatches": mismatches,
        }
    )

def audit_asof_no_lookahead(
    frames: Mapping[str, pd.DataFrame],
    decision_times: Sequence[pd.Timestamp],
) -> Dict[str, Any]:
    """``assert_no_lookahead`` on each provided TF at sampled decision times."""
    failures: List[str] = []
    checked = 0
    for t in decision_times:
        ts = pd.Timestamp(t)
        for res, df in frames.items():
            minutes = int(RESOLUTION_MINUTES.get(str(res), 0) or 0)
            if minutes <= 0 or df is None or df.empty:
                continue
            checked += 1
            try:
                assert_no_lookahead(df, ts, resolution_minutes=minutes)
            except AssertionError as exc:
                failures.append(f"{res}@{ts.isoformat()}: {exc}")
    return _jsonify(
        {
            "ok": len(failures) == 0,
            "checked": checked,
            "failures": failures[:20],
        }
    )

def channel_scale_is_per_window(
    window: np.ndarray,
    longer: np.ndarray,
    *,
    feature_cols: Sequence[str],
    resolution_minutes: int = 5,
) -> bool:
    """v14 ``ret_1`` z-score uses the window, not a longer series."""
    cols = list(feature_cols)
    if "ret_1" not in cols:
        return True
    idx = cols.index("ret_1")
    scaled_win = scale_v14_feature_window(
        window,
        feature_cols=cols,
        resolution_minutes=int(resolution_minutes),
        zscore_returns=True,
    )
    scaled_long = scale_v14_feature_window(
        longer,
        feature_cols=cols,
        resolution_minutes=int(resolution_minutes),
        zscore_returns=True,
    )
    t = int(window.shape[0])
    same_as_prefix = np.allclose(
        scaled_win[:, idx], scaled_long[:t, idx], atol=1e-5
    )
    return not bool(same_as_prefix)

def last_bar_matrix(
    featured: pd.DataFrame,
    *,
    cols: Optional[Sequence[str]] = None,
    fill_nonfinite: bool = True,
) -> Tuple[np.ndarray, Tuple[str, ...]]:
    """Last-bar v14 continuous features. No sequence flatten."""
    names = tuple(cols) if cols is not None else fusion_feature_cols_v14()
    frame = featured
    missing = [c for c in names if c not in frame.columns]
    if missing:
        zeros = pd.DataFrame(0.0, index=frame.index, columns=missing)
        frame = pd.concat([frame, zeros], axis=1)
    values = frame.loc[:, list(names)].to_numpy(dtype=np.float64)
    if fill_nonfinite:
        values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
    return values.astype(np.float32, copy=False), names

def direction_matrix(
    labeled: pd.DataFrame,
    *,
    thetas: Optional[Mapping[str, float]] = None,
) -> np.ndarray:
    """``(n, H)`` frozen-theta classes. Non-finite return is -1."""
    theta_map = dict(LABEL_V2_THETA_FROZEN)
    if thetas is not None:
        theta_map.update({str(k): float(v) for k, v in thetas.items()})
    n = len(labeled)
    y = np.full((n, len(LABEL_V2_HORIZON_KEYS)), -1, dtype=np.int64)
    for j, key in enumerate(LABEL_V2_HORIZON_KEYS):
        col = f"{key}_ret"
        if col not in labeled.columns:
            continue
        y[:, j] = direction_from_return_array(
            labeled[col].to_numpy(dtype=np.float64),
            float(theta_map[str(key)]),
        )
    return y

def embargoed_neighbor_probs(
    features: np.ndarray,
    labels: np.ndarray,
    train_idx: np.ndarray,
    query_idx: np.ndarray,
    *,
    horizon_bars: int,
    embargo_bars: int = FUSION_EMBARGO_BARS,
    n_neighbors: int = 32,
    train_cap: int = 8000,
    query_cap: int = 2000,
    seed: int = 42,
) -> Dict[str, Any]:
    """``P(class | neighbors)`` with time embargo and non-overlapping paths.

    Neighbors whose index is within ``max(embargo, horizon_bars)`` of the query
    are dropped. Train-only fit. Query rows should be validation indices.
    """
    rng = np.random.default_rng(int(seed))
    x = np.nan_to_num(np.asarray(features, dtype=np.float64), nan=0.0)
    y = np.asarray(labels, dtype=np.int64)
    train = np.asarray(train_idx, dtype=np.int64)
    query = np.asarray(query_idx, dtype=np.int64)
    if train.size > int(train_cap):
        pick = np.sort(rng.choice(train.size, size=int(train_cap), replace=False))
        train = train[pick]
    if query.size > int(query_cap):
        pick = np.sort(rng.choice(query.size, size=int(query_cap), replace=False))
        query = query[pick]
    valid_train = train[y[train] >= 0]
    if valid_train.size < 8 or query.size == 0:
        return {
            "ok": False,
            "reason": "insufficient_rows",
            "near_base_rate": True,
            "mean_probs": None,
            "base_rate": None,
            "total_variation": None,
        }
    n_ask = min(max(int(n_neighbors) * 8, int(n_neighbors)), int(valid_train.size))
    nn = NearestNeighbors(n_neighbors=n_ask, algorithm="auto")
    nn.fit(x[valid_train])
    _, ind = nn.kneighbors(x[query])
    min_gap = max(int(embargo_bars), int(horizon_bars), 1)
    k_keep = int(n_neighbors)
    counts = np.zeros(LABEL_V2_DIRECTION_CARDINALITY, dtype=np.float64)
    n_used = 0
    skipped = 0
    for q_i, neigh in enumerate(ind):
        q_pos = int(query[q_i])
        if y[q_pos] < 0:
            continue
        kept: List[int] = []
        for local in neigh:
            t_pos = int(valid_train[int(local)])
            if abs(t_pos - q_pos) < min_gap:
                continue
            kept.append(t_pos)
            if len(kept) >= k_keep:
                break
        if len(kept) < 8:
            skipped += 1
            continue
        n_used += 1
        for t_pos in kept:
            cid = int(y[t_pos])
            if 0 <= cid < LABEL_V2_DIRECTION_CARDINALITY:
                counts[cid] += 1.0
    total = float(counts.sum())
    if total <= 0.0:
        mean_p = np.full(LABEL_V2_DIRECTION_CARDINALITY, 1.0 / 3.0)
    else:
        mean_p = counts / total
    train_y = y[valid_train]
    base = np.array(
        [
            float(np.mean(train_y == c)) if train_y.size else 1.0 / 3.0
            for c in range(LABEL_V2_DIRECTION_CARDINALITY)
        ],
        dtype=np.float64,
    )
    tv = 0.5 * float(np.abs(mean_p - base).sum())
    names = {
        LABEL_V2_DIRECTION_NAMES[c]: float(mean_p[c])
        for c in range(LABEL_V2_DIRECTION_CARDINALITY)
    }
    base_names = {
        LABEL_V2_DIRECTION_NAMES[c]: float(base[c])
        for c in range(LABEL_V2_DIRECTION_CARDINALITY)
    }
    return _jsonify(
        {
            "ok": True,
            "n_queries": n_used,
            "n_skipped": skipped,
            "k": k_keep,
            "min_gap": min_gap,
            "mean_probs": names,
            "base_rate": base_names,
            "total_variation": tv,
            "near_base_rate": tv < float(KNN_TV_NEAR),
        }
    )

def _sign_class(values: np.ndarray) -> np.ndarray:
    out = np.full(values.shape, _NEUTRAL, dtype=np.int64)
    finite = np.isfinite(values)
    out[finite & (values > 0.0)] = _BULL
    out[finite & (values < 0.0)] = _BEAR
    return out

def naive_baselines(
    y: np.ndarray,
    features: np.ndarray,
    feature_names: Sequence[str],
    eval_idx: np.ndarray,
    *,
    horizon_bars: int,
) -> Dict[str, Any]:
    """Always-class, last-return sign, and RSI momentum on non-overlapping rows."""
    names = [str(n) for n in feature_names]
    x = np.asarray(features, dtype=np.float64)
    yt_all = np.asarray(y, dtype=np.int64)
    kept = nonoverlap_indices(np.asarray(eval_idx, dtype=np.int64), int(horizon_bars))
    kept = kept[(kept >= 0) & (kept < len(yt_all)) & (yt_all[kept] >= 0)]
    if kept.size == 0:
        return {"ok": False, "reason": "empty_eval", "best_balanced_acc": 0.0}
    yt = yt_all[kept]
    preds: Dict[str, np.ndarray] = {
        "always_bear": np.full(kept.shape, _BEAR, dtype=np.int64),
        "always_neutral": np.full(kept.shape, _NEUTRAL, dtype=np.int64),
        "always_bull": np.full(kept.shape, _BULL, dtype=np.int64),
    }
    if RETURN_COL in names:
        preds["last_return_sign"] = _sign_class(x[kept, names.index(RETURN_COL)])
    if MOMENTUM_COL in names:
        preds["rsi_momentum"] = _sign_class(x[kept, names.index(MOMENTUM_COL)] - 50.0)
    scores: Dict[str, Any] = {}
    best = -1.0
    best_name = ""
    for name, pred in preds.items():
        bal = balanced_accuracy(yt, pred)
        raw = raw_accuracy(yt, pred)
        scores[name] = {"balanced_acc": bal, "raw_acc": raw, "n": int(kept.size)}
        if bal > best:
            best = bal
            best_name = name
    return _jsonify(
        {
            "ok": True,
            "n": int(kept.size),
            "horizon_bars": int(horizon_bars),
            "baselines": scores,
            "best_name": best_name,
            "best_balanced_acc": best,
        }
    )

def _fit_predict(
    model: Any,
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_eval: np.ndarray,
) -> np.ndarray:
    model.fit(x_train, y_train)
    return np.asarray(model.predict(x_eval), dtype=np.int64)

def simple_model_scores(
    features: np.ndarray,
    labels: np.ndarray,
    *,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    horizon_bars: int,
    n_dev: int,
    seed: int = 42,
    folds: int = 3,
    embargo_bars: int = FUSION_EMBARGO_BARS,
) -> Dict[str, Any]:
    """Last-bar logistic and histogram GBM. Test indices are never used."""
    x = np.nan_to_num(np.asarray(features, dtype=np.float64), nan=0.0)
    y = np.asarray(labels, dtype=np.int64)
    models = {
        "logistic": LogisticRegression(
            max_iter=300,
            solver="lbfgs",
            random_state=int(seed),
        ),
        "hist_gbm": HistGradientBoostingClassifier(
            max_depth=3,
            max_iter=40,
            learning_rate=0.1,
            random_state=int(seed),
        ),
    }
    fold_rows: Dict[str, List[float]] = {name: [] for name in models}
    for tr_sl, va_sl in walk_forward_slices(
        int(n_dev), folds=int(folds), embargo=int(embargo_bars)
    ):
        tr = np.arange(tr_sl.start or 0, tr_sl.stop)
        va = np.arange(va_sl.start or 0, va_sl.stop)
        tr = tr[y[tr] >= 0]
        va = nonoverlap_indices(va[y[va] >= 0], int(horizon_bars))
        if tr.size < 16 or va.size < 4:
            continue
        if len(np.unique(y[tr])) < 2:
            continue
        for name, proto in models.items():
            pred = _fit_predict(proto, x[tr], y[tr], x[va])
            fold_rows[name].append(balanced_accuracy(y[va], pred))
    val_kept = nonoverlap_indices(
        np.asarray(val_idx, dtype=np.int64)[y[val_idx] >= 0],
        int(horizon_bars),
    )
    train_kept = np.asarray(train_idx, dtype=np.int64)
    train_kept = train_kept[y[train_kept] >= 0]
    val_scores: Dict[str, Any] = {}
    best_val = -1.0
    best_name = ""
    if train_kept.size >= 16 and val_kept.size >= 4 and len(np.unique(y[train_kept])) >= 2:
        for name, proto in models.items():
            pred = _fit_predict(proto, x[train_kept], y[train_kept], x[val_kept])
            bal = balanced_accuracy(y[val_kept], pred)
            val_scores[name] = {
                "balanced_acc": bal,
                "raw_acc": raw_accuracy(y[val_kept], pred),
                "n": int(val_kept.size),
                "walk_forward_mean": (
                    float(np.mean(fold_rows[name])) if fold_rows[name] else None
                ),
            }
            if bal > best_val:
                best_val = bal
                best_name = name
    return _jsonify(
        {
            "ok": bool(val_scores),
            "val": val_scores,
            "best_name": best_name,
            "best_val_balanced_acc": best_val if val_scores else 0.0,
            "test_unused": True,
        }
    )

def decide_branch(report: Mapping[str, Any]) -> str:
    """Single next-action token. No Transformer retrain is implied except later_."""
    d1 = dict(report.get("d1_index_audit") or {})
    if d1 and not d1.get("ok", True):
        return "fail_index_audit"
    d2 = dict(report.get("d2_intervals") or {})
    if d2 and not d2.get("ok", True):
        return "fail_class_intervals"
    d9 = dict(report.get("d9_truncation") or {})
    if d9 and not d9.get("ok", True):
        return "fix_leakage"
    d5 = dict(report.get("d5_knn") or {})
    d6 = dict(report.get("d6_baselines") or {})
    d7 = dict(report.get("d7_simple_models") or {})
    if not d5 or not d6 or not d7:
        return "review"
    near = True
    for key in LABEL_V2_HORIZON_KEYS:
        row = (d5.get("horizons") or {}).get(key) or {}
        if not bool(row.get("near_base_rate", True)):
            near = False
    d6_best = max(
        float(((d6.get("horizons") or {}).get(k) or {}).get("best_balanced_acc") or 0.0)
        for k in LABEL_V2_HORIZON_KEYS
    )
    d7_best = max(
        float(
            ((d7.get("horizons") or {}).get(k) or {}).get("best_val_balanced_acc") or 0.0
        )
        for k in LABEL_V2_HORIZON_KEYS
    )
    lift = d7_best - d6_best
    transformer_max = max(float(v) for v in PUBLISHED_TRANSFORMER_BALANCED_ACC.values())
    if near and lift < float(BASELINE_MARGIN):
        return "stop_target_or_features"
    if lift >= float(BASELINE_MARGIN) and transformer_max < float(TRANSFORMER_CHANCE_MAX):
        return "later_direction_only"
    return "review"

def _spread_indices(n: int, k: int, *, lo: int = 0, hi: Optional[int] = None) -> np.ndarray:
    end = int(n if hi is None else hi)
    start = max(int(lo), 0)
    if end <= start:
        return np.zeros(0, dtype=np.int64)
    count = min(int(k), end - start)
    if count <= 0:
        return np.zeros(0, dtype=np.int64)
    return np.linspace(start, end - 1, num=count, dtype=np.int64)

def run_fusion_diagnostics(
    df5m: pd.DataFrame,
    *,
    frames: Optional[Mapping[str, pd.DataFrame]] = None,
    seed: int = 42,
    n_index_samples: int = 100,
    n_truncation_samples: int = 20,
    knn_train_cap: int = 8000,
    knn_val_cap: int = 2000,
    knn_k: int = 32,
    fit_simple_models: bool = True,
    feature_fn: Optional[FeatureFn] = None,
    featured: Optional[pd.DataFrame] = None,
    score_test: bool = False,
    ablate_groups: bool = False,
    transformer_test: Optional[Mapping[str, Any]] = None,
    decision_times: Optional[np.ndarray] = None,
    transformer_logits: Optional[Mapping[str, np.ndarray]] = None,
    transformer_y: Optional[Mapping[str, np.ndarray]] = None,
    transformer_window_logits: Optional[np.ndarray] = None,
    transformer_window_times: Optional[np.ndarray] = None,
    class_weight_sensitivity: bool = False,
    score_path_regression: bool = False,
) -> Dict[str, Any]:
    """Run D1–D7 / D9 / expectancy. Never fits ``MtfFusionTransformer``.

    Args:
        df5m: 5m OHLCV with ``time`` / OHLC (optional ``atr``).
        frames: Optional native TF frames for as-of lookahead checks.
        seed: RNG seed for neighbor subsamples and sklearn models.
        n_index_samples: D1 rows drawn from the train portion.
        n_truncation_samples: D9 prefix recomputes.
        knn_train_cap: Train-neighbor subsample size.
        knn_val_cap: Validation query subsample size.
        knn_k: Neighbors kept after embargo.
        fit_simple_models: When false, skip D5–D7 (tests / leakage abort).
        feature_fn: Override for D9. Defaults to native 5m features.
        featured: Precomputed 5m features aligned to ``df5m``. Skips a second
            full ``add_native_tf_features`` when provided.
        score_test: Run the last-bar direction-predictability bake-off.
        ablate_groups: Val-only knockout / add-one-in / vol+time (needs score_test).
        transformer_test: Optional v15 JSON metrics (not used as a stop token).
        decision_times: Stride-4 window timestamps for an aligned test row.
        transformer_logits: Optional per-horizon (n, C) logits on stride rows.
        transformer_y: Optional per-horizon labels aligned to those logits.
        transformer_window_logits: Optional (n_windows, H, C) checkpoint logits.
        transformer_window_times: Window timestamps aligned to those logits.
        class_weight_sensitivity: Extra val fit with inverse-frequency weights.
        score_path_regression: Last-bar signed path-edge regression bake-off.

    Returns:
        JSON-ready dict with a ``branch`` token.
    """
    labeled_full = compute_fusion_path_targets(df5m)
    labeled = trim_label_v2_tail(labeled_full)
    n = len(labeled)
    slices = purged_index_slices(n)
    train_sl = slices["train"]
    train_hi = train_sl.stop or 0
    d1_idx = _spread_indices(
        n, int(n_index_samples), lo=0, hi=max(train_hi - 24, 1)
    )
    report: Dict[str, Any] = {
        "n_labeled": n,
        "split": {
            "train": [train_sl.start or 0, train_sl.stop],
            "val": [slices["val"].start, slices["val"].stop],
            "test": [slices["test"].start, slices["test"].stop],
            "test_unused": not bool(score_test),
        },
        "d1_index_audit": audit_label_index(labeled_full, sample_indices=d1_idx),
        "d2_intervals": audit_class_intervals(labeled),
    }
    if not report["d1_index_audit"]["ok"] or not report["d2_intervals"]["ok"]:
        report["branch"] = decide_branch(report)
        report["notes"] = "Label index or class-interval audit failed. Stop."
        return _jsonify(report)
    theta_pack = extra_theta_mix(labeled)
    report["theta_mix"] = {
        "overall": theta_pack.get("overall"),
        "thetas": theta_pack.get("thetas"),
        "mix": theta_pack.get("mix"),
    }
    report["expectancy_minus_cost"] = expectancy_minus_cost(theta_pack["table"])
    fn = feature_fn if feature_fn is not None else default_native_feature_fn
    trunc_lo = max(min(train_hi // 4, max(train_hi - 1, 0)), 0)
    trunc_idx = _spread_indices(
        n, int(n_truncation_samples), lo=trunc_lo, hi=max(train_hi, 1)
    )
    feat_cols = list(fusion_feature_cols_v14())
    if featured is not None and feature_fn is None:
        feat_frame = featured
        d9_native = {
            "ok": True,
            "skipped": "precomputed_featured",
            "n_checked": 0,
        }
    else:
        feat_frame = featured if featured is not None else fn(labeled)
        d9_native = audit_feature_truncation(
            labeled if featured is None else feat_frame,
            sample_indices=trunc_idx,
            feature_fn=fn,
            cols=feat_cols if featured is None else list(feat_frame.columns),
            full_featured=feat_frame,
        )
    d9_10m = audit_10m_truncation(labeled, sample_indices=trunc_idx)
    asof_frames: Dict[str, pd.DataFrame] = {"5m": labeled}
    asof_frames["10m"] = build_10m_ohlcv_from_5m(labeled)
    if frames:
        for res in FUSION_INPUT_RESOLUTIONS:
            if res in frames and frames[res] is not None:
                asof_frames[str(res)] = frames[res]
    times = pd.to_datetime(labeled["time"], utc=True)
    asof_times = [times.iloc[int(i)] for i in trunc_idx if 0 <= int(i) < n]
    d9_asof = audit_asof_no_lookahead(asof_frames, asof_times)
    dummy_win = np.linspace(-1.0, 1.0, 16, dtype=np.float32).reshape(16, 1)
    dummy_win = np.repeat(dummy_win, max(len(feat_cols), 1), axis=1)
    dummy_long = np.vstack([dummy_win, dummy_win * 2.0])
    scale_ok = channel_scale_is_per_window(
        dummy_win, dummy_long, feature_cols=feat_cols[: dummy_win.shape[1]]
    )
    report["d9_truncation"] = {
        "ok": bool(d9_native.get("ok"))
        and bool(d9_10m.get("ok"))
        and bool(d9_asof.get("ok")),
        "native": d9_native,
        "ten_minute": d9_10m,
        "asof": d9_asof,
        "channel_scale_per_window": bool(scale_ok),
    }
    if not report["d9_truncation"]["ok"] or not fit_simple_models:
        report["branch"] = decide_branch(report)
        report["notes"] = (
            "Ignore D5–D7 accuracy until truncation matches. "
            "Do not train MtfFusionTransformer."
        )
        return _jsonify(report)

    if len(feat_frame) != n:
        feat_frame = feat_frame.iloc[:n].reset_index(drop=True)
    x, names = last_bar_matrix(feat_frame)
    raw_x, _raw_names = last_bar_matrix(feat_frame, fill_nonfinite=False)
    y = direction_matrix(labeled)
    train_idx = np.arange(slices["train"].start or 0, slices["train"].stop or 0)
    val_idx = np.arange(slices["val"].start or 0, slices["val"].stop or 0)
    n_dev = int(slices["val"].stop or 0)
    bars = horizon_bars_map()
    d5_h: Dict[str, Any] = {}
    d6_h: Dict[str, Any] = {}
    d7_h: Dict[str, Any] = {}
    for j, key in enumerate(LABEL_V2_HORIZON_KEYS):
        d5_h[str(key)] = embargoed_neighbor_probs(
            x,
            y[:, j],
            train_idx,
            val_idx,
            horizon_bars=int(bars[str(key)]),
            n_neighbors=int(knn_k),
            train_cap=int(knn_train_cap),
            query_cap=int(knn_val_cap),
            seed=int(seed) + j,
        )
        d6_h[str(key)] = naive_baselines(
            y[:, j],
            x,
            names,
            val_idx,
            horizon_bars=int(bars[str(key)]),
        )
        d7_h[str(key)] = simple_model_scores(
            x,
            y[:, j],
            train_idx=train_idx,
            val_idx=val_idx,
            horizon_bars=int(bars[str(key)]),
            n_dev=n_dev,
            seed=int(seed),
        )
    report["d5_knn"] = {"horizons": d5_h}
    report["d6_baselines"] = {"horizons": d6_h}
    report["d7_simple_models"] = {"horizons": d7_h}
    print("D5–D7 last-bar baselines done.")
    report["published_transformer_val_balanced_acc"] = dict(
        PUBLISHED_TRANSFORMER_BALANCED_ACC
    )
    if transformer_test:
        report["transformer_test_report"] = dict(transformer_test)
    if bool(score_test) or bool(ablate_groups):

        print("Running last-bar direction-predictability bake-off...")
        report["direction_predictability"] = run_direction_predictability(
            raw_x,
            names,
            y,
            return_matrix(labeled),
            labeled["time"].to_numpy(),
            seed=int(seed),
            score_test=bool(score_test),
            ablate_groups=bool(ablate_groups),
            decision_times=decision_times,
            transformer_test=transformer_test,
            transformer_logits=transformer_logits,
            transformer_y=transformer_y,
            transformer_window_logits=transformer_window_logits,
            transformer_window_times=transformer_window_times,
            class_weight_sensitivity=bool(class_weight_sensitivity),
            mag_targets=path_magnitude_targets(labeled),
        )
    if bool(score_path_regression):

        print("Running last-bar path-edge regression bake-off...")
        report["path_regression"] = run_path_regression(
            raw_x,
            names,
            labeled,
            seed=int(seed),
            score_test=True,
        )
    report["branch"] = decide_branch(report)
    report["notes"] = (
        "stop_target_or_features: do not retrain the Transformer. "
        "later_direction_only: one direction-only run with path weights at 0 "
        "and class weights off. Live stays v11. "
        "direction_predictability readings are pre-registered and are not "
        "a decide_branch token. path_regression is a last-bar path-edge "
        "experiment, not a fused retrain."
    )
    return _jsonify(report)


## Direction predictability

In [ ]:
"""Last-bar direction predictability bake-off. No Transformer training.

Fits unweighted last-bar logistic / HGB on train, picks on val log-loss,
and scores one confirmatory test pass. Pre-registered readings decide
chance vs magnitude vs side — not ``decide_branch``.
"""

from __future__ import annotations

from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, r2_score, roc_auc_score

CHANCE_BALANCED_ACC = 1.0 / 3.0
CHANCE_AUC = 0.50
NOISE_FLOOR = 0.02
NEUTRAL_DUMP_RATE = 0.95
N_BOOT = 200
N_CLASSES = LABEL_V2_DIRECTION_CARDINALITY

VOL_TIME_COLS: Tuple[str, ...] = (
    "rv_16",
    "rv_96",
    "atr_contraction",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
)

PREREGISTERED_READINGS: Dict[str, str] = {
    "no_direction": (
        "Balanced-acc CI includes 0.333 and BEAR-vs-BULL AUC CI includes 0.50"
    ),
    "magnitude_not_side": (
        "NEUTRAL-vs-rest AUC CI excludes 0.50; BEAR-vs-BULL AUC CI includes 0.50"
    ),
    "real_side_signal": (
        "BEAR-vs-BULL AUC CI excludes 0.50 and log-loss beats the train class-prior"
    ),
    "transformer_left_juice": (
        "Last-bar BEAR-vs-BULL AUC CI exceeds Transformer by > 0.02 "
        "and Transformer is not a NEUTRAL dump"
    ),
}

def softmax_rows(logits: np.ndarray) -> np.ndarray:
    """Row-wise softmax for (n, C) logits."""
    z = np.asarray(logits, dtype=np.float64)
    z = z - np.max(z, axis=-1, keepdims=True)
    exp = np.exp(z)
    denom = np.maximum(exp.sum(axis=-1, keepdims=True), 1e-12)
    return exp / denom

def confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_classes: int = N_CLASSES,
) -> np.ndarray:
    """(C, C) counts; rows are true class, columns predicted."""
    yt = np.asarray(y_true, dtype=np.int64)
    yp = np.asarray(y_pred, dtype=np.int64)
    mat = np.zeros((int(n_classes), int(n_classes)), dtype=np.int64)
    valid = (yt >= 0) & (yp >= 0)
    for t, p in zip(yt[valid], yp[valid]):
        if t < n_classes and p < n_classes:
            mat[int(t), int(p)] += 1
    return mat

def per_class_recall(mat: np.ndarray) -> Dict[str, float]:
    """Recall per LABEL_V2 class name from a confusion matrix."""
    out: Dict[str, float] = {}
    for c in range(mat.shape[0]):
        row = float(mat[c].sum())
        name = str(LABEL_V2_DIRECTION_NAMES.get(c, c))
        out[name] = float(mat[c, c] / row) if row > 0.0 else None  # type: ignore[assignment]
    return out

def train_class_prior(y: np.ndarray, n_classes: int = N_CLASSES) -> np.ndarray:
    """Mean one-hot prior on labeled rows (y >= 0)."""
    yt = np.asarray(y, dtype=np.int64)
    yt = yt[yt >= 0]
    prior = np.full(int(n_classes), 1.0 / float(n_classes), dtype=np.float64)
    if yt.size == 0:
        return prior
    counts = np.bincount(yt, minlength=int(n_classes)).astype(np.float64)
    total = float(counts.sum())
    if total <= 0.0:
        return prior
    return counts / total

def multiclass_log_loss(
    y_true: np.ndarray,
    proba: np.ndarray,
    *,
    labels: Optional[Sequence[int]] = None,
) -> float:
    """Mean NLL. Rows with y < 0 are dropped."""
    yt = np.asarray(y_true, dtype=np.int64)
    p = np.asarray(proba, dtype=np.float64)
    valid = yt >= 0
    if not np.any(valid):
        return float("nan")
    labs = list(labels) if labels is not None else list(range(N_CLASSES))
    clipped = np.clip(p[valid], 1e-12, 1.0)
    clipped = clipped / clipped.sum(axis=1, keepdims=True)
    return float(log_loss(yt[valid], clipped, labels=labs))

def prior_log_loss(y_true: np.ndarray, prior: np.ndarray) -> float:
    """NLL of a constant class-prior predictor."""
    n = int(np.asarray(y_true).shape[0])
    tiled = np.repeat(np.asarray(prior, dtype=np.float64).reshape(1, -1), n, axis=0)
    return multiclass_log_loss(y_true, tiled)

def roc_auc_safe(y_bin: np.ndarray, scores: np.ndarray) -> Optional[float]:
    """Binary AUC, or None when only one class is present."""
    yb = np.asarray(y_bin, dtype=np.int64)
    sc = np.asarray(scores, dtype=np.float64)
    if yb.size < 4 or len(np.unique(yb)) < 2:
        return None
    try:
        return float(roc_auc_score(yb, sc))
    except ValueError:
        return None

def neutral_vs_rest_auc(y: np.ndarray, proba: np.ndarray) -> Optional[float]:
    """AUC for NEUTRAL vs BEAR+BULL using P(NEUTRAL)."""
    yt = np.asarray(y, dtype=np.int64)
    p = np.asarray(proba, dtype=np.float64)
    valid = yt >= 0
    if not np.any(valid):
        return None
    y_bin = (yt[valid] == _NEUTRAL).astype(np.int64)
    return roc_auc_safe(y_bin, p[valid, _NEUTRAL])

def bear_vs_bull_auc(y: np.ndarray, proba: np.ndarray) -> Optional[float]:
    """AUC for BULL vs BEAR on non-NEUTRAL rows using P(BULL|side)."""
    yt = np.asarray(y, dtype=np.int64)
    p = np.asarray(proba, dtype=np.float64)
    mask = (yt == _BEAR) | (yt == _BULL)
    if int(mask.sum()) < 4:
        return None
    side = p[mask, _BEAR] + p[mask, _BULL]
    score = p[mask, _BULL] / np.maximum(side, 1e-12)
    y_bin = (yt[mask] == _BULL).astype(np.int64)
    return roc_auc_safe(y_bin, score)

def block_bootstrap_ci(
    values: np.ndarray,
    *,
    block_len: int,
    n_boot: int = N_BOOT,
    seed: int = 0,
    stat: Any = None,
) -> Tuple[Optional[float], Optional[float]]:
    """Percentile CI from a moving-block bootstrap of ``stat(values[idx])``."""
    arr = np.asarray(values)
    n = int(arr.shape[0])
    if n < 8:
        return None, None
    fn = stat if stat is not None else (lambda x: float(np.mean(x)))
    bl = max(int(block_len), 1)
    rng = np.random.default_rng(int(seed))
    n_blocks = int(np.ceil(n / float(bl)))
    stats: List[float] = []
    max_start = max(n - bl + 1, 1)
    for _ in range(int(n_boot)):
        starts = rng.integers(0, max_start, size=n_blocks)
        idx = np.concatenate([np.arange(int(s), min(int(s) + bl, n)) for s in starts])
        idx = idx[:n]
        try:
            val = float(fn(arr[idx]))
        except (TypeError, ValueError):
            continue
        if np.isfinite(val):
            stats.append(val)
    if len(stats) < 8:
        return None, None
    lo, hi = np.percentile(np.asarray(stats, dtype=np.float64), [2.5, 97.5])
    return float(lo), float(hi)

def balanced_acc_ci(
    y: np.ndarray,
    pred: np.ndarray,
    *,
    block_len: int,
    seed: int = 0,
) -> Tuple[Optional[float], Optional[float]]:
    """Bootstrap CI for balanced accuracy on paired (y, pred) rows."""
    yt = np.asarray(y, dtype=np.int64)
    yp = np.asarray(pred, dtype=np.int64)
    paired = np.stack([yt, yp], axis=1)

    def _stat(rows: np.ndarray) -> float:
        return balanced_accuracy(rows[:, 0], rows[:, 1])

    return block_bootstrap_ci(paired, block_len=block_len, seed=seed, stat=_stat)

def auc_ci(
    y_bin: np.ndarray,
    scores: np.ndarray,
    *,
    block_len: int,
    seed: int = 0,
) -> Tuple[Optional[float], Optional[float]]:
    """Bootstrap CI for binary AUC."""
    paired = np.stack(
        [np.asarray(y_bin, dtype=np.float64), np.asarray(scores, dtype=np.float64)],
        axis=1,
    )

    def _stat(rows: np.ndarray) -> float:
        val = roc_auc_safe(rows[:, 0].astype(np.int64), rows[:, 1])
        return float("nan") if val is None else float(val)

    return block_bootstrap_ci(paired, block_len=block_len, seed=seed, stat=_stat)

def ci_includes(lo: Optional[float], hi: Optional[float], value: float) -> bool:
    """True when the CI is missing or covers ``value``."""
    if lo is None or hi is None:
        return True
    return float(lo) <= float(value) <= float(hi)

def train_only_zscore(
    x_train: np.ndarray,
    x_eval: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Column z-score using train mean/std only."""
    tr = np.asarray(x_train, dtype=np.float64)
    ev = np.asarray(x_eval, dtype=np.float64)
    mean = np.mean(tr, axis=0)
    std = np.std(tr, axis=0)
    std = np.where(std < 1e-8, 1.0, std)
    return (tr - mean) / std, (ev - mean) / std

def inverse_frequency_sample_weights(
    y: np.ndarray,
    n_classes: int = N_CLASSES,
) -> np.ndarray:
    """Per-row inverse-frequency weights, mean-normalized."""
    yt = np.asarray(y, dtype=np.int64)
    counts = np.bincount(yt, minlength=int(n_classes)).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = 1.0 / counts[yt]
    weights *= float(len(yt)) / float(weights.sum())
    return weights.astype(np.float32)

def _new_models(seed: int) -> Dict[str, Any]:
    return {
        "logistic": LogisticRegression(
            max_iter=300,
            solver="lbfgs",
            random_state=int(seed),
        ),
        "hist_gbm": HistGradientBoostingClassifier(
            max_depth=3,
            max_iter=40,
            learning_rate=0.1,
            random_state=int(seed),
        ),
    }

def _align_proba(model: Any, proba: np.ndarray) -> np.ndarray:
    """Map sklearn proba columns onto classes 0..C-1."""
    classes = np.asarray(getattr(model, "classes_", np.arange(N_CLASSES)), dtype=np.int64)
    out = np.zeros((proba.shape[0], N_CLASSES), dtype=np.float64)
    for j, cls in enumerate(classes):
        if 0 <= int(cls) < N_CLASSES:
            out[:, int(cls)] = proba[:, j]
    row = out.sum(axis=1, keepdims=True)
    row = np.where(row <= 0.0, 1.0, row)
    return out / row

def fit_predict_proba(
    name: str,
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_eval: np.ndarray,
    *,
    seed: int,
    sample_weight: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """Fit one last-bar model. Logistic uses train-only z-score."""
    models = _new_models(seed)
    model = models[str(name)]
    x_tr = np.asarray(x_train, dtype=np.float64)
    x_ev = np.asarray(x_eval, dtype=np.float64)
    if str(name) == "logistic":
        x_tr, x_ev = train_only_zscore(x_tr, x_ev)
    fit_kw: Dict[str, Any] = {}
    if sample_weight is not None:
        fit_kw["sample_weight"] = sample_weight
    model.fit(x_tr, y_train, **fit_kw)
    pred = np.asarray(model.predict(x_ev), dtype=np.int64)
    proba = _align_proba(model, np.asarray(model.predict_proba(x_ev), dtype=np.float64))
    return pred, proba

def direction_expectancy_atr(
    ret: np.ndarray,
    y_pred: np.ndarray,
    *,
    cost: float = 0.0,
) -> Optional[float]:
    """Mean ATR PnL: long ``+ret``, short ``-ret``, NEUTRAL 0, minus cost on sides."""
    r = np.asarray(ret, dtype=np.float64)
    yp = np.asarray(y_pred, dtype=np.int64)
    finite = np.isfinite(r)
    if not np.any(finite):
        return None
    pnl = np.zeros(r.shape, dtype=np.float64)
    side = finite & ((yp == _BEAR) | (yp == _BULL))
    pnl[finite & (yp == _BULL)] = r[finite & (yp == _BULL)] - float(cost)
    pnl[finite & (yp == _BEAR)] = -r[finite & (yp == _BEAR)] - float(cost)
    pnl[finite & (yp == _NEUTRAL)] = 0.0
    if not np.any(side | (finite & (yp == _NEUTRAL))):
        return None
    return float(np.mean(pnl[finite]))

def decomposition(
    y: np.ndarray,
    pred: np.ndarray,
    proba: np.ndarray,
    *,
    prior: np.ndarray,
    block_len: int,
    seed: int,
    ret: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """Confusion, AUCs, log-loss vs prior, bootstrap CIs, expectancy."""
    yt = np.asarray(y, dtype=np.int64)
    yp = np.asarray(pred, dtype=np.int64)
    p = np.asarray(proba, dtype=np.float64)
    mat = confusion_matrix(yt, yp)
    nvr = neutral_vs_rest_auc(yt, p)
    bvb = bear_vs_bull_auc(yt, p)
    ll = multiclass_log_loss(yt, p)
    pll = prior_log_loss(yt, prior)
    bal = balanced_accuracy(yt, yp)
    bal_lo, bal_hi = balanced_acc_ci(yt, yp, block_len=block_len, seed=seed)
    nvr_lo = nvr_hi = bvb_lo = bvb_hi = None
    valid = yt >= 0
    if np.any(valid):
        y_nvr = (yt[valid] == _NEUTRAL).astype(np.int64)
        nvr_lo, nvr_hi = auc_ci(
            y_nvr, p[valid, _NEUTRAL], block_len=block_len, seed=seed + 1
        )
        side = (yt == _BEAR) | (yt == _BULL)
        if int(side.sum()) >= 8:
            denom = np.maximum(p[side, _BEAR] + p[side, _BULL], 1e-12)
            bvb_lo, bvb_hi = auc_ci(
                (yt[side] == _BULL).astype(np.int64),
                p[side, _BULL] / denom,
                block_len=max(int(block_len) // 2, 1),
                seed=seed + 2,
            )
    dump_rate = float(np.mean(yp == _NEUTRAL)) if yp.size else None
    after_cost = {}
    if ret is not None:
        for cost in COST_ATR:
            after_cost[str(cost)] = direction_expectancy_atr(ret, yp, cost=float(cost))
    return _jsonify(
        {
            "n": int(yt.size),
            "balanced_acc": bal,
            "balanced_acc_ci": [bal_lo, bal_hi],
            "raw_acc": raw_accuracy(yt, yp),
            "log_loss": ll,
            "prior_log_loss": pll,
            "log_loss_beats_prior": (
                bool(np.isfinite(ll) and np.isfinite(pll) and ll < pll)
                if ll is not None and pll is not None
                else False
            ),
            "neutral_vs_rest_auc": nvr,
            "neutral_vs_rest_auc_ci": [nvr_lo, nvr_hi],
            "bear_vs_bull_auc": bvb,
            "bear_vs_bull_auc_ci": [bvb_lo, bvb_hi],
            "confusion": mat.tolist(),
            "recall": per_class_recall(mat),
            "neutral_argmax_rate": dump_rate,
            "neutral_dump": (
                dump_rate is not None and float(dump_rate) >= NEUTRAL_DUMP_RATE
            ),
            "expectancy_atr_after_cost": after_cost,
        }
    )

def classify_reading(row: Mapping[str, Any]) -> str:
    """Map one decomposition to a pre-registered token."""
    bal_lo, bal_hi = _ci_pair(row.get("balanced_acc_ci"))
    nvr_lo, nvr_hi = _ci_pair(row.get("neutral_vs_rest_auc_ci"))
    bvb_lo, bvb_hi = _ci_pair(row.get("bear_vs_bull_auc_ci"))
    no_side = ci_includes(bvb_lo, bvb_hi, CHANCE_AUC)
    no_bal = ci_includes(bal_lo, bal_hi, CHANCE_BALANCED_ACC)
    nvr_signal = not ci_includes(nvr_lo, nvr_hi, CHANCE_AUC)
    side_signal = not no_side
    beats_prior = bool(row.get("log_loss_beats_prior"))
    if side_signal and beats_prior:
        return "real_side_signal"
    if nvr_signal and no_side:
        return "magnitude_not_side"
    if no_bal and no_side:
        return "no_direction"
    if nvr_signal:
        return "magnitude_not_side"
    return "no_direction"

def _ci_pair(raw: Any) -> Tuple[Optional[float], Optional[float]]:
    if not isinstance(raw, (list, tuple)) or len(raw) < 2:
        return None, None
    lo = None if raw[0] is None else float(raw[0])
    hi = None if raw[1] is None else float(raw[1])
    return lo, hi

def classify_transformer_juice(
    last_bar: Mapping[str, Any],
    transformer: Mapping[str, Any],
) -> bool:
    """True when last-bar BEAR-vs-BULL AUC CI clears Transformer by > 0.02."""
    if bool(transformer.get("neutral_dump")):
        return False
    lb = last_bar.get("bear_vs_bull_auc")
    xf = transformer.get("bear_vs_bull_auc")
    if lb is None or xf is None:
        return False
    lo, _hi = _ci_pair(last_bar.get("bear_vs_bull_auc_ci"))
    if lo is None:
        return float(lb) - float(xf) > NOISE_FLOOR
    return float(lo) - float(xf) > NOISE_FLOOR

def map_times_to_rows(
    labeled_times: np.ndarray,
    query_times: np.ndarray,
) -> np.ndarray:
    """Integer row positions for query timestamps; -1 when missing."""
    idx = pd_index(labeled_times)
    q = pd_index(query_times)
    return np.asarray(idx.get_indexer(q), dtype=np.int64)

def pd_index(values: np.ndarray) -> Any:
    """UTC DatetimeIndex without importing pandas at module top for tests..."""
    import pandas as pd

    return pd.DatetimeIndex(pd.to_datetime(values, utc=True))

def finite_feature_mask(features: np.ndarray) -> np.ndarray:
    """Rows that are finite on columns that are actually populated.

    All-NaN columns (OI/funding when the 5m parquet has no derivatives) are
    ignored so EMA200 warm-up is still dropped without emptying the split.
    """
    x = np.asarray(features, dtype=np.float64)
    if x.ndim != 2 or x.size == 0:
        return np.ones(int(x.shape[0]) if x.ndim else 0, dtype=bool)
    col_rate = np.isfinite(x).mean(axis=0)
    keep_cols = col_rate >= 0.50
    if not np.any(keep_cols):
        return np.ones(x.shape[0], dtype=bool)
    return np.isfinite(x[:, keep_cols]).all(axis=1)

def column_index_map(names: Sequence[str]) -> Dict[str, int]:
    return {str(n): i for i, n in enumerate(names)}

def group_column_mask(
    names: Sequence[str],
    group: Sequence[str],
) -> np.ndarray:
    """Boolean mask over feature columns in ``group`` that exist in ``names``."""
    lookup = column_index_map(names)
    mask = np.zeros(len(names), dtype=bool)
    for col in group:
        if col in lookup:
            mask[lookup[col]] = True
    return mask

def kept_eval_indices(
    idx: np.ndarray,
    y: np.ndarray,
    feat_ok: np.ndarray,
    horizon_bars: int,
) -> np.ndarray:
    """Valid, finite-feature, non-overlapping eval rows."""
    raw = np.asarray(idx, dtype=np.int64)
    raw = raw[(raw >= 0) & (raw < len(y))]
    raw = raw[(y[raw] >= 0) & feat_ok[raw]]
    return nonoverlap_indices(raw, int(horizon_bars))

def permutation_null_balanced_acc(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
    *,
    seed: int,
) -> Optional[float]:
    """HGB balanced acc after shuffling train labels."""
    if y_train.size < 16 or y_val.size < 4:
        return None
    rng = np.random.default_rng(int(seed))
    y_perm = np.array(y_train, copy=True)
    rng.shuffle(y_perm)
    if len(np.unique(y_perm)) < 2:
        return None
    pred, _proba = fit_predict_proba(
        "hist_gbm", x_train, y_perm, x_val, seed=int(seed) + 7
    )
    return balanced_accuracy(y_val, pred)

def pick_model_on_val(
    x: np.ndarray,
    y: np.ndarray,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    *,
    seed: int,
    sample_weight: Optional[np.ndarray] = None,
) -> Tuple[str, Dict[str, Any]]:
    """Choose logistic vs HGB by val log-loss (unweighted bake-off)."""
    scores: Dict[str, Any] = {}
    best_name = "hist_gbm"
    best_ll = float("inf")
    prior = train_class_prior(y[train_idx])
    for name in ("logistic", "hist_gbm"):
        pred, proba = fit_predict_proba(
            name,
            x[train_idx],
            y[train_idx],
            x[val_idx],
            seed=seed,
            sample_weight=sample_weight,
        )
        ll = multiclass_log_loss(y[val_idx], proba)
        scores[name] = {
            "log_loss": ll,
            "balanced_acc": balanced_accuracy(y[val_idx], pred),
        }
        if np.isfinite(ll) and ll < best_ll:
            best_ll = ll
            best_name = name
    scores["chosen"] = best_name
    scores["train_prior"] = prior.tolist()
    return best_name, scores

def walk_forward_variance(
    x: np.ndarray,
    y: np.ndarray,
    *,
    n_dev: int,
    horizon_bars: int,
    seed: int,
    feat_ok: np.ndarray,
) -> Dict[str, List[float]]:
    """Last-bar fold balanced acc; not used to pick the test model."""
    out: Dict[str, List[float]] = {"logistic": [], "hist_gbm": []}
    for tr_sl, va_sl in walk_forward_slices(int(n_dev), folds=3, embargo=FUSION_EMBARGO_BARS):
        tr = np.arange(tr_sl.start or 0, tr_sl.stop)
        va = np.arange(va_sl.start or 0, va_sl.stop)
        tr = tr[(tr < len(y)) & (y[tr] >= 0) & feat_ok[tr]]
        va = kept_eval_indices(va, y, feat_ok, horizon_bars)
        if tr.size < 16 or va.size < 4 or len(np.unique(y[tr])) < 2:
            continue
        for name in out:
            pred, _p = fit_predict_proba(name, x[tr], y[tr], x[va], seed=seed)
            out[name].append(balanced_accuracy(y[va], pred))
    return out

def score_split(
    name: str,
    x: np.ndarray,
    y: np.ndarray,
    ret: np.ndarray,
    train_idx: np.ndarray,
    eval_idx: np.ndarray,
    *,
    seed: int,
    block_len: int,
    col_mask: Optional[np.ndarray] = None,
    sample_weight: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """Fit on train columns, decompose eval."""
    cols = col_mask if col_mask is not None else np.ones(x.shape[1], dtype=bool)
    if not np.any(cols):
        return {"ok": False, "reason": "empty_columns"}
    pred, proba = fit_predict_proba(
        name,
        x[train_idx][:, cols],
        y[train_idx],
        x[eval_idx][:, cols],
        seed=seed,
        sample_weight=sample_weight,
    )
    prior = train_class_prior(y[train_idx])
    row = decomposition(
        y[eval_idx],
        pred,
        proba,
        prior=prior,
        block_len=block_len,
        seed=seed,
        ret=ret[eval_idx],
    )
    row["ok"] = True
    row["reading"] = classify_reading(row)
    return row

def logits_decomposition(
    y: np.ndarray,
    logits: np.ndarray,
    *,
    prior: np.ndarray,
    block_len: int,
    seed: int,
    ret: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """Decompose a frozen classifier from logits (no refit)."""
    proba = softmax_rows(logits)
    pred = np.argmax(proba, axis=-1).astype(np.int64)
    row = decomposition(
        y, pred, proba, prior=prior, block_len=block_len, seed=seed, ret=ret
    )
    row["ok"] = True
    row["reading"] = classify_reading(row)
    row["source"] = "logits"
    return row

def magnitude_vs_rv16(
    x: np.ndarray,
    names: Sequence[str],
    targets: Mapping[str, np.ndarray],
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    *,
    seed: int,
) -> Dict[str, Any]:
    """HGB R² / Spearman vs an rv_16-only baseline on val."""
    lookup = column_index_map(names)
    if "rv_16" not in lookup:
        return {"ok": False, "reason": "missing_rv_16"}
    rv_i = lookup["rv_16"]
    out: Dict[str, Any] = {"ok": True, "targets": {}}
    for field, arr in targets.items():
        y = np.asarray(arr, dtype=np.float64)
        tr = train_idx[np.isfinite(y[train_idx])]
        va = val_idx[np.isfinite(y[val_idx])]
        if tr.size < 16 or va.size < 8:
            out["targets"][str(field)] = {"ok": False, "reason": "insufficient_rows"}
            continue
        full = HistGradientBoostingRegressor(
            max_depth=3, max_iter=40, learning_rate=0.1, random_state=int(seed)
        )
        base = HistGradientBoostingRegressor(
            max_depth=3, max_iter=40, learning_rate=0.1, random_state=int(seed)
        )
        full.fit(x[tr], y[tr])
        base.fit(x[tr][:, [rv_i]], y[tr])
        pred_f = full.predict(x[va])
        pred_b = base.predict(x[va][:, [rv_i]])
        yv = y[va]
        out["targets"][str(field)] = {
            "ok": True,
            "r2": _r2(yv, pred_f),
            "spearman": _spearman(yv, pred_f),
            "rv16_r2": _r2(yv, pred_b),
            "rv16_spearman": _spearman(yv, pred_b),
            "n": int(va.size),
        }
    return _jsonify(out)

def _r2(y: np.ndarray, pred: np.ndarray) -> Optional[float]:
    if y.size < 4:
        return None
    try:
        return float(r2_score(y, pred))
    except ValueError:
        return None

def _spearman(y: np.ndarray, pred: np.ndarray) -> Optional[float]:
    if y.size < 4:
        return None
    from scipy.stats import spearmanr

    coef, _p = spearmanr(y, pred)
    if coef is None or not np.isfinite(coef):
        return None
    return float(coef)

def ablation_table(
    chosen: str,
    x: np.ndarray,
    y: np.ndarray,
    ret: np.ndarray,
    names: Sequence[str],
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    *,
    seed: int,
    block_len: int,
) -> Dict[str, Any]:
    """Val-only knockout, add-one-in, and vol+time null."""
    full_mask = np.ones(x.shape[1], dtype=bool)
    full = score_split(
        chosen, x, y, ret, train_idx, val_idx, seed=seed, block_len=block_len
    )
    knockouts: Dict[str, Any] = {}
    add_one: Dict[str, Any] = {}
    for group, cols in V14_FEATURE_GROUPS.items():
        gmask = group_column_mask(names, cols)
        if not np.any(gmask):
            continue
        drop = score_split(
            chosen,
            x,
            y,
            ret,
            train_idx,
            val_idx,
            seed=seed,
            block_len=block_len,
            col_mask=full_mask & ~gmask,
        )
        only = score_split(
            chosen,
            x,
            y,
            ret,
            train_idx,
            val_idx,
            seed=seed,
            block_len=block_len,
            col_mask=gmask,
        )
        full_bal = float(full.get("balanced_acc") or 0.0)
        drop_bal = float(drop.get("balanced_acc") or 0.0)
        delta = full_bal - drop_bal
        knockouts[str(group)] = {
            **drop,
            "delta_vs_full": delta,
            "readable": abs(delta) >= NOISE_FLOOR,
        }
        add_one[str(group)] = only
    vol_mask = group_column_mask(names, VOL_TIME_COLS)
    vol_time = score_split(
        chosen,
        x,
        y,
        ret,
        train_idx,
        val_idx,
        seed=seed,
        block_len=block_len,
        col_mask=vol_mask,
    )
    return {
        "full": full,
        "knockout": knockouts,
        "add_one_in": add_one,
        "vol_time_null": vol_time,
        "noise_floor": NOISE_FLOOR,
    }

def run_direction_predictability(
    features: np.ndarray,
    names: Sequence[str],
    labels: np.ndarray,
    returns: np.ndarray,
    labeled_times: np.ndarray,
    *,
    seed: int = 42,
    score_test: bool = True,
    ablate_groups: bool = True,
    decision_times: Optional[np.ndarray] = None,
    transformer_test: Optional[Mapping[str, Any]] = None,
    transformer_logits: Optional[Mapping[str, np.ndarray]] = None,
    transformer_y: Optional[Mapping[str, np.ndarray]] = None,
    transformer_window_logits: Optional[np.ndarray] = None,
    transformer_window_times: Optional[np.ndarray] = None,
    class_weight_sensitivity: bool = False,
    mag_targets: Optional[Mapping[str, Dict[str, np.ndarray]]] = None,
) -> Dict[str, Any]:
    """Last-bar bake-off for all Label V2 horizons."""
    raw = np.asarray(features, dtype=np.float64)
    feat_ok = finite_feature_mask(raw)
    x = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
    y_all = np.asarray(labels, dtype=np.int64)
    ret_all = np.asarray(returns, dtype=np.float64)
    n = int(x.shape[0])
    slices = purged_index_slices(n)
    bars = horizon_bars_map()
    report: Dict[str, Any] = {
        "preregistered_readings": dict(PREREGISTERED_READINGS),
        "noise_floor": NOISE_FLOOR,
        "use_class_weights": False,
        "split": {
            "train": [slices["train"].start or 0, slices["train"].stop],
            "val": [slices["val"].start, slices["val"].stop],
            "test": [slices["test"].start, slices["test"].stop],
        },
        "horizons": {},
        "transformer_json": dict(transformer_test or {}),
    }
    stride_pos: Optional[np.ndarray] = None
    time_align: Dict[str, Any] = {"ok": False}
    if decision_times is not None:
        stride_pos = map_times_to_rows(labeled_times, decision_times)
        test_sl = slices["test"]
        mapped = stride_pos[stride_pos >= 0]
        in_test = mapped[
            (mapped >= int(test_sl.start or 0)) & (mapped < int(test_sl.stop or n))
        ]
        test_times = pd_index(
            labeled_times[int(test_sl.start or 0) : int(test_sl.stop or n)]
        )
        if in_test.size and len(test_times):
            in_test_times = pd_index(labeled_times[in_test])
            inside = bool(
                in_test_times.min() >= test_times.min()
                and in_test_times.max() <= test_times.max()
            )
            time_align = {
                "ok": inside,
                "n_mapped": int((stride_pos >= 0).sum()),
                "n_mapped_in_calendar_test": int(in_test.size),
                "stride_test_min": str(in_test_times.min()),
                "stride_test_max": str(in_test_times.max()),
                "calendar_test_min": str(test_times.min()),
                "calendar_test_max": str(test_times.max()),
            }
        else:
            time_align = {"ok": False, "reason": "empty_join"}
    report["time_align"] = time_align

    train_sl = slices["train"]
    val_sl = slices["val"]
    test_sl = slices["test"]
    n_dev = int(val_sl.stop or 0)

    for j, key in enumerate(LABEL_V2_HORIZON_KEYS):
        print(
            f"  horizon {key} ({j + 1}/{len(LABEL_V2_HORIZON_KEYS)}) "
            f"score_test={score_test} ablate={ablate_groups}"
        )
        y = y_all[:, j] if y_all.ndim == 2 else y_all
        ret = ret_all[:, j] if ret_all.ndim == 2 else ret_all
        h_bars = int(bars[str(key)])
        block_len = max(h_bars, 12)
        train_idx = np.arange(train_sl.start or 0, train_sl.stop or 0)
        train_idx = train_idx[(y[train_idx] >= 0) & feat_ok[train_idx]]
        val_idx = kept_eval_indices(
            np.arange(val_sl.start or 0, val_sl.stop or 0), y, feat_ok, h_bars
        )
        test_idx = kept_eval_indices(
            np.arange(test_sl.start or 0, test_sl.stop or n), y, feat_ok, h_bars
        )
        horizon: Dict[str, Any] = {
            "horizon_bars": h_bars,
            "n_train": int(train_idx.size),
            "n_val": int(val_idx.size),
            "n_test": int(test_idx.size),
        }
        if train_idx.size < 16 or val_idx.size < 4 or len(np.unique(y[train_idx])) < 2:
            horizon["ok"] = False
            horizon["reason"] = "insufficient_rows"
            report["horizons"][str(key)] = horizon
            print(f"  horizon {key} skipped n_train={train_idx.size} n_val={val_idx.size}")
            continue
        chosen, pick = pick_model_on_val(
            x, y, train_idx, val_idx, seed=seed
        )
        horizon["model_selection"] = pick
        horizon["chosen"] = chosen
        horizon["walk_forward"] = {
            name: {
                "mean": float(np.mean(vals)) if vals else None,
                "std": float(np.std(vals)) if len(vals) > 1 else None,
                "n_folds": int(len(vals)),
            }
            for name, vals in walk_forward_variance(
                x, y, n_dev=n_dev, horizon_bars=h_bars, seed=seed, feat_ok=feat_ok
            ).items()
        }
        horizon["val"] = score_split(
            chosen,
            x,
            y,
            ret,
            train_idx,
            val_idx,
            seed=seed,
            block_len=block_len,
        )
        horizon["val"]["permutation_null_balanced_acc"] = permutation_null_balanced_acc(
            x[train_idx], y[train_idx], x[val_idx], y[val_idx], seed=seed
        )
        if ablate_groups:
            horizon["ablation_val"] = ablation_table(
                chosen,
                x,
                y,
                ret,
                names,
                train_idx,
                val_idx,
                seed=seed,
                block_len=block_len,
            )
        if score_test and test_idx.size >= 4:
            horizon["test"] = score_split(
                chosen,
                x,
                y,
                ret,
                train_idx,
                test_idx,
                seed=seed,
                block_len=block_len,
            )
            vol_mask = group_column_mask(names, VOL_TIME_COLS)
            horizon["test_vol_time_null"] = score_split(
                chosen,
                x,
                y,
                ret,
                train_idx,
                test_idx,
                seed=seed,
                block_len=block_len,
                col_mask=vol_mask,
            )
        else:
            horizon["test_unused"] = True
        if stride_pos is not None:
            stride_test = stride_pos[stride_pos >= 0]
            stride_test = stride_test[
                (stride_test >= int(test_sl.start or 0))
                & (stride_test < int(test_sl.stop or n))
            ]
            stride_test = kept_eval_indices(stride_test, y, feat_ok, h_bars)
            if stride_test.size >= 4:
                horizon["test_stride4"] = score_split(
                    chosen,
                    x,
                    y,
                    ret,
                    train_idx,
                    stride_test,
                    seed=seed,
                    block_len=block_len,
                )
        if class_weight_sensitivity:
            weights = inverse_frequency_sample_weights(y[train_idx])
            horizon["class_weight_sensitivity_val"] = score_split(
                chosen,
                x,
                y,
                ret,
                train_idx,
                val_idx,
                seed=seed,
                block_len=block_len,
                sample_weight=weights,
            )
        if mag_targets is not None and str(key) in mag_targets:
            horizon["magnitude"] = magnitude_vs_rv16(
                x,
                names,
                mag_targets[str(key)],
                train_idx,
                val_idx,
                seed=seed,
            )
        prior = train_class_prior(y[train_idx])

        def _window_logits_for(sl: slice) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
            if transformer_window_logits is None or transformer_window_times is None:
                return None, None
            wpos = map_times_to_rows(labeled_times, transformer_window_times)
            keep = (
                (wpos >= int(sl.start or 0))
                & (wpos < int(sl.stop or n))
                & (wpos >= 0)
                & feat_ok[np.clip(wpos, 0, n - 1)]
                & (y[np.clip(wpos, 0, n - 1)] >= 0)
            )
            if int(keep.sum()) < 4:
                return None, None
            return transformer_window_logits[keep, j, :], y[wpos[keep]]

        xf_val_logits, xf_val_y = _window_logits_for(val_sl)
        if xf_val_logits is not None and xf_val_y is not None:
            horizon["transformer_val"] = logits_decomposition(
                xf_val_y,
                xf_val_logits,
                prior=prior,
                block_len=block_len,
                seed=seed,
            )
        xf_logits = (transformer_logits or {}).get(str(key))
        xf_y = (transformer_y or {}).get(str(key))
        if xf_logits is None:
            xf_logits, xf_y = _window_logits_for(test_sl)
        if xf_logits is not None and xf_y is not None:
            horizon["transformer"] = logits_decomposition(
                xf_y,
                xf_logits,
                prior=prior,
                block_len=block_len,
                seed=seed,
            )
            lb_cmp = horizon.get("test_stride4") or horizon.get("test") or {}
            horizon["transformer_left_juice"] = classify_transformer_juice(
                lb_cmp, horizon["transformer"]
            )
        elif transformer_test:
            raw_json = (transformer_test.get("test_metrics") or {}).get(str(key)) or {}
            horizon["transformer_json"] = dict(raw_json)
            horizon["transformer_confusion"] = False
        horizon["discovery_reading"] = (horizon.get("val") or {}).get("reading")
        horizon["confirmatory_reading"] = (horizon.get("test") or {}).get("reading")
        report["horizons"][str(key)] = horizon
        print(f"  horizon {key} done reading={horizon.get('confirmatory_reading')}")
    return _jsonify(report)

def path_magnitude_targets(
    labeled: Any,
) -> Dict[str, Dict[str, np.ndarray]]:
    """``|ret|`` and path MFE/MAE columns per horizon."""
    out: Dict[str, Dict[str, np.ndarray]] = {}
    for key in LABEL_V2_HORIZON_KEYS:
        pack: Dict[str, np.ndarray] = {}
        ret_col = f"{key}_ret"
        if ret_col in labeled.columns:
            pack["abs_ret"] = np.abs(labeled[ret_col].to_numpy(dtype=np.float64))
        for field in LABEL_V2_REG_FIELDS:
            col = f"{key}_{field}"
            if col in labeled.columns:
                pack[str(field)] = labeled[col].to_numpy(dtype=np.float64)
        if pack:
            out[str(key)] = pack
    return out

def return_matrix(labeled: Any) -> np.ndarray:
    """``(n, H)`` ATR-normalized endpoint returns."""
    n = len(labeled)
    r = np.full((n, len(LABEL_V2_HORIZON_KEYS)), np.nan, dtype=np.float64)
    for j, key in enumerate(LABEL_V2_HORIZON_KEYS):
        col = f"{key}_ret"
        if col in labeled.columns:
            r[:, j] = labeled[col].to_numpy(dtype=np.float64)
    return r


## Path-edge regression

In [ ]:
"""Last-bar path-edge regression bake-off. No Transformer training.

Primary target is signed path quality
``(long_mfe - long_mae) - (short_mfe - short_mae)``, not 3-class direction
and not ``|ret|``. Policy, not this module, would later threshold a scalar.
Live v11 is unchanged.
"""

from __future__ import annotations

from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

CHANCE_AUC = 0.50
PATH_SIGN_FLOOR = 0.25
N_BOOT_REG = N_BOOT

PREREGISTERED_READINGS: Dict[str, str] = {
    "no_path_signal": (
        "Signed path-edge Spearman CI includes 0 and sign-AUC CI includes 0.50"
    ),
    "vol_not_edge": (
        "|path_edge| tracks rv_16 (Spearman CI excludes 0) but signed "
        "path-edge Spearman CI includes 0"
    ),
    "real_path_edge": (
        "Signed Spearman CI excludes 0, beats rv_16 by > 0.02, and "
        "path sign-AUC CI excludes 0.50"
    ),
    "endpoint_still_dead": (
        "Path-edge is real but sign-AUC vs endpoint ret on |R|>=theta "
        "includes 0.50"
    ),
}

def signed_path_edge(
    long_mfe: np.ndarray,
    long_mae: np.ndarray,
    short_mfe: np.ndarray,
    short_mae: np.ndarray,
) -> np.ndarray:
    """Long-path quality minus short-path quality (ATR units).

    Args:
        long_mfe: Upside excursion.
        long_mae: Adverse move to that upside.
        short_mfe: Downside excursion.
        short_mae: Adverse move to that downside.

    Returns:
        Positive when the long path was cleaner than the short path.
    """
    long_edge = np.asarray(long_mfe, dtype=np.float64) - np.asarray(
        long_mae, dtype=np.float64
    )
    short_edge = np.asarray(short_mfe, dtype=np.float64) - np.asarray(
        short_mae, dtype=np.float64
    )
    return long_edge - short_edge

def path_target_pack(labeled: Any, key: str) -> Dict[str, np.ndarray]:
    """Named arrays for one Label V2 horizon."""
    pack: Dict[str, np.ndarray] = {}
    ret_col = f"{key}_ret"
    if ret_col in labeled.columns:
        pack["ret"] = labeled[ret_col].to_numpy(dtype=np.float64)
    needed = ("long_mfe", "long_mae", "short_mfe", "short_mae")
    cols = {field: f"{key}_{field}" for field in needed}
    if all(col in labeled.columns for col in cols.values()):
        pack["path_signed"] = signed_path_edge(
            labeled[cols["long_mfe"]].to_numpy(dtype=np.float64),
            labeled[cols["long_mae"]].to_numpy(dtype=np.float64),
            labeled[cols["short_mfe"]].to_numpy(dtype=np.float64),
            labeled[cols["short_mae"]].to_numpy(dtype=np.float64),
        )
        pack["long_edge"] = labeled[cols["long_mfe"]].to_numpy(
            dtype=np.float64
        ) - labeled[cols["long_mae"]].to_numpy(dtype=np.float64)
        pack["short_edge"] = labeled[cols["short_mfe"]].to_numpy(
            dtype=np.float64
        ) - labeled[cols["short_mae"]].to_numpy(dtype=np.float64)
    return pack

def spearman_safe(y: np.ndarray, pred: np.ndarray) -> Optional[float]:
    """Spearman rho on finite pairs."""
    yt = np.asarray(y, dtype=np.float64)
    yp = np.asarray(pred, dtype=np.float64)
    mask = np.isfinite(yt) & np.isfinite(yp)
    if int(mask.sum()) < 8:
        return None
    from scipy.stats import spearmanr

    coef, _p = spearmanr(yt[mask], yp[mask])
    if coef is None or not np.isfinite(coef):
        return None
    return float(coef)

def r2_safe(y: np.ndarray, pred: np.ndarray) -> Optional[float]:
    """R² on finite pairs; None when sklearn refuses."""
    yt = np.asarray(y, dtype=np.float64)
    yp = np.asarray(pred, dtype=np.float64)
    mask = np.isfinite(yt) & np.isfinite(yp)
    if int(mask.sum()) < 8:
        return None
    try:
        return float(r2_score(yt[mask], yp[mask]))
    except ValueError:
        return None

def sign_auc(
    y: np.ndarray,
    pred: np.ndarray,
    *,
    min_abs: float,
) -> Optional[float]:
    """AUC of pred vs sign(y) on rows with |y| >= min_abs."""
    yt = np.asarray(y, dtype=np.float64)
    yp = np.asarray(pred, dtype=np.float64)
    mask = np.isfinite(yt) & np.isfinite(yp) & (np.abs(yt) >= float(min_abs))
    if int(mask.sum()) < 8:
        return None
    y_bin = (yt[mask] > 0.0).astype(np.int64)
    return roc_auc_safe(y_bin, yp[mask])

def sign_expectancy_atr(
    ret: np.ndarray,
    pred: np.ndarray,
    *,
    cost: float,
    deadzone: float = 0.0,
) -> Optional[float]:
    """Mean ATR PnL from sign(pred), minus cost on non-zero sides."""
    r = np.asarray(ret, dtype=np.float64)
    p = np.asarray(pred, dtype=np.float64)
    finite = np.isfinite(r) & np.isfinite(p)
    if not np.any(finite):
        return None
    side = np.zeros(p.shape, dtype=np.float64)
    side[p > float(deadzone)] = 1.0
    side[p < -float(deadzone)] = -1.0
    pnl = side * r
    traded = finite & (side != 0.0)
    pnl[traded] -= float(cost)
    return float(np.mean(pnl[finite]))

def kept_reg_indices(
    idx: np.ndarray,
    y: np.ndarray,
    feat_ok: np.ndarray,
    horizon_bars: int,
) -> np.ndarray:
    """Finite-target, finite-feature, non-overlapping eval rows."""
    raw = np.asarray(idx, dtype=np.int64)
    raw = raw[(raw >= 0) & (raw < len(y))]
    raw = raw[np.isfinite(y[raw]) & feat_ok[raw]]
    return nonoverlap_indices(raw, int(horizon_bars))

def _new_regressors(seed: int) -> Dict[str, Any]:
    return {
        "ridge": Ridge(alpha=1.0),
        "hist_gbm": HistGradientBoostingRegressor(
            max_depth=3,
            max_iter=40,
            learning_rate=0.1,
            random_state=int(seed),
        ),
    }

def fit_predict_reg(
    name: str,
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_eval: np.ndarray,
    *,
    seed: int,
) -> np.ndarray:
    """Fit Ridge (train-only z-score) or HGB (raw)."""
    models = _new_regressors(seed)
    model = models[str(name)]
    x_tr = np.asarray(x_train, dtype=np.float64)
    x_ev = np.asarray(x_eval, dtype=np.float64)
    y_tr = np.asarray(y_train, dtype=np.float64)
    if str(name) == "ridge":
        x_tr, x_ev = train_only_zscore(x_tr, x_ev)
    model.fit(x_tr, y_tr)
    return np.asarray(model.predict(x_ev), dtype=np.float64)

def spearman_ci(
    y: np.ndarray,
    pred: np.ndarray,
    *,
    block_len: int,
    seed: int,
) -> Tuple[Optional[float], Optional[float]]:
    """Block-bootstrap CI for Spearman rho."""
    yt = np.asarray(y, dtype=np.float64)
    yp = np.asarray(pred, dtype=np.float64)
    mask = np.isfinite(yt) & np.isfinite(yp)
    paired = np.stack([yt[mask], yp[mask]], axis=1)
    if paired.shape[0] < 8:
        return None, None

    def _stat(rows: np.ndarray) -> float:
        val = spearman_safe(rows[:, 0], rows[:, 1])
        return float("nan") if val is None else float(val)

    return block_bootstrap_ci(
        paired, block_len=block_len, n_boot=N_BOOT_REG, seed=seed, stat=_stat
    )

def sign_auc_ci(
    y: np.ndarray,
    pred: np.ndarray,
    *,
    min_abs: float,
    block_len: int,
    seed: int,
) -> Tuple[Optional[float], Optional[float]]:
    """Block-bootstrap CI for sign AUC."""
    yt = np.asarray(y, dtype=np.float64)
    yp = np.asarray(pred, dtype=np.float64)
    mask = np.isfinite(yt) & np.isfinite(yp) & (np.abs(yt) >= float(min_abs))
    paired = np.stack([yt[mask], yp[mask]], axis=1)
    if paired.shape[0] < 8:
        return None, None

    def _stat(rows: np.ndarray) -> float:
        val = sign_auc(rows[:, 0], rows[:, 1], min_abs=0.0)
        return float("nan") if val is None else float(val)

    return block_bootstrap_ci(
        paired, block_len=block_len, n_boot=N_BOOT_REG, seed=seed, stat=_stat
    )

def _ci_pair(raw: Any) -> Tuple[Optional[float], Optional[float]]:
    if not isinstance(raw, (list, tuple)) or len(raw) < 2:
        return None, None
    lo = None if raw[0] is None else float(raw[0])
    hi = None if raw[1] is None else float(raw[1])
    return lo, hi

def classify_path_reading(row: Mapping[str, Any]) -> str:
    """Map one path-edge decomposition to a pre-registered token."""
    sp_lo, sp_hi = _ci_pair(row.get("spearman_ci"))
    auc_lo, auc_hi = _ci_pair(row.get("sign_auc_path_ci"))
    ret_lo, ret_hi = _ci_pair(row.get("sign_auc_ret_ci"))
    signed_signal = not ci_includes(sp_lo, sp_hi, 0.0)
    sign_signal = not ci_includes(auc_lo, auc_hi, CHANCE_AUC)
    sp = row.get("spearman")
    rv = row.get("rv16_spearman")
    beats_rv = (
        sp is not None
        and rv is not None
        and float(sp) - float(rv) > NOISE_FLOOR
    )
    abs_lo, abs_hi = _ci_pair(row.get("abs_vs_rv16_spearman_ci"))
    vol_only = (not signed_signal) and (not ci_includes(abs_lo, abs_hi, 0.0))
    if signed_signal and sign_signal and beats_rv:
        if ci_includes(ret_lo, ret_hi, CHANCE_AUC):
            return "endpoint_still_dead"
        return "real_path_edge"
    if vol_only:
        return "vol_not_edge"
    return "no_path_signal"

def score_reg_split(
    name: str,
    x: np.ndarray,
    y: np.ndarray,
    ret: np.ndarray,
    train_idx: np.ndarray,
    eval_idx: np.ndarray,
    *,
    seed: int,
    block_len: int,
    theta: float,
    rv_idx: Optional[int],
    col_mask: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    """Fit on train, decompose eval for path_signed ``y``."""
    cols = col_mask if col_mask is not None else np.ones(x.shape[1], dtype=bool)
    if not np.any(cols):
        return {"ok": False, "reason": "empty_columns"}
    pred = fit_predict_reg(
        name, x[train_idx][:, cols], y[train_idx], x[eval_idx][:, cols], seed=seed
    )
    yt = y[eval_idx]
    rt = ret[eval_idx]
    sp = spearman_safe(yt, pred)
    sp_lo, sp_hi = spearman_ci(yt, pred, block_len=block_len, seed=seed)
    path_auc = sign_auc(yt, pred, min_abs=PATH_SIGN_FLOOR)
    path_lo, path_hi = sign_auc_ci(
        yt, pred, min_abs=PATH_SIGN_FLOOR, block_len=block_len, seed=seed
    )
    ret_auc = sign_auc(rt, pred, min_abs=float(theta))
    ret_lo, ret_hi = sign_auc_ci(
        rt, pred, min_abs=float(theta), block_len=block_len, seed=seed + 1
    )
    rv_pred = None
    rv_sp = None
    if rv_idx is not None:
        rv_pred = fit_predict_reg(
            "hist_gbm",
            x[train_idx][:, [rv_idx]],
            y[train_idx],
            x[eval_idx][:, [rv_idx]],
            seed=seed + 3,
        )
        rv_sp = spearman_safe(yt, rv_pred)
    abs_y = np.abs(yt)
    abs_rv_sp = None
    abs_lo, abs_hi = None, None
    if rv_idx is not None:
        abs_rv_sp = spearman_safe(abs_y, x[eval_idx][:, rv_idx])
        abs_lo, abs_hi = spearman_ci(
            abs_y, x[eval_idx][:, rv_idx], block_len=block_len, seed=seed + 5
        )
    after_cost = {}
    for cost in COST_ATR:
        after_cost[str(cost)] = sign_expectancy_atr(rt, pred, cost=float(cost))
    row = {
        "ok": True,
        "n": int(eval_idx.size),
        "spearman": sp,
        "spearman_ci": [sp_lo, sp_hi],
        "r2": r2_safe(yt, pred),
        "rv16_spearman": rv_sp,
        "rv16_r2": r2_safe(yt, rv_pred) if rv_pred is not None else None,
        "abs_vs_rv16_spearman": abs_rv_sp,
        "abs_vs_rv16_spearman_ci": [abs_lo, abs_hi],
        "sign_auc_path": path_auc,
        "sign_auc_path_ci": [path_lo, path_hi],
        "sign_auc_ret": ret_auc,
        "sign_auc_ret_ci": [ret_lo, ret_hi],
        "theta": float(theta),
        "path_sign_floor": PATH_SIGN_FLOOR,
        "expectancy_atr_after_cost": after_cost,
    }
    row["reading"] = classify_path_reading(row)
    return _jsonify(row)

def pick_regressor_on_val(
    x: np.ndarray,
    y: np.ndarray,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    *,
    seed: int,
) -> Tuple[str, Dict[str, Any]]:
    """Choose ridge vs HGB by val Spearman of the primary target."""
    scores: Dict[str, Any] = {}
    best_name = "hist_gbm"
    best_sp = float("-inf")
    for name in ("ridge", "hist_gbm"):
        pred = fit_predict_reg(name, x[train_idx], y[train_idx], x[val_idx], seed=seed)
        sp = spearman_safe(y[val_idx], pred)
        scores[name] = {"spearman": sp, "r2": r2_safe(y[val_idx], pred)}
        if sp is not None and float(sp) > best_sp:
            best_sp = float(sp)
            best_name = name
    scores["chosen"] = best_name
    return best_name, scores

def permutation_null_spearman(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_val: np.ndarray,
    y_val: np.ndarray,
    *,
    seed: int,
) -> Optional[float]:
    """HGB Spearman after shuffling train targets."""
    if y_train.size < 16 or y_val.size < 8:
        return None
    rng = np.random.default_rng(int(seed))
    y_perm = np.array(y_train, copy=True)
    rng.shuffle(y_perm)
    pred = fit_predict_reg("hist_gbm", x_train, y_perm, x_val, seed=int(seed) + 9)
    return spearman_safe(y_val, pred)

def run_path_regression(
    features: np.ndarray,
    names: Sequence[str],
    labeled: Any,
    *,
    seed: int = 42,
    score_test: bool = True,
) -> Dict[str, Any]:
    """Last-bar path-edge bake-off for all Label V2 horizons."""
    raw = np.asarray(features, dtype=np.float64)
    feat_ok = finite_feature_mask(raw)
    x = np.nan_to_num(raw, nan=0.0, posinf=0.0, neginf=0.0)
    n = int(x.shape[0])
    slices = purged_index_slices(n)
    bars = horizon_bars_map()
    lookup = {str(name): i for i, name in enumerate(names)}
    rv_idx = lookup.get("rv_16")
    train_sl = slices["train"]
    val_sl = slices["val"]
    test_sl = slices["test"]
    n_dev = int(val_sl.stop or 0)
    report: Dict[str, Any] = {
        "preregistered_readings": dict(PREREGISTERED_READINGS),
        "noise_floor": NOISE_FLOOR,
        "primary_target": "path_signed",
        "not_primary": ["abs_ret", "direction_class"],
        "split": {
            "train": [train_sl.start or 0, train_sl.stop],
            "val": [val_sl.start, val_sl.stop],
            "test": [test_sl.start, test_sl.stop],
        },
        "horizons": {},
    }
    vol_mask = group_column_mask(names, VOL_TIME_COLS)
    for key in LABEL_V2_HORIZON_KEYS:
        print(f"  path-regression horizon {key}")
        pack = path_target_pack(labeled, str(key))
        y = pack.get("path_signed")
        ret = pack.get("ret")
        horizon: Dict[str, Any] = {"horizon_bars": int(bars[str(key)])}
        if y is None or ret is None:
            horizon["ok"] = False
            horizon["reason"] = "missing_path_columns"
            report["horizons"][str(key)] = horizon
            continue
        h_bars = int(bars[str(key)])
        block_len = max(h_bars, 12)
        theta = float(LABEL_V2_THETA_FROZEN.get(str(key), 0.50))
        train_idx = np.arange(train_sl.start or 0, train_sl.stop or 0)
        train_idx = train_idx[np.isfinite(y[train_idx]) & feat_ok[train_idx]]
        val_idx = kept_reg_indices(
            np.arange(val_sl.start or 0, val_sl.stop or 0), y, feat_ok, h_bars
        )
        test_idx = kept_reg_indices(
            np.arange(test_sl.start or 0, test_sl.stop or n), y, feat_ok, h_bars
        )
        horizon["n_train"] = int(train_idx.size)
        horizon["n_val"] = int(val_idx.size)
        horizon["n_test"] = int(test_idx.size)
        if train_idx.size < 16 or val_idx.size < 8:
            horizon["ok"] = False
            horizon["reason"] = "insufficient_rows"
            report["horizons"][str(key)] = horizon
            continue
        chosen, pick = pick_regressor_on_val(
            x, y, train_idx, val_idx, seed=seed
        )
        horizon["model_selection"] = pick
        horizon["chosen"] = chosen
        wf: List[float] = []
        for tr_sl, va_sl in walk_forward_slices(
            n_dev, folds=3, embargo=FUSION_EMBARGO_BARS
        ):
            tr = np.arange(tr_sl.start or 0, tr_sl.stop)
            va = np.arange(va_sl.start or 0, va_sl.stop)
            tr = tr[(tr < len(y)) & np.isfinite(y[tr]) & feat_ok[tr]]
            va = kept_reg_indices(va, y, feat_ok, h_bars)
            if tr.size < 16 or va.size < 8:
                continue
            pred = fit_predict_reg(chosen, x[tr], y[tr], x[va], seed=seed)
            sp = spearman_safe(y[va], pred)
            if sp is not None:
                wf.append(sp)
        horizon["walk_forward_spearman"] = {
            "mean": float(np.mean(wf)) if wf else None,
            "std": float(np.std(wf)) if len(wf) > 1 else None,
            "n_folds": int(len(wf)),
        }
        horizon["val"] = score_reg_split(
            chosen,
            x,
            y,
            ret,
            train_idx,
            val_idx,
            seed=seed,
            block_len=block_len,
            theta=theta,
            rv_idx=rv_idx,
        )
        horizon["val"]["permutation_null_spearman"] = permutation_null_spearman(
            x[train_idx], y[train_idx], x[val_idx], y[val_idx], seed=seed
        )
        horizon["val_vol_time_null"] = score_reg_split(
            chosen,
            x,
            y,
            ret,
            train_idx,
            val_idx,
            seed=seed,
            block_len=block_len,
            theta=theta,
            rv_idx=rv_idx,
            col_mask=vol_mask,
        )
        if score_test and test_idx.size >= 8:
            horizon["test"] = score_reg_split(
                chosen,
                x,
                y,
                ret,
                train_idx,
                test_idx,
                seed=seed,
                block_len=block_len,
                theta=theta,
                rv_idx=rv_idx,
            )
            horizon["test_vol_time_null"] = score_reg_split(
                chosen,
                x,
                y,
                ret,
                train_idx,
                test_idx,
                seed=seed,
                block_len=block_len,
                theta=theta,
                rv_idx=rv_idx,
                col_mask=vol_mask,
            )
        else:
            horizon["test_unused"] = True
        if "ret" in pack:
            ret_val = score_reg_split(
                chosen,
                x,
                pack["ret"],
                ret,
                train_idx,
                val_idx,
                seed=seed,
                block_len=block_len,
                theta=theta,
                rv_idx=rv_idx,
            )
            horizon["val_endpoint_ret_control"] = {
                "spearman": ret_val.get("spearman"),
                "spearman_ci": ret_val.get("spearman_ci"),
                "sign_auc_ret": ret_val.get("sign_auc_ret"),
                "reading": ret_val.get("reading"),
                "note": "Same last-bar model trained on signed R_h, not path_edge.",
            }
        horizon["discovery_reading"] = (horizon.get("val") or {}).get("reading")
        horizon["confirmatory_reading"] = (horizon.get("test") or {}).get("reading")
        report["horizons"][str(key)] = horizon
        print(f"  path {key} done reading={horizon.get('confirmatory_reading')}")
    return _jsonify(report)


## 02 Load OHLCV

In [ ]:
from datetime import datetime, timedelta, timezone

cache_dir = Path("/content/cache")
cache_dir.mkdir(parents=True, exist_ok=True)
parquet_5m = cache_dir / "btcusd_5m_raw.parquet"
cfg = default_fusion_training_config()
history_days = int(cfg.get("history_days") or 900)
base_url = str(cfg.get("base_url") or "https://api.india.delta.exchange")

if parquet_5m.is_file():
    df5m = pd.read_parquet(parquet_5m)
    print(f"Loaded cache {parquet_5m} rows={len(df5m)}")
else:
    end_dt = datetime.now(timezone.utc)
    start_dt = end_dt - timedelta(days=history_days)
    print(f"Fetching BTCUSD 5m for {history_days}d...")
    df5m = fetch_candles(
        "BTCUSD",
        "5m",
        int(start_dt.timestamp()),
        int(end_dt.timestamp()),
        base_url,
    )
    quality = validate_ohlcv_completeness(df5m, "5m", symbol="BTCUSD")
    print("5m", json.dumps(quality, indent=2, default=str))
    df5m.to_parquet(parquet_5m, index=False)

frames = {"5m": df5m}
for res in ("30m", "1h", "2h"):
    path = cache_dir / f"btcusd_{res}_raw.parquet"
    if path.is_file():
        frames[res] = pd.read_parquet(path)
        print(f"Loaded cache {path} rows={len(frames[res])}")
if all(k in frames for k in ("30m", "1h", "2h")):
    frames = fusion_frames_from_fetch(
        frames["5m"], frames["30m"], frames["1h"], frames["2h"]
    )
print("frames", {k: len(v) for k, v in frames.items()})


## 03 Run diagnostics

In [ ]:
score_test = True
ablate_groups = True
path_regression = False
report = run_fusion_diagnostics(
    df5m,
    frames=frames,
    score_test=score_test,
    ablate_groups=ablate_groups,
    score_path_regression=path_regression,
)
out_dir = Path("/content/export/fusion_diagnostics")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "report.json"
out_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("wrote", out_path)
print("d1_ok", report.get("d1_index_audit", {}).get("ok"))
print("d2_ok", report.get("d2_intervals", {}).get("ok"))
print("d9_ok", report.get("d9_truncation", {}).get("ok"))
print("branch", report.get("branch"), "(diagnostics token, not the bake-off)")
dp = report.get("direction_predictability") or {}
for key, horizon in (dp.get("horizons") or {}).items():
    print(
        key,
        "discovery",
        horizon.get("discovery_reading"),
        "confirmatory",
        horizon.get("confirmatory_reading"),
    )
pr = report.get("path_regression") or {}
for key, horizon in (pr.get("horizons") or {}).items():
    print(
        "path",
        key,
        horizon.get("discovery_reading"),
        horizon.get("confirmatory_reading"),
    )


## 04 Branch

In [ ]:
print(json.dumps({
    "branch": report.get("branch"),
    "notes": report.get("notes"),
    "expectancy_minus_cost": report.get("expectancy_minus_cost"),
}, indent=2, default=str))
print(
    "Pre-registered readings live under direction_predictability. "
    "decide_branch is a diagnostics token, not the experiment verdict. "
    "Checkpoint confusion is CLI-only (best.pt)."
)
